# PeekaPlate 5-Model Dish Benchmark B (TRELLIS.2-4B + MoGe-2 on GMn9C8.pdf Hero Dishes)

- **Enabled Models:** `['trellis2-4b', 'moge2-relief']`
- **Dataset Slice:** `[4:19]` from `yacine105/peekaplate-dish-test-photos`
- **Compliance:** `publication_eligible: False` (private R&D comparison studio).


In [ ]:
import base64
import binascii
import hashlib
import json
import math
import os
import re
import shutil
import struct
import time
from io import BytesIO
from pathlib import Path
from zipfile import ZIP_DEFLATED, ZipFile

import requests
from PIL import Image, ImageOps

CHAT_API = 'https://openrouter.ai/api/v1/chat/completions'
OPENROUTER_MODELS = {
    'gemini-3.1-flash-lite-image': 'google/gemini-3.1-flash-lite-image',
    'gemini-3.1-flash-image': 'google/gemini-3.1-flash-image',
}
FALLBACK_IMAGE_MODELS = {
    'sd-turbo': {
        'repo': 'stabilityai/sd-turbo',
        'revision': 'b261bac6fd2cf515557d5d0707481eafa0485ec2',
        'steps': 4,
        'guidance': 0.0,
        'size': 512,
        'seed': 42,
    },
    'dreamshaper-8': {
        'repo': 'Lykon/dreamshaper-8',
        'revision': 'a7e52b98680b1ba8ff7bce97c7f9f2e2e5337917',
        'steps': 25,
        'guidance': 7.0,
        'size': 512,
        'seed': 42,
    },
}
DINOV3_REPO = 'facebook/dinov3-vitl16-pretrain-lvd1689m'
DINOV3_REVISION = 'ea8dc2863c51be0a264bab82070e3e8836b02d51'
TRELLIS_MODEL_REPO = 'microsoft/TRELLIS.2-4B'
TRELLIS_MODEL_REVISION = 'af44b45f2e35a493886929c6d786e563ec68364d'
GROUNDING_DINO_REPO = 'IDEA-Research/grounding-dino-tiny'
GROUNDING_DINO_REVISION = 'a2bb814dd30d776dcf7e30523b00659f4f141c71'
SAM_REPO = 'facebook/sam-vit-base'
SAM_REVISION = '70c1a07f894ebb5b307fd9eaaee97b9dfc16068f'
PLATE_PRESERVATION_MODE = 'preserve_plate'
MOBILE_OPTIMIZED_MAX_FACES = 35000
DEMO_TRUTHFUL_LABEL = 'Private single-photo 3D demo (not published T0/T1/T2 menu evidence)'
DISH_PROMPT = (
    'Studio food photo of fictional dish Ember Garden: five browned purple sweet potato dumplings, '
    'roasted golden carrots, and parsley on one matte ivory round plate. '
    'Elevated three-quarter view showing the full plate rim with wide margin on a plain dark slate background, '
    'soft diffuse light. Only one plate; no utensils, bowls, hands, or text.'
)
MAX_RESPONSE_BYTES = 32 * 1024 * 1024
MAX_IMAGE_BYTES = 20 * 1024 * 1024
MAX_PIXELS = 4096 * 4096
MIME_TO_FORMAT = {'image/png': 'PNG', 'image/jpeg': 'JPEG', 'image/webp': 'WEBP'}
DATA_URL_RE = re.compile(r'^data:(image/(?:png|jpeg|webp));base64,(.+)$', re.DOTALL)
MAX_USER_DISHES = 64
SUPPORTED_IMAGE_SUFFIXES = ('.png', '.jpg', '.jpeg', '.webp')


class DemoError(RuntimeError):
    """A fixed diagnostic safe to display or package without leaking secrets."""


def secret_source(name, input_root=Path('/kaggle/input'), environ=None):
    """Resolve a credential and report its source, never its value.

    Kaggle Secrets is preferred, but an API-pushed kernel can start with no secret
    attached: on 2026-11-22 UserSecretsClient raised ConnectionError inside Kaggle and
    TRELLIS.2-4B then skipped every dish as a missing token. A private attached dataset
    file named after the secret keeps scheduled reruns working; the environment is last.
    """
    try:
        from kaggle_secrets import UserSecretsClient
        value = UserSecretsClient().get_secret(name) or None
        if value:
            return value, 'kaggle_secrets'
    except Exception:
        pass
    input_path = Path(input_root)
    candidates = []
    if input_path.is_dir():
        if name.upper() == 'HF_TOKEN':
            candidates.extend(sorted(input_path.rglob('hf_token.txt')))
        candidates.extend(sorted(input_path.rglob(f'*{name.lower()}*')))
    seen = set()
    unique_candidates = []
    for c in candidates:
        if c not in seen and c.is_file():
            seen.add(c)
            unique_candidates.append(c)
    for candidate in unique_candidates:
        value = candidate.read_text(encoding='utf-8').strip()
        if value:
            rel = candidate.relative_to(input_path)
            return value, f'dataset:{rel}'
    value = (environ if environ is not None else os.environ).get(name) or None
    return value, 'environment' if value else 'missing'


def kaggle_secret(name):
    return secret_source(name)[0]


def dump_kaggle_input_tree(root=Path('/kaggle/input'), max_depth=3):
    """Print the directory tree under `root` up to `max_depth` (never file contents)."""
    root_path = Path(root)
    print(f'Listing {root_path} tree (max depth {max_depth}):', flush=True)
    if not root_path.exists():
        print(f'  {root_path} does not exist.', flush=True)
        return
    for item in sorted(root_path.rglob('*')):
        try:
            rel = item.relative_to(root_path)
            if len(rel.parts) <= max_depth:
                indent = '  ' * len(rel.parts)
                suffix = '/' if item.is_dir() else ''
                print(f'{indent}{item.name}{suffix}', flush=True)
        except Exception:
            pass


def preflight_dinov3(token, downloader=None):
    """Check gated DINOv3 access via small config.json before heavy builds or paid calls."""
    if not token:
        return False, 'missing_hf_token'
    try:
        if downloader is None:
            from huggingface_hub import hf_hub_download
            downloader = hf_hub_download
        downloader(DINOV3_REPO, 'config.json', revision=DINOV3_REVISION, token=token or False)
        return True, 'ok'
    except Exception as exc:
        name = type(exc).__name__
        message = str(exc)
        if any(code in message for code in ('401', '403', 'GatedRepo')) or 'gated' in message.lower():
            return False, 'missing_or_unauthorized_hf_token'
        return False, f'preflight_unavailable:{name}'


def decode_image(payload):
    try:
        if not isinstance(payload, dict) or payload.get('error'):
            raise ValueError
        choices = payload['choices']
        if not isinstance(choices, list) or not choices:
            raise ValueError
        images = choices[0]['message']['images']
        if not isinstance(images, list) or len(images) != 1:
            raise ValueError
        url = images[0]['image_url']['url']
        if not isinstance(url, str):
            raise ValueError
        match = DATA_URL_RE.match(url.strip())
        if not match:
            raise ValueError
        declared_mime, encoded = match.group(1), match.group(2)
        if len(encoded) > 4 * ((MAX_IMAGE_BYTES + 2) // 3):
            raise ValueError
        raw = base64.b64decode(encoded, validate=True)
        if not raw or len(raw) > MAX_IMAGE_BYTES:
            raise ValueError
        with Image.open(BytesIO(raw)) as image:
            width, height = image.size
            if (image.format != MIME_TO_FORMAT[declared_mime]
                    or width * height > MAX_PIXELS
                    or getattr(image, 'n_frames', 1) != 1):
                raise ValueError
            image.verify()
        with Image.open(BytesIO(raw)) as image:
            mode = 'RGBA' if 'A' in image.getbands() or 'transparency' in image.info else 'RGB'
            pixels = ImageOps.exif_transpose(image).convert(mode)
            pixels.info.clear()
            return pixels
    except (AttributeError, binascii.Error, Image.DecompressionBombError, KeyError,
            OSError, SyntaxError, TypeError, ValueError):
        raise DemoError('OpenRouter returned an invalid or unsupported image.') from None


def request_image(model, api_key, post=requests.post, sleep=time.sleep, max_retries=2):
    last_error = None
    for attempt in range(max_retries + 1):
        try:
            with post(
                CHAT_API,
                headers={'Authorization': f'Bearer {api_key}'},
                json={
                    'model': model,
                    'messages': [{'role': 'user', 'content': DISH_PROMPT}],
                    'modalities': ['image', 'text'],
                    'image_config': {'aspect_ratio': '1:1'},
                },
                timeout=(15, 180),
                stream=True,
                allow_redirects=False,
            ) as response:
                if response.status_code == 429:
                    last_error = DemoError('OpenRouter HTTP 429.')
                    if attempt < max_retries:
                        sleep(2 * (attempt + 1))
                        continue
                    raise last_error
                if response.status_code != 200:
                    raise DemoError(f'OpenRouter HTTP {response.status_code}.')
                body = bytearray()
                for chunk in response.iter_content(64 * 1024):
                    body.extend(chunk)
                    if len(body) > MAX_RESPONSE_BYTES:
                        raise DemoError('OpenRouter response exceeded the size limit.')
            return decode_image(json.loads(body))
        except DemoError:
            raise
        except requests.RequestException:
            raise DemoError('OpenRouter connection or read failed.') from None
        except (UnicodeError, ValueError):
            raise DemoError('OpenRouter returned invalid JSON.') from None
    raise last_error or DemoError('OpenRouter request failed.')


def _exif_orientation(image):
    try:
        return image.getexif().get(0x0112, 1)
    except (AttributeError, OSError, TypeError, ValueError):
        return 1


def stage_source_image(src, dst):
    """Stage dataset photo bytes at `dst` so `source_sha256` always names the true source.

    Kaggle's Pillow/zlib emitted different PNG bytes for identical pixels, which silently replaced
    the dataset digest with a re-encode digest. A single-frame PNG without EXIF orientation is
    therefore copied byte-for-byte; anything that must be re-encoded gets its own
    `input_image_sha256` plus a named transformation, so a derived artifact can never occupy the
    source identity.
    """
    src, dst = Path(src), Path(dst)
    with src.open('rb') as stream:
        source_sha256 = hashlib.file_digest(stream, 'sha256').hexdigest()
    transformation = 'none'
    with Image.open(src) as im:
        width, height = im.size
        if getattr(im, 'n_frames', 1) != 1 or width * height > MAX_PIXELS:
            raise ValueError('Unsupported multi-frame or oversized input image.')
        if im.format == 'PNG' and _exif_orientation(im) in (None, 1):
            shutil.copyfile(src, dst)
            size = [width, height]
        else:
            mode = 'RGBA' if 'A' in im.getbands() or 'transparency' in im.info else 'RGB'
            steps = []
            if _exif_orientation(im) not in (None, 1):
                steps.append('exif_transpose')
            if im.mode != mode:
                steps.append(f'to_{mode.lower()}')
            steps.append('png_encode')
            pixels = ImageOps.exif_transpose(im).convert(mode)
            pixels.info.clear()
            pixels.save(dst, format='PNG')
            size = list(pixels.size)
            transformation = '+'.join(steps)
    with dst.open('rb') as stream:
        input_image_sha256 = hashlib.file_digest(stream, 'sha256').hexdigest()
    return {
        'source_sha256': source_sha256,
        'source_bytes': src.stat().st_size,
        'input_image_sha256': input_image_sha256,
        'input_bytes': dst.stat().st_size,
        'input_transformation': transformation,
        'input_byte_identical': transformation == 'none',
        'size': size,
    }


def discover_user_dish_images(results_dir, input_root=None, max_dishes=MAX_USER_DISHES):
    """Discover up to max_dishes attached user dish photos and optional sibling _mask files."""
    root = Path(input_root) if input_root is not None else Path('/kaggle') / 'input'
    if not root.is_dir():
        return {}
    all_images = sorted(
        (
            path for path in root.rglob('*')
            if path.is_file()
            and not path.name.startswith('.')
            and path.suffix.lower() in SUPPORTED_IMAGE_SUFFIXES
        ),
        key=lambda path: path.relative_to(root).as_posix(),
    )
    candidates = [p for p in all_images if not p.stem.endswith('_mask')][DISH_SLICE_START:min(DISH_SLICE_END, DISH_SLICE_START + max_dishes)]
    mask_lookup = {
        (p.parent, p.stem[:-5]): p
        for p in all_images
        if p.stem.endswith('_mask')
    }

    dish_meta_lookup = {}
    for dm_file in sorted(root.rglob('dishes_manifest.json')):
        try:
            for item in json.loads(dm_file.read_text(encoding='utf-8')).get('dishes', []):
                dish_meta_lookup[item['label']] = item
        except Exception:
            pass
    results_dir = Path(results_dir)
    discovered = {}
    for idx, src in enumerate(candidates, start=1):
        raw_label = re.sub(r'[^a-zA-Z0-9._-]+', '-', src.stem).strip('-.') or f'dish_{idx}'
        label = raw_label if raw_label not in discovered else f'{raw_label}_{idx}'
        folder = results_dir / label
        folder.mkdir(parents=True, exist_ok=True)
        target = folder / 'input_image.png'
        reviewed_mask = mask_lookup.get((src.parent, src.stem))
        try:
            staged = stage_source_image(src, target)
            discovered[label] = {
                'label': label,
                'title': (dish_meta_lookup.get(src.stem) or {}).get('title'),
                'tier': (dish_meta_lookup.get(src.stem) or {}).get('tier'),
                'page': (dish_meta_lookup.get(src.stem) or {}).get('page'),
                'model': 'user-photo-upload',
                'image_source': 'user-uploaded-dataset',
                'original_filename': src.name,
                'dataset_slug': src.parent.name,
                'size': staged['size'],
                'source_sha256': staged['source_sha256'],
                'source_bytes': staged['source_bytes'],
                'input_image_sha256': staged['input_image_sha256'],
                'input_bytes': staged['input_bytes'],
                'input_transformation': staged['input_transformation'],
                'input_byte_identical': staged['input_byte_identical'],
                'image_status': 'success',
                'image_path': target,
                'reviewed_mask_path': reviewed_mask,
                'directory': folder,
            }
        except Exception as err:
            discovered[label] = {
                'label': label,
                'model': 'user-photo-upload',
                'image_source': 'user-uploaded-dataset',
                'original_filename': src.name,
                'image_status': 'failed',
                'image_error': f'Invalid user dish photo ({type(err).__name__}).',
                'directory': folder,
            }
    return discovered


def build_job_manifest(
    label,
    input_entry,
    revisions,
    plate_mode=PLATE_PRESERVATION_MODE,
    seed=42,
    mobile_max_faces=MOBILE_OPTIMIZED_MAX_FACES,
):
    """Build an immutable Path-free Docker worker input contract for one dish photo."""
    src_path = input_entry.get('image_path')
    src_sha = input_entry.get('source_sha256')
    if not src_sha and src_path and Path(src_path).is_file():
        with Path(src_path).open('rb') as stream:
            src_sha = hashlib.file_digest(stream, 'sha256').hexdigest()
    input_sha = input_entry.get('input_image_sha256')
    if not input_sha and src_path and Path(src_path).is_file():
        with Path(src_path).open('rb') as stream:
            input_sha = hashlib.file_digest(stream, 'sha256').hexdigest()
    seed_bytes = f'{label}:{src_sha or "none"}:{plate_mode}:{seed}'.encode('utf-8')
    job_id = 'demo-' + hashlib.sha256(seed_bytes).hexdigest()[:16]
    reviewed = input_entry.get('reviewed_mask_path')
    return {
        'schema_version': 1,
        'job_id': job_id,
        'dish_label': label,
        'purpose': 'private single-photo 3D demo',
        'demo_label': DEMO_TRUTHFUL_LABEL,
        'publication_eligible': False,
        'image_source': input_entry.get('image_source', 'unknown'),
        'original_filename': input_entry.get('original_filename', 'input_image.png'),
        'input_filename': Path(src_path).name if src_path else 'input_image.png',
        'input_size': list(input_entry.get('size', [])),
        'source_sha256': src_sha,
        'input_image_sha256': input_sha,
        'input_transformation': input_entry.get('input_transformation') or 'none',
        'input_byte_identical': bool(input_entry.get('input_byte_identical', input_sha == src_sha)),
        'plate_preservation_mode': plate_mode,
        'reviewed_mask': Path(reviewed).name if reviewed else None,
        'seed': int(seed),
        'export_profiles': {
            'raw': {'trellis_decimation_target': 300000, 'trellis_texture_size': 2048},
            'mobile': {'max_faces': int(mobile_max_faces), 'format': 'glb'},
        },
        'pinned_revisions': {
            k: v for k, v in revisions.items()
            if not k.startswith(('input_', 'cutout_'))
        },
    }


def build_worker_receipt(job_manifest, input_entry, dish_runs, results_dir):
    """Build a verifiable Path-free Docker worker output receipt from generated artifacts."""
    root = Path(results_dir)
    in_path = input_entry.get('image_path')
    in_rec = file_record(in_path, root) if in_path and Path(in_path).is_file() else None
    cut_path = input_entry.get('cutout_path')
    cut_rec = file_record(cut_path, root) if cut_path and Path(cut_path).is_file() else None
    models = {}
    for run in dish_runs.values():
        backend = run.get('backend', 'unknown')
        entry = {
            'status': run.get('status', 'unknown'),
            'mobile_status': mobile_delivery_status(run),
            'appearance': run.get('appearance', 'unknown'),
        }
        if run.get('mobile_error'):
            entry['mobile_error'] = run['mobile_error']
        cm = input_entry.get('cutout_metrics') or {}
        if 'plate_preserved' in run:
            entry['plate_preserved'] = bool(run['plate_preserved'])
        elif 'plate_preserved' in cm:
            entry['plate_preserved'] = bool(cm['plate_preserved'])
        if 'plate_method' in run:
            entry['plate_method'] = run['plate_method']
        elif 'plate_method' in cm:
            entry['plate_method'] = cm['plate_method']
        if run.get('status') == 'success' and run.get('artifact_paths'):
            raw_glb = Path(run['artifact_paths'][0])
            raw_info = file_record(raw_glb, root)
            raw_m = run.get('metrics', {})
            entry['raw_glb'] = {
                **raw_info,
                'vertices': raw_m.get('vertices', raw_m.get('vertex_count')),
                'faces': raw_m.get('faces'),
                'has_vertex_color': raw_m.get('has_vertex_color'),
                'elapsed_seconds': raw_m.get('elapsed_seconds'),
            }
            mob_path = run.get('mobile_glb_path')
            if mob_path and Path(mob_path).is_file():
                mob_info = file_record(mob_path, root)
                mob_m = run.get('mobile_metrics', {})
                entry['mobile_glb'] = {
                    **mob_info,
                    'vertices': mob_m.get('vertices', mob_m.get('vertex_count')),
                    'faces': mob_m.get('faces'),
                    'has_vertex_color': mob_m.get('has_vertex_color'),
                }
        elif run.get('error'):
            entry['error'] = run['error']
        models[backend] = entry
    cut_metrics = dict(input_entry.get('cutout_metrics', {}))
    return {
        'schema_version': 1,
        'job_id': job_manifest['job_id'],
        'dish_label': job_manifest['dish_label'],
        'demo_label': DEMO_TRUTHFUL_LABEL,
        'publication_eligible': False,
        'source_sha256': job_manifest.get('source_sha256'),
        'input_image_sha256': in_rec['sha256'] if in_rec else job_manifest.get('input_image_sha256'),
        'input_transformation': job_manifest.get('input_transformation', 'none'),
        'cutout_sha256': cut_rec['sha256'] if cut_rec else None,
        'plate_preserved': bool(cut_metrics.get('plate_preserved', False)),
        'cutout_metrics': cut_metrics,
        'models': models,
        'pinned_revisions': job_manifest.get('pinned_revisions', {}),
    }

def collect_openrouter_images(directory, api_key, generate=request_image):
    attempts = {}
    for label, model in OPENROUTER_MODELS.items():
        folder = directory / label
        folder.mkdir(parents=True, exist_ok=True)
        entry = attempts[label] = {
            'label': label,
            'model': model,
            'image_source': 'openrouter',
            'directory': folder,
        }
        try:
            if not api_key:
                raise DemoError('OPENROUTER_API_KEY not attached in Kaggle Secrets.')
            with generate(model, api_key) as image:
                target = folder / 'input_image.png'
                image.save(target)
                entry['size'] = list(image.size)
            entry['image_status'] = 'success'
            entry['image_path'] = target
        except DemoError as error:
            entry.update(image_status='failed', image_error=str(error))
        except OSError:
            entry.update(image_status='failed', image_error='Could not save conditioning image.')
    return attempts


def _glb_item(items, index):
    if type(index) is not int or not 0 <= index < len(items):
        raise ValueError('Invalid glTF reference.')
    return items[index]


def _glb_buffer_view(data, binary, index):
    view = _glb_item(data['bufferViews'], index)
    offset = view.get('byteOffset', 0)
    size = view['byteLength']
    if (type(view['buffer']) is not int or view['buffer'] != 0
            or type(offset) is not int or type(size) is not int
            or offset < 0 or size <= 0 or offset + size > len(binary)):
        raise ValueError('Invalid embedded buffer view.')
    return binary[offset:offset + size], view


def _glb_accessor(data, binary, index, kind, formats):
    accessor = _glb_item(data['accessors'], index)
    component = accessor['componentType']
    fmt = struct.Struct('<' + formats[component])
    view_bytes, view = _glb_buffer_view(data, binary, accessor['bufferView'])
    count = accessor['count']
    offset = accessor.get('byteOffset', 0)
    stride = view.get('byteStride', fmt.size)
    if (accessor['type'] != kind or 'sparse' in accessor
            or type(count) is not int or count < 1
            or type(offset) is not int or offset < 0
            or type(stride) is not int or stride < fmt.size
            or offset + (count - 1) * stride + fmt.size > len(view_bytes)):
        raise ValueError('Invalid backed mesh accessor.')
    return count, (fmt.unpack_from(view_bytes, offset + i * stride) for i in range(count))


def _glb_has_bound_texture(data, binary, primitive, vertex_count, verified_images):
    if 'material' not in primitive:
        return False
    material = _glb_item(data['materials'], primitive['material'])
    pbr = material.get('pbrMetallicRoughness', {})
    found = False
    for owner, name in (
        (pbr, 'baseColorTexture'), (pbr, 'metallicRoughnessTexture'),
        (material, 'normalTexture'), (material, 'occlusionTexture'),
        (material, 'emissiveTexture'),
    ):
        if name not in owner:
            continue
        binding = owner[name]
        texture = _glb_item(data['textures'], binding['index'])
        webp = texture.get('extensions', {}).get('EXT_texture_webp')
        source_index = webp['source'] if webp is not None else texture['source']
        image = _glb_item(data['images'], source_index)
        if source_index not in verified_images:
            image_bytes, _ = _glb_buffer_view(data, binary, image['bufferView'])
            expected_format = {'image/png': 'PNG', 'image/jpeg': 'JPEG', 'image/webp': 'WEBP'}[image['mimeType']]
            if 'uri' in image or (webp is not None) != (expected_format == 'WEBP'):
                raise ValueError('Invalid embedded image reference.')
            with Image.open(BytesIO(image_bytes)) as decoded:
                if decoded.format != expected_format or decoded.width * decoded.height > MAX_PIXELS:
                    raise ValueError('Invalid embedded image format or dimensions.')
                decoded.load()
            verified_images.add(source_index)
        coord = binding.get('extensions', {}).get('KHR_texture_transform', {}).get(
            'texCoord', binding.get('texCoord', 0)
        )
        if type(coord) is not int or coord < 0:
            raise ValueError('Invalid texture coordinate set.')
        index = primitive['attributes'][f'TEXCOORD_{coord}']
        uv = _glb_item(data['accessors'], index)
        if uv['componentType'] != 5126 and uv.get('normalized') is not True:
            raise ValueError('Integer texture coordinates must be normalized.')
        count, coordinates = _glb_accessor(data, binary, index, 'VEC2', {5121: 'BB', 5123: 'HH', 5126: 'ff'})
        if count != vertex_count or not all(math.isfinite(v) for pair in coordinates for v in pair):
            raise ValueError('Invalid texture coordinates.')
        found = True
    return found


def validate_glb(path):
    path = Path(path)
    with path.open('rb') as stream:
        if stream.read(4) != b'glTF':
            raise DemoError('Invalid GLB magic.')
        header = stream.read(16)
        if len(header) != 16:
            raise DemoError('Truncated GLB header.')
        version, length, json_length, chunk_type = struct.unpack('<IIII', header)
        file_size = path.stat().st_size
        if (version != 2 or length != file_size or chunk_type != 0x4E4F534A
                or json_length == 0 or json_length % 4 or json_length > min(length - 20, 16 * 1024 * 1024)):
            raise DemoError('Invalid GLB version or chunk bounds.')
        try:
            data = json.loads(stream.read(json_length))
        except (UnicodeError, ValueError):
            raise DemoError('Invalid GLB JSON chunk.') from None
        remaining = length - stream.tell()
        binary_header = stream.read(8)
        if len(binary_header) != 8:
            raise DemoError('GLB is missing its embedded binary chunk.')
        binary_length, binary_type = struct.unpack('<II', binary_header)
        if binary_type != 0x004E4942 or binary_length % 4 or binary_length != remaining - 8:
            raise DemoError('Invalid GLB binary chunk bounds.')
        binary = memoryview(stream.read(binary_length))
    try:
        buffer, = data['buffers']
        size = buffer['byteLength']
        if ('uri' in buffer or type(size) is not int or size <= 0
                or not 0 <= len(binary) - size <= 3):
            raise ValueError('Invalid embedded buffer.')
        binary = binary[:size]
        meshes = data['meshes']
        primitives = [primitive for mesh in meshes for primitive in mesh['primitives']]
        vertex_count = 0
        has_texture = False
        verified_images = set()
        for primitive in primitives:
            if primitive.get('mode', 4) != 4:
                raise ValueError('Expected triangle geometry.')
            attrs = primitive['attributes']
            count, values = _glb_accessor(data, binary, attrs['POSITION'], 'VEC3', {5126: 'fff'})
            positions = list(values)
            if not all(math.isfinite(v) for point in positions for v in point):
                raise ValueError('Non-finite position.')
            if 'indices' in primitive:
                index_count, values = _glb_accessor(
                    data, binary, primitive['indices'], 'SCALAR', {5121: 'B', 5123: 'H', 5125: 'I'}
                )
                indices = [index for (index,) in values]
                if any(index >= count for index in indices):
                    raise ValueError('Triangle index outside position accessor.')
            else:
                index_count, indices = count, range(count)
            if index_count < 3 or index_count % 3:
                raise ValueError('Incomplete triangle geometry.')
            for start in range(0, index_count, 3):
                a, b, c = (positions[indices[start + step]] for step in range(3))
                ab = [y - x for x, y in zip(a, b)]
                ac = [y - x for x, y in zip(a, c)]
                if any(ab[i] * ac[j] != ab[j] * ac[i] for i, j in ((0, 1), (0, 2), (1, 2))):
                    break
            else:
                raise ValueError('No nondegenerate triangle.')
            vertex_count += count
            has_texture |= _glb_has_bound_texture(data, binary, primitive, count, verified_images)
    except (AttributeError, IndexError, KeyError, OSError, TypeError, ValueError, Image.DecompressionBombError) as exc:
        raise DemoError('GLB contains invalid mesh geometry or embedded texture data.') from exc
    if not has_texture:
        raise DemoError('GLB has no material-bound embedded texture on its mesh.')
    return {
        'bytes': file_size,
        'meshes': len(meshes),
        'primitives': len(primitives),
        'vertex_count': vertex_count,
        'has_vertex_color': any('COLOR_0' in primitive['attributes'] for primitive in primitives),
        'has_texture': has_texture,
    }


def mobile_delivery_status(run):
    return run.get('mobile_status', 'missing' if run.get('status') == 'success' else 'not_run')


def collect_mobile_glb(source, destination, stats, run):
    run['mobile_status'] = 'failed' if stats.get('mobile_status') == 'failed' else 'missing'
    if not source.is_file():
        return
    try:
        metrics = validate_glb(source)
        shutil.copyfile(source, destination)
    except (DemoError, OSError) as exc:
        run.update(mobile_status='failed', mobile_error=f'Mobile GLB export unavailable ({type(exc).__name__}).')
        return
    run.update(
        mobile_status='available',
        mobile_glb_path=destination,
        mobile_metrics={
            **{name: stats[f'mobile_{name}'] for name in ('vertices', 'faces') if stats.get(f'mobile_{name}') is not None},
            **metrics,
        },
    )


def file_record(path, root):
    path = Path(path)
    with path.open('rb') as stream:
        digest = hashlib.file_digest(stream, 'sha256').hexdigest()
    return {'path': path.relative_to(root).as_posix(), 'sha256': digest, 'bytes': path.stat().st_size}


def write_results(directory, inputs, runs_3d, settings, revisions, extra_files=()):
    directory = Path(directory)
    directory.mkdir(parents=True, exist_ok=True)
    manifest = {
        'schema_version': 3,
        'purpose': 'private single-photo 3D demo & Docker worker blueprint',
        'demo_label': DEMO_TRUTHFUL_LABEL,
        'plate_preservation_mode': PLATE_PRESERVATION_MODE,
        'publication_eligible': False,
        'prompt': DISH_PROMPT,
        'trellis_settings': settings,
        'source_revisions': revisions,
        'inputs': {},
        'runs_3d': {},
        'extras': [],
    }
    archived = {}

    def stage(path):
        path = Path(path)
        rec = file_record(path, directory)
        archived[rec['path']] = path
        return rec

    for label, info in inputs.items():
        clean = {
            k: v for k, v in info.items()
            if k not in ('directory', 'image_path', 'cutout_path', 'reviewed_mask_path', 'job_manifest_path')
        }
        artifacts = []
        for key in ('image_path', 'cutout_path', 'job_manifest_path'):
            candidate = info.get(key)
            if candidate and Path(candidate).is_file():
                artifacts.append(stage(candidate))
        clean['artifacts'] = artifacts
        manifest['inputs'][label] = clean

    for key, run_info in runs_3d.items():
        clean = {
            k: v for k, v in run_info.items()
            if k not in ('directory', 'artifact_paths', 'preview_path', 'hf_modes_path', 'mobile_glb_path')
        }
        artifacts = []
        for path in run_info.get('artifact_paths', []):
            if Path(path).is_file():
                artifacts.append(stage(path))
        clean['artifacts'] = artifacts
        manifest['runs_3d'][key] = clean

    for extra in extra_files:
        if extra and Path(extra).is_file():
            manifest['extras'].append(stage(extra))

    manifest_text = json.dumps(manifest, indent=2) + '\n'
    with ZipFile(directory / 'dish-3d-comparison.zip', 'w', compression=ZIP_DEFLATED) as archive:
        for rel_path, disk_path in sorted(archived.items()):
            archive.write(disk_path, rel_path)
        archive.writestr('manifest.json', manifest_text)
    (directory / 'manifest.json').write_text(manifest_text, encoding='utf-8')
    return directory / 'manifest.json', directory / 'dish-3d-comparison.zip'

import os as _os
DISH_SLICE_START = int(_os.environ.get('PEEK_DISH_START', '4'))
DISH_SLICE_END = int(_os.environ.get('PEEK_DISH_END', '19'))
_ENV_MODELS = _os.environ.get('PEEK_MODELS')
ENABLED_MODELS = [m.strip() for m in _ENV_MODELS.split(',') if m.strip()] if _ENV_MODELS else ["trellis2-4b", "moge2-relief"]
# TripoSG's marching-cubes decoder emits 1.4M-7.1M faces; cap the raw mesh so the exported GLB stays
# inside the 50 MB production limit that `generate_model.validate_glb` enforces.
MAX_RAW_FACES = int(_os.environ.get('PEEK_MAX_RAW_FACES', '300000'))
print(f'[config] ENABLED_MODELS={ENABLED_MODELS}, DISH_SLICE=[{DISH_SLICE_START}:{DISH_SLICE_END}], MAX_RAW_FACES={MAX_RAW_FACES}')


## 0. Offline fixture checks (no GPU, no network)

Validates OpenRouter payload decoding, DINOv3 preflight notes, GLB validation, and the
manifest/ZIP writer before any GPU work. Cell 2 of `trellislast-version.ipynb`, unchanged.


In [ ]:
# Offline fixture checks. Run these separately or before Kaggle GPU execution.
from tempfile import TemporaryDirectory
from unittest.mock import MagicMock, Mock

with BytesIO() as png:
    Image.new('RGBA', (8, 8), (255, 100, 0, 128)).save(png, format='PNG')
    encoded_png = base64.b64encode(png.getvalue()).decode('ascii')
fixture_payload = {
    'choices': [{'message': {'images': [{'image_url': {'url': f'data:image/png;base64,{encoded_png}'}}]}}]
}
with decode_image(fixture_payload) as image:
    assert image.mode == 'RGBA' and image.size == (8, 8)

for invalid in (
    {'choices': []},
    {'error': {'message': 'fixture'}, **fixture_payload},
    {'choices': [{'message': {'images': [{'image_url': {'url': 'https://example.com/dish.png'}}]}}]},
    {'choices': [{'message': {'images': [{'image_url': {'url': f'data:image/jpeg;base64,{encoded_png}'}}]}}]},
    {'choices': [{'message': {'images': [{'image_url': {'url': 'data:image/png;base64,' + base64.b64encode(b'bad').decode()}}]}}]},
):
    try:
        decode_image(invalid)
    except DemoError:
        pass
    else:
        raise AssertionError('Invalid OpenRouter chat image payload was accepted')

# Verify Chat Completions payload, 429 retry, and secret non-leakage
ok_reply = MagicMock()
ok_reply.__enter__.return_value = ok_reply
ok_reply.status_code = 200
ok_reply.iter_content.side_effect = lambda _: [json.dumps(fixture_payload).encode()]

retry_reply = MagicMock()
retry_reply.__enter__.return_value = retry_reply
retry_reply.status_code = 429

post_retry = Mock(side_effect=[retry_reply, ok_reply])
sleeps = []
with request_image('google/gemini-3.1-flash-lite-image', 'fixture-secret', post=post_retry, sleep=sleeps.append) as img:
    assert img.size == (8, 8)
assert post_retry.call_count == 2 and sleeps == [2]
sent_json = post_retry.call_args_list[0].kwargs['json']
assert sent_json['modalities'] == ['image', 'text'] and sent_json['messages'][0]['content'] == DISH_PROMPT

# Preflight DINOv3: an absent token is reported separately from a denied one
ok, note = preflight_dinov3(None, downloader=Mock(side_effect=RuntimeError('401 GatedRepoError')))
assert ok is False and note == 'missing_hf_token', note
ok, note = preflight_dinov3('tok', downloader=Mock(side_effect=RuntimeError('401 GatedRepoError')))
assert ok is False and note == 'missing_or_unauthorized_hf_token', note
ok2, note2 = preflight_dinov3('tok', downloader=Mock(return_value='/tmp/config.json'))
assert ok2 is True and note2 == 'ok'

# Verify GLB validator + manifest/ZIP packaging with mixed fallback/OpenRouter provenance
with TemporaryDirectory() as tmp:
    root = Path(tmp)
    img_dir = root / 'sd-turbo'
    img_dir.mkdir(parents=True)
    img_path = img_dir / 'input_image.png'
    Image.new('RGB', (16, 16), (120, 80, 40)).save(img_path)
    inputs = {
        'gemini-3.1-flash-lite-image': {
            'label': 'gemini-3.1-flash-lite-image',
            'model': 'google/gemini-3.1-flash-lite-image',
            'image_source': 'openrouter',
            'image_status': 'failed',
            'image_error': 'OpenRouter HTTP 429.',
        },
        'sd-turbo': {
            'label': 'sd-turbo',
            'model': 'stabilityai/sd-turbo',
            'revision': 'b261bac6fd2cf515557d5d0707481eafa0485ec2',
            'image_source': 'kaggle-open-weights-fallback',
            'image_status': 'success',
            'image_path': img_path,
            'size': [16, 16],
        },
    }
    positions = struct.pack('<9f', 0, 0, 0, 1, 0, 0, 0, 1, 0)
    texcoords = struct.pack('<6f', 0, 0, 1, 0, 0, 1)
    fixture_image = BytesIO()
    Image.new('RGB', (2, 2), (40, 80, 120)).save(fixture_image, format='PNG')
    image_bytes = fixture_image.getvalue()
    fixture_binary = positions + texcoords + image_bytes
    fixture_mesh = {
        'asset': {'version': '2.0'},
        'scene': 0,
        'scenes': [{'nodes': [0]}],
        'nodes': [{'mesh': 0}],
        'buffers': [{'byteLength': len(fixture_binary)}],
        'bufferViews': [
            {'buffer': 0, 'byteOffset': 0, 'byteLength': len(positions), 'target': 34962},
            {'buffer': 0, 'byteOffset': len(positions), 'byteLength': len(texcoords), 'target': 34962},
            {'buffer': 0, 'byteOffset': len(positions) + len(texcoords), 'byteLength': len(image_bytes)},
        ],
        'images': [{'bufferView': 2, 'mimeType': 'image/png'}],
        'textures': [{'source': 0}],
        'materials': [{'pbrMetallicRoughness': {'baseColorTexture': {'index': 0}}}],
        'meshes': [{'primitives': [{'attributes': {'POSITION': 0, 'TEXCOORD_0': 1}, 'material': 0}]}],
        'accessors': [
            {'bufferView': 0, 'type': 'VEC3', 'count': 3, 'componentType': 5126,
             'min': [0, 0, 0], 'max': [1, 1, 0]},
            {'bufferView': 1, 'type': 'VEC2', 'count': 3, 'componentType': 5126},
        ],
    }
    mesh_json = json.dumps(fixture_mesh).encode()
    mesh_json += b' ' * (-len(mesh_json) % 4)
    binary_chunk = fixture_binary + b'\0' * (-len(fixture_binary) % 4)
    glb_path = img_dir / 'trellis2_4b.glb'
    glb_path.write_bytes(
        struct.pack('<4sIIII', b'glTF', 2, 28 + len(mesh_json) + len(binary_chunk), len(mesh_json), 0x4E4F534A)
        + mesh_json + struct.pack('<II', len(binary_chunk), 0x004E4942) + binary_chunk
    )
    glb_info = validate_glb(glb_path)
    assert glb_info['has_texture'] is True and glb_info['vertex_count'] == 3

    runs_3d = {
        'sd-turbo/trellis2-4b': {
            'input_label': 'sd-turbo',
            'image_source': 'kaggle-open-weights-fallback',
            'backend': 'TRELLIS.2-4B',
            'appearance': 'pbr_texture',
            'status': 'success',
            'mobile_status': 'missing',
            'artifact_paths': [glb_path],
        }
    }
    manifest_path, zip_path = write_results(root, inputs, runs_3d, {'seed': 42}, {'trellis_model': TRELLIS_MODEL_REVISION})
    manifest = json.loads(manifest_path.read_text(encoding='utf-8'))
    assert manifest['inputs']['sd-turbo']['image_source'] == 'kaggle-open-weights-fallback'
    assert manifest['inputs']['gemini-3.1-flash-lite-image']['image_status'] == 'failed'
    with ZipFile(zip_path) as archive:
        z_manifest = json.loads(archive.read('manifest.json'))
        assert z_manifest['runs_3d']['sd-turbo/trellis2-4b']['mobile_status'] == 'missing'
        for art in z_manifest['runs_3d']['sd-turbo/trellis2-4b']['artifacts']:
            assert hashlib.sha256(archive.read(art['path'])).hexdigest() == art['sha256']
    glb_path.write_bytes(b'glTF' + b'\0' * 20)
    try:
        validate_glb(glb_path)
    except DemoError:
        pass
    else:
        raise AssertionError('Invalid GLB was accepted')
print('Offline comparison fixtures passed.')


In [ ]:
import concurrent.futures
import gc
import importlib.metadata
import os
import shutil
import subprocess
import sys
import uuid

ON_KAGGLE = Path('/kaggle').exists()
ROOT = Path('/kaggle/working') if Path('/kaggle/working').exists() else Path.cwd()
SCRATCH_ROOT = Path('/tmp/peekaplate_scratch') if ON_KAGGLE else ROOT / '.scratch'
SCRATCH_ROOT.mkdir(parents=True, exist_ok=True)
RESULTS_DIR = ROOT / 'results'
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
PYTHON = Path(sys.executable)

HF_CACHE = SCRATCH_ROOT / 'hf_cache'
HF_CACHE.mkdir(parents=True, exist_ok=True)
os.environ['HF_HOME'] = str(HF_CACHE)
os.environ['TORCH_HOME'] = str(SCRATCH_ROOT / 'torch_cache')
os.environ['U2NET_HOME'] = str(SCRATCH_ROOT / 'u2net')
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
os.environ['PIP_DISABLE_PIP_VERSION_CHECK'] = '1'

def run(args, **kwargs):
    process = subprocess.Popen(
        [str(v) for v in args],
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        text=True, errors='replace', bufsize=1, **kwargs
    )
    for line in process.stdout:
        print(line, end='', flush=True)
    ret = process.wait()
    if ret != 0:
        raise RuntimeError(f"Command failed with exit code {ret}: {' '.join(str(a) for a in args)}")

import torch
if not torch.cuda.is_available():
    raise RuntimeError('Enable a CUDA GPU in Kaggle Settings before continuing.')
GPU_INFO = []
for index in range(torch.cuda.device_count()):
    device = torch.cuda.get_device_properties(index)
    vram_gib = round(device.total_memory / 2**30, 2)
    GPU_INFO.append({'index': index, 'name': device.name, 'vram_gib': vram_gib, 'cc': list(torch.cuda.get_device_capability(index))})
    print(f'GPU {index}: {device.name} ({vram_gib} GiB, cc={GPU_INFO[-1]["cc"]})')
print('Python:', sys.version.split()[0], 'PyTorch:', torch.__version__, 'CUDA:', torch.version.cuda)
GPU_COUNT = torch.cuda.device_count()
MAX_GPU_WORKERS = max(1, min(GPU_COUNT, 2))
print(f'Active CUDA GPUs: {GPU_COUNT}, worker pool size: {MAX_GPU_WORKERS}')

NVCC_AVAILABLE = os.name == 'posix' and shutil.which('nvcc') is not None
if NVCC_AVAILABLE:
    compiler = subprocess.check_output(['nvcc', '--version'], text=True)
    print(compiler.strip())

HF_TOKEN, HF_TOKEN_SOURCE = secret_source('HF_TOKEN')
print(f'TRELLIS.2 HF token source: {HF_TOKEN_SOURCE}')
if 'trellis2-4b' in ENABLED_MODELS and (HF_TOKEN_SOURCE == 'missing' or not HF_TOKEN):
    print('[trellis2-4b] HF token is missing while trellis2-4b is enabled; dumping /kaggle/input and aborting.', flush=True)
    if globals().get('dump_kaggle_input_tree'):
        dump_kaggle_input_tree('/kaggle/input', max_depth=3)
    else:
        _in_root = Path('/kaggle/input')
        print(f'Listing {_in_root} tree (max depth 3):', flush=True)
        if _in_root.exists():
            for _p in sorted(_in_root.rglob('*')):
                try:
                    if len(_p.relative_to(_in_root).parts) <= 3:
                        print(f'  {_p}', flush=True)
                except Exception:
                    pass
    raise RuntimeError('Aborting kernel early: trellis2-4b is in ENABLED_MODELS but HF token is missing.')
DINOV3_OK, DINOV3_NOTE = preflight_dinov3(HF_TOKEN)
print(f'TRELLIS.2 DINOv3 preflight: ok={DINOV3_OK} ({DINOV3_NOTE})')

# Pin Kaggle preinstalled scientific stack so runtime installs never upgrade numpy/numba mid-kernel
PIN_DISTRIBUTIONS = (
    'numpy', 'scipy', 'torch', 'torchvision', 'numba', 'llvmlite',
    'scikit-image', 'pillow', 'opencv-python-headless', 'transformers',
    'diffusers', 'accelerate',
)
constraints_lines = []
for dist_name in PIN_DISTRIBUTIONS:
    try:
        dist_ver = importlib.metadata.version(dist_name)
        constraints_lines.append(f'{dist_name}=={dist_ver}')
    except importlib.metadata.PackageNotFoundError:
        pass
constraints_path = SCRATCH_ROOT / 'kaggle_constraints.txt'
constraints_path.write_text('\n'.join(constraints_lines) + '\n', encoding='utf-8')

# Install only missing lightweight runtime packages without downgrading/upgrading Kaggle NumPy or Numba
REQUIRED_PACKAGES = [
    ('omegaconf', 'omegaconf>=2.3.0'),
    ('einops', 'einops>=0.7.0'),
    ('trimesh', 'trimesh>=4.6.0'),
    ('rembg', 'rembg[cpu]==2.0.67'),
    ('onnxruntime', 'onnxruntime'),
    ('scipy', 'scipy'),
    ('pymeshlab', 'pymeshlab'),
    ('diffusers', 'diffusers>=0.30.0'),
    ('accelerate', 'accelerate>=0.30.0'),
]
missing_specs = []
for mod_name, spec in REQUIRED_PACKAGES:
    try:
        __import__(mod_name)
    except ImportError:
        missing_specs.append(spec)
if missing_specs:
    print('Installing missing runtime packages with constraints:', missing_specs)
    install_env = os.environ.copy()
    for secret_name in ('OPENROUTER_API_KEY', 'HF_TOKEN'):
        install_env.pop(secret_name, None)
    try:
        run([PYTHON, '-m', 'pip', 'install', '-q', '-c', str(constraints_path), *missing_specs], env=install_env)
    except RuntimeError:
        print('Retrying constrained base packages + --no-deps rembg==2.0.67...')
        non_rembg = [s for s in missing_specs if not s.startswith('rembg')]
        extra_deps = [s for s in ('pooch', 'pymatting', 'filetype', 'jsonschema', 'watchdog') if s not in non_rembg]
        run([PYTHON, '-m', 'pip', 'install', '-q', '-c', str(constraints_path), *non_rembg, *extra_deps], env=install_env)
        if any(s.startswith('rembg') for s in missing_specs):
            run([PYTHON, '-m', 'pip', 'install', '-q', '--no-deps', 'rembg==2.0.67'], env=install_env)
else:
    print('All lightweight runtime packages already present.')

REVISIONS = {
    'dinov3': DINOV3_REVISION,
    'trellis_model': TRELLIS_MODEL_REVISION,
    'grounding_dino': GROUNDING_DINO_REVISION,
    'sam_vit_base': SAM_REVISION,
}
DOCKER_LOCK_PACKAGES = (
    'torch', 'torchvision', 'transformers', 'diffusers', 'accelerate',
    'huggingface-hub', 'safetensors', 'omegaconf', 'einops', 'trimesh',
    'pymeshlab', 'rembg', 'onnxruntime', 'scipy', 'scikit-image', 'numpy', 'pillow',
)
RUNTIME_PACKAGE_VERSIONS = {}
for _pkg in DOCKER_LOCK_PACKAGES:
    try:
        RUNTIME_PACKAGE_VERSIONS[_pkg] = importlib.metadata.version(_pkg)
    except importlib.metadata.PackageNotFoundError:
        pass


In [ ]:
"""Watertight MoGe-2 2.5D/3D dish relief + ceramic plate builder & photographic UV PBR baker.

NOTE: All artifacts emitted by this module are private single-photo R&D benchmarks
with `publication_eligible: false` (never published T0/T1/T2 menu evidence). All
food surface color comes strictly from unmodified pixels of the captured dish photo.
"""


import argparse
import hashlib
from io import BytesIO
import json
import math
from pathlib import Path
import shutil
import struct
import time
from typing import Any

import numpy as np
from PIL import Image, ImageOps


MAX_MOBILE_FACES = 35000
MAX_PIXELS = 4096 * 4096
STAGED_INPUT_NAME = 'input_image.png'
STAGED_TRANSFORMATION_NONE = 'none'


def sha256_file(path: Path) -> str:
    """Return the SHA-256 hex digest of `path`'s encoded bytes."""
    with Path(path).open('rb') as stream:
        return hashlib.file_digest(stream, 'sha256').hexdigest()


def _exif_orientation(image: Image.Image) -> int | None:
    """Return the EXIF orientation tag (0x0112) without forcing a full decode."""
    try:
        return image.getexif().get(0x0112, 1)
    except (AttributeError, KeyError, OSError, TypeError, ValueError):
        return 1


def _staging_eligible_for_verbatim_copy(image: Image.Image) -> bool:
    """True when `input_image.png` can be a byte-identical copy of the source file."""
    return (
        image.format == 'PNG'
        and getattr(image, 'n_frames', 1) == 1
        and image.width * image.height <= MAX_PIXELS
        and _exif_orientation(image) in (None, 1)
    )


def _staging_transformation_steps(image: Image.Image, target_mode: str) -> list[str]:
    """Name every operation applied when the source must be re-encoded."""
    steps: list[str] = []
    if getattr(image, 'n_frames', 1) != 1:
        steps.append('first_frame')
    if _exif_orientation(image) not in (None, 1):
        steps.append('exif_transpose')
    if image.mode != target_mode:
        steps.append(f'to_{target_mode.lower()}')
    steps.append('png_encode')
    return steps


def stage_source_image(src: Path, dst: Path) -> dict[str, Any]:
    """Stage `src` at `dst` without letting derived bytes occupy the source identity.

    `source_sha256` is hashed from the untouched source bytes before any decode. A single-frame PNG
    (the encoded form of `dst`) without EXIF orientation is copied byte-for-byte, so the staged
    artifact and the declared digest describe the same bytes. Otherwise the derived bytes get their
    own `input_image_sha256` plus a named `input_transformation`, and `source_sha256` still names the
    original source bytes.
    """
    src, dst = Path(src), Path(dst)
    source_sha256 = sha256_file(src)
    input_transformation = STAGED_TRANSFORMATION_NONE

    with Image.open(src) as im:
        if _staging_eligible_for_verbatim_copy(im):
            shutil.copyfile(src, dst)
            size = [im.width, im.height]
        else:
            steps = _staging_transformation_steps(im, 'RGB')
            rgb = ImageOps.exif_transpose(im).convert('RGB')
            rgb.save(dst, format='PNG', optimize=True)
            input_transformation = '+'.join(steps)
            size = list(rgb.size)

    input_image_sha256 = sha256_file(dst)
    return {
        'source_sha256': source_sha256,
        'source_bytes': src.stat().st_size,
        'input_filename': dst.name,
        'input_image_sha256': input_image_sha256,
        'input_bytes': dst.stat().st_size,
        'input_transformation': input_transformation,
        'input_byte_identical': input_transformation == STAGED_TRANSFORMATION_NONE,
        'size': size,
    }


def _ensure_vec3_normals(vertices: np.ndarray, faces: np.ndarray) -> np.ndarray:
    """Compute area-weighted smooth per-vertex normals."""
    normals = np.zeros_like(vertices, dtype=np.float64)
    if len(faces) == 0:
        normals[:, 1] = 1.0
        return normals.astype(np.float32)

    v0 = vertices[faces[:, 0]].astype(np.float64)
    v1 = vertices[faces[:, 1]].astype(np.float64)
    v2 = vertices[faces[:, 2]].astype(np.float64)
    fn = np.cross(v1 - v0, v2 - v0)
    np.add.at(normals, faces[:, 0], fn)
    np.add.at(normals, faces[:, 1], fn)
    np.add.at(normals, faces[:, 2], fn)
    norms = np.linalg.norm(normals, axis=1, keepdims=True)
    bad = norms[:, 0] < 1e-12
    norms[bad] = 1.0
    normals /= norms
    normals[bad] = np.array([0.0, 1.0, 0.0], dtype=np.float64)
    return normals.astype(np.float32)


def check_watertight_edges(faces: np.ndarray) -> tuple[bool, int, int]:
    """Return `(is_watertight, boundary_edges, nonmanifold_edges)` for triangle mesh `faces`."""
    if len(faces) == 0:
        return False, 0, 0
    f = np.asarray(faces, dtype=np.int64)
    e01 = np.sort(f[:, [0, 1]], axis=1)
    e12 = np.sort(f[:, [1, 2]], axis=1)
    e20 = np.sort(f[:, [2, 0]], axis=1)
    edges = np.vstack([e01, e12, e20])
    _, counts = np.unique(edges, axis=0, return_counts=True)
    boundary = int(np.sum(counts == 1))
    nonmanifold = int(np.sum(counts > 2))
    return (boundary == 0 and nonmanifold == 0), boundary, nonmanifold


def write_pbr_glb(
    vertices: np.ndarray,
    faces: np.ndarray,
    uvs: np.ndarray,
    base_color_image: Image.Image,
    out_path: Path,
    *,
    normals: np.ndarray | None = None,
    normal_image: Image.Image | None = None,
    metallic: float = 0.0,
    roughness: float = 0.48,
    double_sided: bool = False,
    image_format: str = 'PNG',
    extras: dict[str, Any] | None = None,
) -> dict[str, Any]:
    """Write a valid binary glTF 2.0 `.glb` embedding PNG or JPEG PBR texture(s)."""
    verts = np.ascontiguousarray(vertices, dtype=np.float32)
    tris = np.ascontiguousarray(faces, dtype=np.uint32)
    texcoords = np.ascontiguousarray(uvs, dtype=np.float32)
    if normals is None:
        norms = _ensure_vec3_normals(verts, tris)
    else:
        norms = np.ascontiguousarray(normals, dtype=np.float32)

    if len(verts) == 0 or len(tris) == 0:
        raise ValueError('Cannot export empty mesh to GLB.')
    if not (np.isfinite(verts).all() and np.isfinite(norms).all() and np.isfinite(texcoords).all()):
        raise ValueError('Cannot export non-finite mesh attributes to GLB.')

    fmt_u = image_format.upper()
    if fmt_u not in ('PNG', 'JPEG'):
        raise ValueError(f'Unsupported core glTF 2.0 image format: {image_format!r}')
    mime = 'image/jpeg' if fmt_u == 'JPEG' else 'image/png'

    img_buf = BytesIO()
    if fmt_u == 'JPEG':
        base_rgb = base_color_image.convert('RGB')
        base_rgb.save(img_buf, format='JPEG', quality=88, optimize=True)
    else:
        base_rgb = base_color_image.convert('RGB') if base_color_image.mode != 'RGBA' else base_color_image
        base_rgb.save(img_buf, format='PNG', optimize=True)
    base_bytes = img_buf.getvalue()

    norm_bytes: bytes | None = None
    if normal_image is not None:
        nbuf = BytesIO()
        normal_image.convert('RGB').save(nbuf, format='PNG', optimize=True)
        norm_bytes = nbuf.getvalue()

    bin_parts: list[bytes] = []
    buffer_views: list[dict[str, Any]] = []
    byte_offset = 0

    def _append_view(data: bytes, target: int | None = None) -> int:
        nonlocal byte_offset
        pad = (-len(data)) % 4
        bin_parts.append(data + (b'\x00' * pad))
        bv: dict[str, Any] = {'buffer': 0, 'byteOffset': byte_offset, 'byteLength': len(data)}
        if target is not None:
            bv['target'] = target
        idx = len(buffer_views)
        buffer_views.append(bv)
        byte_offset += len(data) + pad
        return idx

    idx_bytes = tris.reshape(-1).tobytes()
    bv_idx = _append_view(idx_bytes, target=34963)
    bv_pos = _append_view(verts.tobytes(), target=34962)
    bv_norm = _append_view(norms.tobytes(), target=34962)
    bv_uv = _append_view(texcoords.tobytes(), target=34962)
    bv_base_img = _append_view(base_bytes)
    bv_norm_img = _append_view(norm_bytes) if norm_bytes is not None else None

    v_min = [float(x) for x in verts.min(axis=0)]
    v_max = [float(x) for x in verts.max(axis=0)]

    images = [{'bufferView': bv_base_img, 'mimeType': mime}]
    textures = [{'sampler': 0, 'source': 0}]
    material: dict[str, Any] = {
        'name': 'DishPhotoCeramicPBR',
        'pbrMetallicRoughness': {
            'baseColorTexture': {'index': 0},
            'metallicFactor': float(metallic),
            'roughnessFactor': float(roughness),
        },
        'doubleSided': bool(double_sided),
    }
    if bv_norm_img is not None:
        images.append({'bufferView': bv_norm_img, 'mimeType': 'image/png'})
        textures.append({'sampler': 0, 'source': 1})
        material['normalTexture'] = {'index': 1, 'scale': 0.85}

    asset_extras = {'publication_eligible': False}
    if extras:
        asset_extras.update(extras)
        asset_extras['publication_eligible'] = False

    gltf = {
        'asset': {
            'version': '2.0',
            'generator': 'peekaplate-multi-model-3d',
            'extras': asset_extras,
        },
        'scene': 0,
        'scenes': [{'nodes': [0]}],
        'nodes': [{'mesh': 0, 'name': 'dish_3d'}],
        'meshes': [{
            'primitives': [{
                'attributes': {'POSITION': 1, 'NORMAL': 2, 'TEXCOORD_0': 3},
                'indices': 0,
                'material': 0,
            }]
        }],
        'materials': [material],
        'samplers': [{
            'magFilter': 9729,
            'minFilter': 9987 if min(base_rgb.size) >= 4 else 9729,
            'wrapS': 33071,
            'wrapT': 33071,
        }],
        'textures': textures,
        'images': images,
        'buffers': [{'byteLength': byte_offset}],
        'bufferViews': buffer_views,
        'accessors': [
            {
                'bufferView': bv_idx,
                'componentType': 5125,  # UNSIGNED_INT
                'count': int(tris.size),
                'type': 'SCALAR',
                'min': [int(tris.min())],
                'max': [int(tris.max())],
            },
            {
                'bufferView': bv_pos,
                'componentType': 5126,  # FLOAT
                'count': int(len(verts)),
                'type': 'VEC3',
                'min': v_min,
                'max': v_max,
            },
            {
                'bufferView': bv_norm,
                'componentType': 5126,
                'count': int(len(norms)),
                'type': 'VEC3',
            },
            {
                'bufferView': bv_uv,
                'componentType': 5126,
                'count': int(len(texcoords)),
                'type': 'VEC2',
            },
        ],
    }

    json_bytes = json.dumps(gltf, separators=(',', ':'), ensure_ascii=False).encode('utf-8')
    json_pad = (-len(json_bytes)) % 4
    json_chunk = json_bytes + (b' ' * json_pad)
    bin_blob = b''.join(bin_parts)
    total_len = 12 + 8 + len(json_chunk) + 8 + len(bin_blob)

    out_path = Path(out_path)
    out_path.parent.mkdir(parents=True, exist_ok=True)
    with out_path.open('wb') as f:
        f.write(struct.pack('<4sII', b'glTF', 2, total_len))
        f.write(struct.pack('<II', len(json_chunk), 0x4E4F534A))
        f.write(json_chunk)
        f.write(struct.pack('<II', len(bin_blob), 0x004E4942))
        f.write(bin_blob)

    wt, b_edges, nm_edges = check_watertight_edges(tris)
    return {
        'vertices': int(len(verts)),
        'faces': int(len(tris)),
        'bytes': int(out_path.stat().st_size),
        'has_texture': True,
        'has_vertex_color': False,
        'watertight': bool(wt),
        'boundary_edges': b_edges,
        'nonmanifold_edges': nm_edges,
        'publication_eligible': False,
    }


def _fast_chamfer_distance(mask: np.ndarray) -> np.ndarray:
    """Compute normalized [0, 1] distance from background in pure NumPy."""
    h, w = mask.shape
    inf = float(h + w) * 4.0
    dist = np.where(mask, inf, 0.0).astype(np.float32)
    # Forward / backward chamfer passes in NumPy slices (fast, O(H+W) ops only? Use iterative relaxation)
    # 25 iterations of vector shift min gives smooth radial distance up to 50% grid radius
    iters = max(24, max(h, w) // 2)
    diag = np.float32(1.4142135)
    for _ in range(iters):
        p = np.pad(dist, 1, mode='constant', constant_values=0.0)
        cand = np.minimum.reduce([
            p[1:-1, 1:-1],
            p[:-2, 1:-1] + 1.0,
            p[2:, 1:-1] + 1.0,
            p[1:-1, :-2] + 1.0,
            p[1:-1, 2:] + 1.0,
            p[:-2, :-2] + diag,
            p[:-2, 2:] + diag,
            p[2:, :-2] + diag,
            p[2:, 2:] + diag,
        ])
        dist = np.where(mask, cand, 0.0)
    mx = float(dist.max())
    return (dist / mx) if mx > 1e-6 else dist


def _remove_diagonal_pinches_and_keep_largest(quads: np.ndarray) -> np.ndarray:
    """Ensure active quad union has simple closed boundary loops without checkerboard pinches."""
    q = quads.copy()
    h, w = q.shape
    # Fill 1-cell enclosed background holes first
    padded = np.pad(q, 1, mode='constant', constant_values=False)
    n4 = (
        padded[:-2, 1:-1].astype(np.int8)
        + padded[2:, 1:-1].astype(np.int8)
        + padded[1:-1, :-2].astype(np.int8)
        + padded[1:-1, 2:].astype(np.int8)
    )
    q = q | (n4 == 4)

    # Keep only the largest 4-connected component (with > 4 quads) so zero floating specks cause pinches
    visited = np.zeros_like(q, dtype=bool)
    best_comp: list[tuple[int, int]] = []
    for r in range(h):
        for c in range(w):
            if q[r, c] and not visited[r, c]:
                stack = [(r, c)]
                visited[r, c] = True
                comp = []
                while stack:
                    cr, cc = stack.pop()
                    comp.append((cr, cc))
                    for dr, dc in ((-1, 0), (1, 0), (0, -1), (0, 1)):
                        nr, nc = cr + dr, cc + dc
                        if 0 <= nr < h and 0 <= nc < w and q[nr, nc] and not visited[nr, nc]:
                            visited[nr, nc] = True
                            stack.append((nr, nc))
                if len(comp) > len(best_comp):
                    best_comp = comp

    if best_comp:
        clean = np.zeros_like(q, dtype=bool)
        rr, cc = zip(*best_comp)
        clean[np.array(rr), np.array(cc)] = True
        q = clean

    # Iteratively bridge any 2x2 diagonal checkerboard pinch so boundary is strictly 2-manifold
    for _ in range(8):
        tl = q[:-1, :-1]
        tr = q[:-1, 1:]
        bl = q[1:, :-1]
        br = q[1:, 1:]
        pinch_a = tl & br & (~tr) & (~bl)
        pinch_b = tr & bl & (~tl) & (~br)
        if not (pinch_a.any() or pinch_b.any()):
            break
        q[:-1, 1:][pinch_a] = True
        q[:-1, :-1][pinch_b] = True
    return q


def _sanitize_depth_map(
    depth_map: np.ndarray | None,
    fg_mask: np.ndarray | None = None,
) -> np.ndarray | None:
    """Replace non-finite MoGe-2 background/mask depth values with the finite far-rim percentile."""
    if depth_map is None:
        return None
    dm = np.asarray(depth_map, dtype=np.float32)
    if dm.ndim > 2:
        dm = np.squeeze(dm)
    while dm.ndim > 2:
        dm = dm[0]
    if dm.ndim != 2 or dm.size == 0:
        return None
    finite = np.isfinite(dm)
    if not finite.any():
        return None
    if fg_mask is not None and fg_mask.shape == dm.shape and (finite & fg_mask).any():
        ref_vals = dm[finite & fg_mask]
    else:
        ref_vals = dm[finite]
    hi = float(np.percentile(ref_vals, 97))
    return np.where(finite, dm, np.float32(hi)).astype(np.float32, copy=False)


def _sanitize_normal_map(normal_map: np.ndarray | None) -> np.ndarray | None:
    """Convert float/uint8 MoGe-2 normal maps to finite uint8 RGB images."""
    if normal_map is None:
        return None
    nm = np.asarray(normal_map)
    if nm.ndim == 3 and nm.shape[0] == 3 and nm.shape[-1] != 3:
        nm = np.transpose(nm, (1, 2, 0))
    if np.issubdtype(nm.dtype, np.floating):
        nm_f = nm.astype(np.float32, copy=False)
        if nm_f.ndim == 3 and nm_f.shape[-1] == 3:
            bad = ~np.isfinite(nm_f).all(axis=-1)
            if bad.any():
                nm_f = nm_f.copy()
                nm_f[bad] = np.array([0.0, 0.0, 1.0], dtype=np.float32)
        else:
            nm_f = np.nan_to_num(nm_f, nan=0.0, posinf=1.0, neginf=-1.0)
        return np.clip((nm_f * 0.5 + 0.5) * 255.0, 0, 255).astype(np.uint8)
    return np.asarray(nm, dtype=np.uint8)


def _build_watertight_relief_geometry(
    rgba_arr: np.ndarray,
    grid_res: int,
    depth_map: np.ndarray | None,
) -> tuple[np.ndarray, np.ndarray, np.ndarray, tuple[int, int, int]]:
    """Build watertight top photo-relief + bottom lathed ceramic plate mesh."""
    img_h, img_w = rgba_arr.shape[:2]
    depth_map = _sanitize_depth_map(depth_map, rgba_arr[..., 3] > 20)
    aspect = img_w / max(1, img_h)
    gh = max(20, int(grid_res))
    gw = max(20, int(round(grid_res * aspect)))
    if gw > 1.45 * grid_res:
        gw = int(round(grid_res * 1.45))
        gh = max(20, int(round(gw / max(0.25, aspect))))

    # Resample RGBA alpha & luminance onto vertex grid (gh x gw)
    ys_px = np.linspace(0, img_h - 1, gh).astype(np.int32)
    xs_px = np.linspace(0, img_w - 1, gw).astype(np.int32)
    grid_rgba = rgba_arr[ys_px[:, None], xs_px[None, :]]
    v_alpha = grid_rgba[..., 3] > 20
    # Estimate ceramic color from near-rim pixels (alpha>180 and outer 20% distance)
    lum = (
        0.299 * grid_rgba[..., 0]
        + 0.587 * grid_rgba[..., 1]
        + 0.114 * grid_rgba[..., 2]
    ).astype(np.float32) / 255.0

    # Active quads where at least 2 corners or center are inside foreground alpha
    q_count = (
        v_alpha[:-1, :-1].astype(np.int8)
        + v_alpha[:-1, 1:].astype(np.int8)
        + v_alpha[1:, :-1].astype(np.int8)
        + v_alpha[1:, 1:].astype(np.int8)
    )
    quads = q_count >= 2
    if int(quads.sum()) < 4:
        # Fallback to full center circle if mask is nearly empty
        yy, xx = np.mgrid[0:gh - 1, 0:gw - 1]
        cy, cx = (gh - 2) / 2.0, (gw - 2) / 2.0
        quads = (((yy - cy) / max(1.0, cy)) ** 2 + ((xx - cx) / max(1.0, cx)) ** 2) <= 0.65
    quads = _remove_diagonal_pinches_and_keep_largest(quads)

    # Mark vertices touched by active quads
    v_active = np.zeros((gh, gw), dtype=bool)
    v_active[:-1, :-1] |= quads
    v_active[:-1, 1:] |= quads
    v_active[1:, :-1] |= quads
    v_active[1:, 1:] |= quads

    # Sample ceramic glaze color from outer foreground rim (safe non-white neutral)
    d_norm = _fast_chamfer_distance(v_active)
    rim_mask = v_active & (d_norm > 0.01) & (d_norm < 0.22)
    if rim_mask.any():
        rim_rgb = np.median(grid_rgba[rim_mask, :3], axis=0).astype(np.uint8)
        ceramic_rgb = (
            int(0.75 * rim_rgb[0] + 0.25 * 225),
            int(0.75 * rim_rgb[1] + 0.25 * 222),
            int(0.75 * rim_rgb[2] + 0.25 * 218),
        )
    else:
        ceramic_rgb = (224, 220, 214)

    # Height relief map y_top(r, c)
    if depth_map is not None:
        dm = np.asarray(depth_map, dtype=np.float32)
        if dm.shape != (img_h, img_w):
            dm_im = Image.fromarray(dm).resize((gw, gh), Image.Resampling.BILINEAR)
            d_grid = np.asarray(dm_im, dtype=np.float32)
        else:
            d_grid = dm[ys_px[:, None], xs_px[None, :]]
        finite_fg = d_grid[v_active & np.isfinite(d_grid)]
        lo = float(np.percentile(finite_fg, 3)) if len(finite_fg) else 0.0
        hi = float(np.percentile(finite_fg, 97)) if len(finite_fg) else 1.0
        d_grid = np.where(np.isfinite(d_grid), d_grid, np.float32(hi))
        # Invert typical camera depth (closer -> higher relief), clamp softly
        rel_h = np.clip(
            np.nan_to_num((hi - d_grid) / max(1e-5, hi - lo), nan=0.0, posinf=1.0, neginf=0.0),
            0.0,
            1.0,
        )
        # Taper cleanly at the rim silhouette to meet the plate edge
        rim_blend = np.clip(d_norm * 3.5, 0.25, 1.0)
        y_top = 0.09 + (0.28 * rel_h * rim_blend) + 0.05 * np.sqrt(d_norm)
    else:
        # Smooth foreground food/bowl relief prior guided by radial distance and local luminance
        # Compute high-frequency local contrast to lift toppings/ingredients slightly relative to bowl/broth
        pad_l = np.pad(lum, 2, mode='edge')
        local_avg = (
            pad_l[:-4, 2:-2] + pad_l[4:, 2:-2] + pad_l[2:-2, :-4] + pad_l[2:-2, 4:] + lum
        ) / 5.0
        detail = np.clip((lum - local_avg) * 2.5, -0.35, 0.45)
        dome = np.sin(np.clip(d_norm, 0.0, 1.0) * (math.pi * 0.62))
        y_top = 0.09 + 0.24 * dome + 0.045 * detail * np.clip(d_norm * 4.0, 0.0, 1.0)

    # Parametric ceramic bowl/plate underside y_bot(r, c) sitting at y=0 foot ring
    # Rim curves smoothly down into a flat resting base at d_norm >= 0.42
    bowl_curve = np.clip(d_norm / 0.42, 0.0, 1.0)
    y_bot = 0.08 * np.power(1.0 - bowl_curve, 1.45)
    y_top = np.nan_to_num(y_top, nan=0.09, posinf=0.42, neginf=0.09)
    y_top = np.maximum(y_top, y_bot + 0.015)

    # Map image plane -> metric platter coords [-0.9..0.9] centered on foreground bbox
    rs_act, cs_act = np.where(v_active)
    c_mid = 0.5 * (cs_act.min() + cs_act.max())
    r_mid = 0.5 * (rs_act.min() + rs_act.max())
    span = max(1.0, float(max(cs_act.max() - cs_act.min(), rs_act.max() - rs_act.min())))
    scale_xz = 1.65 / span

    jj, ii = np.meshgrid(np.arange(gw, dtype=np.float32), np.arange(gh, dtype=np.float32))
    pos_x = (jj - c_mid) * scale_xz
    pos_z = (ii - r_mid) * scale_xz

    # Active vertex compact indices
    n_act = int(v_active.sum())
    top_vid = np.full((gh, gw), -1, dtype=np.int32)
    top_vid[v_active] = np.arange(n_act, dtype=np.int32)
    bot_vid = np.where(top_vid >= 0, top_vid + n_act, -1)

    top_xyz = np.stack([pos_x[v_active], y_top[v_active], pos_z[v_active]], axis=1).astype(np.float32)
    bot_xyz = np.stack([pos_x[v_active], y_bot[v_active], pos_z[v_active]], axis=1).astype(np.float32)
    vertices = np.vstack([top_xyz, bot_xyz])

    # UV coords: top maps [0..1, 0..H/(H+16)] to unmodified captured photo; bottom maps to bottom ceramic swatch
    uv_scale_v = img_h / float(img_h + 16)
    u_coords = (jj / max(1, gw - 1))[v_active]
    v_coords = (ii / max(1, gh - 1))[v_active] * uv_scale_v
    top_uv = np.stack([u_coords, v_coords], axis=1).astype(np.float32)
    swatch_v = (img_h + 8) / float(img_h + 16)
    bot_uv = np.tile(np.array([[0.5, swatch_v]], dtype=np.float32), (n_act, 1))
    uvs = np.vstack([top_uv, bot_uv])

    # Build top + bottom triangles for every active quad (r, c)
    ar, ac = np.where(quads)
    tl_t = top_vid[ar, ac]
    tr_t = top_vid[ar, ac + 1]
    bl_t = top_vid[ar + 1, ac]
    br_t = top_vid[ar + 1, ac + 1]

    tl_b = bot_vid[ar, ac]
    tr_b = bot_vid[ar, ac + 1]
    bl_b = bot_vid[ar + 1, ac]
    br_b = bot_vid[ar + 1, ac + 1]

    # Top faces CCW looking down (+Y normal)
    t1 = np.stack([tl_t, bl_t, tr_t], axis=1)
    t2 = np.stack([tr_t, bl_t, br_t], axis=1)
    # Bottom faces opposite winding (-Y normal)
    b1 = np.stack([tl_b, tr_b, bl_b], axis=1)
    b2 = np.stack([tr_b, br_b, bl_b], axis=1)

    # Boundary side quads for every active quad side lacking a neighbor (keeps half-edge orientation exact!)
    side_tris: list[np.ndarray] = []
    qh, qw = quads.shape

    def _add_side(a_idx, b_idx):
        at = top_vid[a_idx]
        bt = top_vid[b_idx]
        ab = bot_vid[a_idx]
        bb = bot_vid[b_idx]
        side_tris.append(np.stack([at, bt, bb], axis=1))
        side_tris.append(np.stack([at, bb, ab], axis=1))

    # Top boundary (quad[r,c] has no neighbor at [r-1, c]): top triangle edge is (tr -> tl)
    mask_n = quads.copy()
    mask_n[1:, :] &= ~quads[:-1, :]
    rn, cn = np.where(mask_n)
    if len(rn):
        _add_side((rn, cn + 1), (rn, cn))

    # Bottom boundary (no neighbor at [r+1, c]): top triangle edge is (bl -> br)
    mask_s = quads.copy()
    mask_s[:-1, :] &= ~quads[1:, :]
    rs, cs = np.where(mask_s)
    if len(rs):
        _add_side((rs + 1, cs), (rs + 1, cs + 1))

    # Left boundary (no neighbor at [r, c-1]): top triangle edge is (tl -> bl)
    mask_w = quads.copy()
    mask_w[:, 1:] &= ~quads[:, :-1]
    rw, cw = np.where(mask_w)
    if len(rw):
        _add_side((rw, cw), (rw + 1, cw))

    # Right boundary (no neighbor at [r, c+1]): top triangle edge is (br -> tr)
    mask_e = quads.copy()
    mask_e[:, :-1] &= ~quads[:, 1:]
    re_, ce = np.where(mask_e)
    if len(re_):
        _add_side((re_ + 1, ce + 1), (re_, ce + 1))

    all_blocks = [t1, t2, b1, b2, *side_tris]
    faces = np.vstack(all_blocks).astype(np.uint32)
    return vertices, faces, uvs, ceramic_rgb


def _compose_atlas_with_ceramic_band(
    cutout_rgba_path: Path,
    ceramic_rgb: tuple[int, int, int],
    max_tex_dim: int,
) -> tuple[Image.Image, np.ndarray]:
    """Composite captured foreground RGB onto rim-tinted backdrop plus a bottom 16px ceramic swatch."""
    with Image.open(cutout_rgba_path) as im:
        rgba = ImageOps.exif_transpose(im).convert('RGBA')
    orig_w, orig_h = rgba.size
    if max(orig_w, orig_h) > max_tex_dim:
        rgba.thumbnail((max_tex_dim, max_tex_dim), Image.Resampling.LANCZOS)
    rgba_arr = np.asarray(rgba, dtype=np.uint8)
    h, w = rgba_arr.shape[:2]

    # Blend transparent margins against neutral ceramic color so linear texture sampling has zero dark fringing
    alpha = rgba_arr[..., 3:4].astype(np.float32) / 255.0
    bg = np.array(ceramic_rgb, dtype=np.float32).reshape(1, 1, 3)
    rgb_clean = np.clip(rgba_arr[..., :3].astype(np.float32) * alpha + bg * (1.0 - alpha), 0, 255).astype(np.uint8)

    band = np.broadcast_to(np.array(ceramic_rgb, dtype=np.uint8).reshape(1, 1, 3), (16, w, 3)).copy()
    atlas_arr = np.vstack([rgb_clean, band])
    return Image.fromarray(atlas_arr, mode='RGB'), rgba_arr


def _normal_map_from_rgba(rgba_arr: np.ndarray, ceramic_rgb: tuple[int, int, int]) -> Image.Image:
    """Create a subtle tangent-space normal map from foreground photo luminance gradients."""
    gray = (0.299 * rgba_arr[..., 0] + 0.587 * rgba_arr[..., 1] + 0.114 * rgba_arr[..., 2]) / 255.0
    fg = rgba_arr[..., 3] > 20
    gray = np.where(fg, gray, 0.5)
    gx = np.zeros_like(gray, dtype=np.float32)
    gy = np.zeros_like(gray, dtype=np.float32)
    gx[:, 1:-1] = (gray[:, 2:] - gray[:, :-2]) * 1.15
    gy[1:-1, :] = (gray[2:, :] - gray[:-2, :]) * 1.15
    nz = np.ones_like(gray, dtype=np.float32)
    vec = np.stack([-gx, -gy, nz], axis=-1)
    vec /= np.linalg.norm(vec, axis=-1, keepdims=True).clip(min=1e-6)
    nmap = np.clip((vec * 0.5 + 0.5) * 255.0, 0, 255).astype(np.uint8)
    # Neutral flat tangent normal (128, 128, 255) for the ceramic underside band
    flat_band = np.broadcast_to(np.array([128, 128, 255], dtype=np.uint8).reshape(1, 1, 3), (16, rgba_arr.shape[1], 3)).copy()
    _ = ceramic_rgb
    return Image.fromarray(np.vstack([nmap, flat_band]), mode='RGB')


def build_moge2_relief_and_ceramic_plate_glb(
    cutout_rgba_path: Path,
    raw_glb_path: Path,
    mobile_glb_path: Path | None = None,
    *,
    depth_map: np.ndarray | None = None,
    normal_map: np.ndarray | None = None,
    raw_grid_res: int = 160,
    mobile_grid_res: int = 86,
) -> dict[str, Any]:
    """Export watertight captured-photo dish relief + ceramic plate GLBs (`raw` + `<=35k` mobile)."""
    t0 = time.perf_counter()
    with Image.open(cutout_rgba_path) as im:
        rgba = ImageOps.exif_transpose(im).convert('RGBA')
    rgba_arr = np.asarray(rgba, dtype=np.uint8)

    verts_raw, faces_raw, uvs_raw, ceramic_rgb = _build_watertight_relief_geometry(
        rgba_arr, raw_grid_res, depth_map
    )
    atlas_raw, clean_raw_arr = _compose_atlas_with_ceramic_band(cutout_rgba_path, ceramic_rgb, max_tex_dim=1536)
    normal_map = _sanitize_normal_map(normal_map)
    if normal_map is not None:
        nm_arr = np.asarray(normal_map, dtype=np.uint8)
        nm_im = Image.fromarray(nm_arr).resize((atlas_raw.width, atlas_raw.height - 16), Image.Resampling.BILINEAR)
        flat_band = Image.new('RGB', (atlas_raw.width, 16), (128, 128, 255))
        norm_tex_raw = Image.new('RGB', atlas_raw.size)
        norm_tex_raw.paste(nm_im, (0, 0))
        norm_tex_raw.paste(flat_band, (0, atlas_raw.height - 16))
    else:
        norm_tex_raw = _normal_map_from_rgba(clean_raw_arr, ceramic_rgb)

    raw_metrics = write_pbr_glb(
        verts_raw,
        faces_raw,
        uvs_raw,
        atlas_raw,
        raw_glb_path,
        normal_image=norm_tex_raw,
        roughness=0.42,
        extras={'backend': 'moge2-relief', 'plate_base_added': True},
    )

    mob_metrics: dict[str, Any] | None = None
    if mobile_glb_path is not None:
        g_mob = min(mobile_grid_res, 88)
        verts_m, faces_m, uvs_m, _ = _build_watertight_relief_geometry(rgba_arr, g_mob, depth_map)
        while len(faces_m) > MAX_MOBILE_FACES and g_mob > 28:
            g_mob = max(28, int(g_mob * 0.85))
            verts_m, faces_m, uvs_m, _ = _build_watertight_relief_geometry(rgba_arr, g_mob, depth_map)
        atlas_mob, _ = _compose_atlas_with_ceramic_band(cutout_rgba_path, ceramic_rgb, max_tex_dim=1024)
        mob_metrics = write_pbr_glb(
            verts_m,
            faces_m,
            uvs_m,
            atlas_mob,
            mobile_glb_path,
            roughness=0.42,
            image_format='JPEG',
            extras={'backend': 'moge2-relief-mobile', 'plate_base_added': True},
        )

    elapsed = round(time.perf_counter() - t0, 3)
    raw_metrics['elapsed_seconds'] = elapsed
    if mob_metrics is not None:
        mob_metrics['elapsed_seconds'] = elapsed
    return {
        'raw': raw_metrics,
        'mobile': mob_metrics,
        'elapsed_seconds': elapsed,
        'plate_base_added': True,
        'publication_eligible': False,
    }


def _decimate_vertex_cluster(vertices: np.ndarray, faces: np.ndarray, target_faces: int) -> tuple[np.ndarray, np.ndarray]:
    """Deterministic fallback mesh decimation by voxel vertex clustering.

    Returns the smallest mesh the clustering can reach, never the unchanged input when a target below
    its face count was requested: the previous six-round loop fell through to `return vertices, faces`,
    so Kaggle `benchmark-a/4` exported 5,445,228-face "mobile" GLBs (225-354 MB) while claiming
    `mobile_status=available`. A result still above `target_faces` is returned as measured, and
    :func:`bake_photographic_uv_pbr_glb` then omits the mobile artifact instead of shipping it.
    """
    source_vertices = np.asarray(vertices, dtype=np.float32)
    source_faces = np.asarray(faces, dtype=np.uint32)
    if len(source_faces) <= target_faces:
        return source_vertices, source_faces
    if target_faces < 4:
        raise ValueError(f'Cannot decimate to {target_faces} faces; a closed mesh needs at least 4.')
    bmin = source_vertices.min(axis=0)
    span = max(1e-5, float((source_vertices.max(axis=0) - bmin).max()))
    bins = max(18, int(round(math.sqrt(target_faces * 0.48))))
    best_v = None
    best_f = None
    for _ in range(64):
        q = np.floor(((source_vertices - bmin) / span) * bins).astype(np.int64)
        _, inv, counts = np.unique(q, axis=0, return_inverse=True, return_counts=True)
        new_v = np.zeros((len(counts), 3), dtype=np.float64)
        np.add.at(new_v, inv, source_vertices)
        new_v /= counts[:, None]
        nf = inv[source_faces]
        valid = (nf[:, 0] != nf[:, 1]) & (nf[:, 1] != nf[:, 2]) & (nf[:, 2] != nf[:, 0])
        nf = nf[valid]
        if best_f is None or len(nf) < len(best_f):
            best_v, best_f = new_v.astype(np.float32), nf.astype(np.uint32)
        if len(nf) <= target_faces or bins <= 4:
            return best_v, best_f
        bins = max(4, int(bins * 0.82))
    return best_v, best_f


def bake_photographic_uv_pbr_glb(
    vertices: np.ndarray,
    faces: np.ndarray,
    cutout_rgba_path: Path,
    out_glb_path: Path,
    mobile_glb_path: Path | None = None,
    *,
    texture_size: int = 1536,
    max_mobile_faces: int = MAX_MOBILE_FACES,
) -> dict[str, Any]:
    """Project captured photo `cutout_rgba.png` onto an arbitrary untextured 3D mesh and export GLB(s)."""
    t0 = time.perf_counter()
    v = np.asarray(vertices, dtype=np.float32).copy()
    f = np.asarray(faces, dtype=np.uint32).copy()
    if len(v) == 0 or len(f) == 0:
        raise ValueError('Cannot bake UVs onto empty geometry.')

    # Normalize onto tabletop centered XZ, resting at Y=0
    vmin, vmax = v.min(axis=0), v.max(axis=0)
    span = max(1e-5, float((vmax - vmin).max()))
    v = (v - np.array([0.5 * (vmin[0] + vmax[0]), vmin[1], 0.5 * (vmin[2] + vmax[2])], dtype=np.float32)) * (1.8 / span)

    with Image.open(cutout_rgba_path) as im:
        rgba = ImageOps.exif_transpose(im).convert('RGBA')
    rgba_arr = np.asarray(rgba, dtype=np.uint8)
    ys, xs = np.where(rgba_arr[..., 3] > 20)
    if len(xs):
        u_min, u_max = xs.min() / max(1, rgba_arr.shape[1] - 1), xs.max() / max(1, rgba_arr.shape[1] - 1)
        v_min, v_max = ys.min() / max(1, rgba_arr.shape[0] - 1), ys.max() / max(1, rgba_arr.shape[0] - 1)
        fg_rgb = np.median(rgba_arr[ys, xs, :3], axis=0).astype(np.uint8)
        ceramic_rgb = (int(fg_rgb[0]), int(fg_rgb[1]), int(fg_rgb[2]))
    else:
        u_min, u_max, v_min, v_max = 0.1, 0.9, 0.1, 0.9
        ceramic_rgb = (210, 205, 198)

    atlas_raw, _ = _compose_atlas_with_ceramic_band(cutout_rgba_path, ceramic_rgb, max_tex_dim=texture_size)
    uv_h_scale = (atlas_raw.height - 16) / float(atlas_raw.height)
    swatch_v = (atlas_raw.height - 8) / float(atlas_raw.height)

    def _project_uvs(verts: np.ndarray, tris: np.ndarray) -> np.ndarray:
        norms = _ensure_vec3_normals(verts, tris)
        vx0, vx1 = float(verts[:, 0].min()), float(verts[:, 0].max())
        vz0, vz1 = float(verts[:, 2].min()), float(verts[:, 2].max())
        u = u_min + ((verts[:, 0] - vx0) / max(1e-5, vx1 - vx0)) * (u_max - u_min)
        vv = (v_min + ((verts[:, 2] - vz0) / max(1e-5, vz1 - vz0)) * (v_max - v_min)) * uv_h_scale
        uv = np.stack([np.clip(u, 0.0, 1.0), np.clip(vv, 0.0, uv_h_scale)], axis=1).astype(np.float32)
        # Underside downward normals point to clean ceramic swatch instead of top stretching
        under = norms[:, 1] < -0.25
        uv[under] = np.array([0.5, swatch_v], dtype=np.float32)
        return uv

    uv_raw = _project_uvs(v, f)
    raw_metrics = write_pbr_glb(
        v,
        f,
        uv_raw,
        atlas_raw,
        out_glb_path,
        roughness=0.50,
        extras={'backend': 'photographic-uv-baker'},
    )

    mob_metrics: dict[str, Any] | None = None
    mobile_error: str | None = None
    if mobile_glb_path is not None:
        try:
            vm, fm = _decimate_vertex_cluster(v, f, max_mobile_faces)
            if len(fm) > max_mobile_faces:
                raise ValueError(
                    f'Cannot decimate {len(f)} faces to {max_mobile_faces} by vertex clustering '
                    f'(best {len(fm)}); refusing to export an oversized mobile GLB.'
                )
        except ValueError as exc:
            # A mobile export that cannot honour the face ceiling is omitted and reported, so a
            # raw-resolution mesh is never shipped as a <=35k mobile artifact.
            mobile_error = f'{type(exc).__name__}: {exc}'
            Path(mobile_glb_path).unlink(missing_ok=True)
        else:
            uv_m = _project_uvs(vm, fm)
            atlas_mob, _ = _compose_atlas_with_ceramic_band(cutout_rgba_path, ceramic_rgb, max_tex_dim=1024)
            mob_metrics = write_pbr_glb(
                vm,
                fm,
                uv_m,
                atlas_mob,
                mobile_glb_path,
                roughness=0.50,
                image_format='JPEG',
                extras={'backend': 'photographic-uv-baker-mobile'},
            )

    elapsed = round(time.perf_counter() - t0, 3)
    raw_metrics['elapsed_seconds'] = elapsed
    if mob_metrics is not None:
        mob_metrics['elapsed_seconds'] = elapsed
    return {
        'raw': raw_metrics,
        'mobile': mob_metrics,
        'mobile_error': mobile_error,
        'elapsed_seconds': elapsed,
        'publication_eligible': False,
    }


def infer_moge2_maps(
    image_rgb: Image.Image,
    device: str | None = None,
    model_cache: dict[str, Any] | None = None,
) -> tuple[np.ndarray | None, np.ndarray | None]:
    """Lazy GPU helper for `Ruicheng/moge-2-vitl-normal`; returns `(None, None)` cleanly if offline."""
    try:
        import torch
        from moge.model.v2 import MoGeModel
    except Exception:
        return None, None

    dev = device or ('cuda' if torch.cuda.is_available() else 'cpu')
    model = None if model_cache is None else model_cache.get('moge2')
    if model is None:
        model = MoGeModel.from_pretrained('Ruicheng/moge-2-vitl-normal').to(dev).eval()
        if model_cache is not None:
            model_cache['moge2'] = model

    arr = np.asarray(image_rgb.convert('RGB'), dtype=np.float32) / 255.0
    ten = torch.tensor(arr, dtype=torch.float32, device=dev).permute(2, 0, 1)
    with torch.no_grad():
        out = model.infer(ten)
    depth = (
        _sanitize_depth_map(out['depth'].detach().cpu().numpy())
        if 'depth' in out and out['depth'] is not None else None
    )
    normal = (
        _sanitize_normal_map(out['normal'].detach().cpu().numpy())
        if 'normal' in out and out['normal'] is not None else None
    )
    return depth, normal


def _extract_cutout_from_image_and_mask(src_path: Path, mask_path: Path | None, out_cutout: Path) -> dict[str, Any]:
    """Bind native `_mask.png` if present, else compute corner-background color distance matte."""
    with Image.open(src_path) as im:
        rgb = ImageOps.exif_transpose(im).convert('RGB')
    w, h = rgb.size
    arr = np.asarray(rgb, dtype=np.float32)

    if mask_path is not None and mask_path.is_file():
        with Image.open(mask_path) as mm:
            m = ImageOps.exif_transpose(mm).convert('L')
        if m.size != (w, h):
            m = m.resize((w, h), Image.Resampling.BILINEAR)
        alpha = np.asarray(m, dtype=np.uint8)
        method = 'precomputed_pdf_smask'
        plate_preserved = True
    else:
        # Border color matte fallback for unmasked photos
        pad = max(6, min(w, h) // 25)
        corners = np.concatenate([
            arr[:pad, :pad].reshape(-1, 3),
            arr[:pad, -pad:].reshape(-1, 3),
            arr[-pad:, :pad].reshape(-1, 3),
            arr[-pad:, -pad:].reshape(-1, 3),
        ], axis=0)
        bg = np.median(corners, axis=0)
        dist = np.linalg.norm(arr - bg.reshape(1, 1, 3), axis=2)
        alpha = np.where(dist > 28.0, 255, 0).astype(np.uint8)
        # Keep center safe if low-contrast rim
        if np.mean(alpha > 128) < 0.12:
            yy, xx = np.mgrid[0:h, 0:w]
            ell = (((yy - h / 2) / (0.42 * h)) ** 2 + ((xx - w / 2) / (0.42 * w)) ** 2) <= 1.0
            alpha = np.where(ell, 255, 0).astype(np.uint8)
        method = 'border_color_matte'
        plate_preserved = True

    rgba = np.dstack([arr.astype(np.uint8), alpha])
    Image.fromarray(rgba, mode='RGBA').save(out_cutout, format='PNG', optimize=True)
    cov = round(float(np.mean(alpha > 128) * 100.0), 2)
    return {
        'plate_preserved': plate_preserved,
        'plate_added_pct': 0.0,
        'plate_method': method,
        'fg_coverage_pct': cov,
    }


def run_local_moge2_relief_batch(
    dataset_dir: Path,
    out_dir: Path,
    *,
    v31_results_dir: Path | None = None,
    max_dishes: int = 64,
) -> dict[str, Any]:
    """Run local `moge2-relief` relief+ceramic-plate generation across all dishes & merge v31 TRELLIS outputs."""
    dataset_dir = Path(dataset_dir)
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)

    meta_by_label: dict[str, dict[str, Any]] = {}
    dm_path = dataset_dir / 'dishes_manifest.json'
    if dm_path.is_file():
        for rec in json.loads(dm_path.read_text(encoding='utf-8')).get('dishes', []):
            meta_by_label[rec['label']] = rec

    v31_manifest: dict[str, Any] = {}
    if v31_results_dir is not None and (Path(v31_results_dir) / 'manifest.json').is_file():
        v31_manifest = json.loads((Path(v31_results_dir) / 'manifest.json').read_text(encoding='utf-8'))

    images = sorted(
        (
            p for p in dataset_dir.iterdir()
            if p.is_file() and p.suffix.lower() in ('.png', '.jpg', '.jpeg', '.webp') and not p.stem.endswith('_mask')
        ),
        key=lambda p: p.name,
    )[:max_dishes]

    inputs_manifest: dict[str, Any] = {}
    runs_3d: dict[str, Any] = {}

    for src in images:
        label = src.stem
        dish_dir = out_dir / label
        dish_dir.mkdir(parents=True, exist_ok=True)

        input_dst = dish_dir / STAGED_INPUT_NAME
        staging = stage_source_image(src, input_dst)
        img_size = staging['size']

        cutout_dst = dish_dir / 'cutout_rgba.png'
        v31_in = (v31_manifest.get('inputs') or {}).get(label)
        v31_cut = Path(v31_results_dir) / label / 'cutout_rgba.png' if v31_results_dir else None
        if v31_in and v31_cut and v31_cut.is_file():
            shutil.copy2(v31_cut, cutout_dst)
            cutout_metrics = v31_in.get('cutout_metrics') or {}
        else:
            mask_cand = dataset_dir / f'{label}_mask.png'
            cutout_metrics = _extract_cutout_from_image_and_mask(src, mask_cand if mask_cand.is_file() else None, cutout_dst)

        meta = meta_by_label.get(label, {})
        jm = {
            'job_id': f'rd-{label}',
            'dish_label': label,
            'publication_eligible': False,
            'source_sha256': staging['source_sha256'],
            'source_bytes': staging['source_bytes'],
            'input_filename': staging['input_filename'],
            'input_image_sha256': staging['input_image_sha256'],
            'input_bytes': staging['input_bytes'],
            'input_transformation': staging['input_transformation'],
            'input_byte_identical': staging['input_byte_identical'],
        }
        (dish_dir / 'job_manifest.json').write_text(json.dumps(jm, indent=2), encoding='utf-8')

        inputs_manifest[label] = {
            'label': label,
            'title': meta.get('title') or label.replace('_', ' ').title(),
            'tier': meta.get('tier') or 'custom',
            'page': meta.get('page'),
            'image_source': meta.get('source') or 'user-uploaded-dataset',
            'image_status': 'success',
            'size': img_size,
            'source_sha256': staging['source_sha256'],
            'source_bytes': staging['source_bytes'],
            'input_filename': staging['input_filename'],
            'input_image_sha256': staging['input_image_sha256'],
            'input_bytes': staging['input_bytes'],
            'input_transformation': staging['input_transformation'],
            'input_byte_identical': staging['input_byte_identical'],
            'cutout_metrics': cutout_metrics,
            'artifacts': [
                {'path': f'{label}/input_image.png'},
                {'path': f'{label}/cutout_rgba.png'},
                {'path': f'{label}/job_manifest.json'},
            ],
        }

        # 1) Always build watertight MoGe-2 Relief + Ceramic Plate GLB
        raw_glb = dish_dir / 'moge2_relief.glb'
        mob_glb = dish_dir / 'moge2_relief_mobile.glb'
        moge_res = build_moge2_relief_and_ceramic_plate_glb(cutout_dst, raw_glb, mob_glb)
        runs_3d[f'{label}/moge2-relief'] = {
            'input_label': label,
            'backend': 'MoGe-2 Relief + Ceramic Plate',
            'license': 'MIT',
            'appearance': 'captured_photo_uv_pbr',
            'device': 'cpu-relief-prior',
            'status': 'success',
            'watertight': moge_res['raw']['watertight'],
            'elapsed_seconds': moge_res['elapsed_seconds'],
            'metrics': moge_res['raw'],
            'mobile_metrics': moge_res['mobile'],
            'artifacts': [
                {'path': f'{label}/moge2_relief.glb'},
                {'path': f'{label}/moge2_relief_mobile.glb'},
            ],
        }

        # 2) Preserve existing TRELLIS.2-4B runs from trellislast-version v31 if present
        v31_t_key = f'{label}/trellis2-4b'
        v31_t = (v31_manifest.get('runs_3d') or {}).get(v31_t_key)
        if v31_t and v31_t.get('status') == 'success' and v31_results_dir is not None:
            copied_arts = []
            for art in v31_t.get('artifacts') or []:
                rel = art.get('path')
                if not rel:
                    continue
                src_art = Path(v31_results_dir) / rel
                if src_art.is_file():
                    dst_art = out_dir / rel
                    dst_art.parent.mkdir(parents=True, exist_ok=True)
                    shutil.copy2(src_art, dst_art)
                    copied_arts.append({'path': rel})
            if copied_arts:
                runs_3d[v31_t_key] = {**v31_t, 'license': 'MIT', 'artifacts': copied_arts}

    manifest = {
        'schema_version': 1,
        'demo_label': 'Private single-photo 3D demo (not published T0/T1/T2 menu evidence)',
        'publication_eligible': False,
        'plate_preservation_mode': 'preserve_plate',
        'inputs': inputs_manifest,
        'runs_3d': runs_3d,
    }
    (out_dir / 'manifest.json').write_text(json.dumps(manifest, indent=2), encoding='utf-8')
    return manifest


def purge_hf_and_torch_caches(extra_dirs=()):
    import gc as _gc
    import torch as _torch
    _gc.collect()
    if _torch.cuda.is_available():
        _torch.cuda.empty_cache()
    for target in (HF_CACHE / 'hub', SCRATCH_ROOT / 'torch_cache', *extra_dirs):
        p = Path(target)
        if p.exists():
            shutil.rmtree(p, ignore_errors=True)
    (HF_CACHE / 'hub').mkdir(parents=True, exist_ok=True)
    print('[cleanup] Purged model weights cache; free disk:',
          round(shutil.disk_usage(ROOT).free / (2**30), 2), 'GiB', flush=True)


## 1. Stage 0/1: Dish Discovery & Plate-Preserving Cutouts


In [ ]:
import queue
import signal
import threading
from IPython.display import display, Image as DisplayImage, FileLink
import numpy as np


def run_streaming(cmd, env, prefix, timeout=900):
    tail = []
    timed_out = False
    process = subprocess.Popen(
        [str(v) for v in cmd],
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        errors='replace',
        bufsize=1,
        env=env,
        start_new_session=True,
    )

    def on_timeout():
        nonlocal timed_out
        timed_out = True
        try:
            os.killpg(process.pid, signal.SIGKILL)
        except OSError:
            pass

    timer = threading.Timer(timeout, on_timeout)
    timer.daemon = True
    timer.start()
    try:
        for line in process.stdout:
            text = line.rstrip('\r\n')
            if text:
                print(f'[{prefix}] {text}', flush=True)
                tail.append(text)
                if len(tail) > 40:
                    del tail[:-40]
        ret = process.wait()
    finally:
        timer.cancel()
        if process.stdout is not None:
            process.stdout.close()
    if timed_out:
        raise subprocess.TimeoutExpired(cmd, timeout, output='\n'.join(tail))
    if ret != 0:
        raise RuntimeError(f'{prefix} exited with code {ret}: ' + '\n'.join(tail[-25:]))
    return '\n'.join(tail)


fallback_worker_code = r'''
import json, sys, time
from pathlib import Path
import torch
from diffusers import StableDiffusionPipeline

cfg = json.loads(Path(sys.argv[1]).read_text(encoding='utf-8'))
t0 = time.perf_counter()
print(f"Loading {cfg['repo']}@{cfg['revision'][:8]} (fp16)...", flush=True)
pipe = StableDiffusionPipeline.from_pretrained(
    cfg['repo'],
    revision=cfg['revision'],
    torch_dtype=torch.float16,
    variant='fp16',
    use_safetensors=True,
    safety_checker=None,
    requires_safety_checker=False,
    token=False,
)
if hasattr(pipe, 'enable_attention_slicing'):
    pipe.enable_attention_slicing()
if hasattr(pipe, 'vae') and hasattr(pipe.vae, 'enable_slicing'):
    pipe.vae.enable_slicing()
pipe = pipe.to('cuda')
print(f"Generating {cfg['size']}x{cfg['size']} image ({cfg['steps']} steps, guidance={cfg['guidance']})...", flush=True)
gen = torch.Generator('cuda').manual_seed(int(cfg['seed']))
kwargs = dict(
    prompt=cfg['prompt'],
    num_inference_steps=int(cfg['steps']),
    guidance_scale=float(cfg['guidance']),
    height=int(cfg['size']),
    width=int(cfg['size']),
    generator=gen,
)
if float(cfg['guidance']) > 0:
    kwargs['negative_prompt'] = 'hands, fork, knife, spoon, text, watermark, cropped rim, multiple plates, collage'
try:
    image = pipe(**kwargs).images[0]
except torch.OutOfMemoryError:
    torch.cuda.empty_cache()
    kwargs['height'] = min(512, int(cfg['size']))
    kwargs['width'] = min(512, int(cfg['size']))
    image = pipe(**kwargs).images[0]
extrema = image.convert('RGB').getextrema()
if all(hi <= 5 for _, hi in extrema):
    raise RuntimeError('Generated image is blank/black.')
out = Path(cfg['output_path'])
out.parent.mkdir(parents=True, exist_ok=True)
image.save(out)
stats = {
    'size': list(image.size),
    'elapsed_seconds': round(time.perf_counter() - t0, 2),
    'gpu_name': torch.cuda.get_device_name(0),
}
Path(cfg['stats_path']).write_text(json.dumps(stats), encoding='utf-8')
print(f"Done in {stats['elapsed_seconds']}s on {stats['gpu_name']}", flush=True)
'''
fallback_worker = SCRATCH_ROOT / 'run_fallback_image.py'
fallback_worker.write_text(fallback_worker_code, encoding='utf-8')

image_results = discover_user_dish_images(RESULTS_DIR)
user_ok = [k for k, v in image_results.items() if v.get('image_status') == 'success']

if user_ok:
    print(f'[stage0] Found {len(user_ok)} user dish photo(s): {user_ok}')
else:
    image_results = collect_openrouter_images(RESULTS_DIR, kaggle_secret('OPENROUTER_API_KEY'))
    for label, entry in image_results.items():
        print('OpenRouter:', entry['model'], entry.get('image_error', entry['image_status']))
    openrouter_ok = [k for k, v in image_results.items() if v['image_status'] == 'success']
    if len(openrouter_ok) < 2:
        needed = 2 - len(openrouter_ok)
        fallback_items = list(FALLBACK_IMAGE_MODELS.items())[:needed]
        print(f'Running {needed} ungated open-weights image fallback model(s) with MAX_GPU_WORKERS={MAX_GPU_WORKERS}...')
        fb_gpu_queue = queue.Queue()
        for gid in range(MAX_GPU_WORKERS):
            fb_gpu_queue.put(gid)

        def run_fallback_image(item):
            label, spec = item
            gpu_idx = fb_gpu_queue.get()
            folder = RESULTS_DIR / label
            folder.mkdir(parents=True, exist_ok=True)
            out_path = folder / 'input_image.png'
            stats_path = SCRATCH_ROOT / f'img_stats_{label}.json'
            cfg_path = SCRATCH_ROOT / f'img_cfg_{label}.json'
            cfg_path.write_text(json.dumps({
                **spec,
                'label': label,
                'prompt': DISH_PROMPT,
                'output_path': str(out_path),
                'stats_path': str(stats_path),
            }), encoding='utf-8')
            env = {k: v for k, v in os.environ.items() if k not in ('OPENROUTER_API_KEY', 'HF_TOKEN')}
            env['CUDA_VISIBLE_DEVICES'] = str(gpu_idx)
            entry = {
                'label': label,
                'model': spec['repo'],
                'revision': spec['revision'],
                'image_source': 'kaggle-open-weights-fallback',
                'directory': folder,
            }
            try:
                run_streaming([PYTHON, fallback_worker, cfg_path], env=env, prefix=f'fallback:{label}', timeout=900)
                stats = json.loads(stats_path.read_text(encoding='utf-8'))
                entry.update(image_status='success', image_path=out_path, **stats)
            except (subprocess.SubprocessError, RuntimeError, OSError, ValueError) as exc:
                print(f'Fallback image {label} failed:', str(exc)[-1200:])
                entry.update(image_status='failed', image_error=type(exc).__name__)
            finally:
                fb_gpu_queue.put(gpu_idx)
            return label, entry

        with concurrent.futures.ThreadPoolExecutor(max_workers=MAX_GPU_WORKERS) as pool:
            for label, entry in pool.map(run_fallback_image, fallback_items):
                image_results[label] = entry

ACTIVE_LABELS = [k for k, v in image_results.items() if v.get('image_status') == 'success'][:MAX_USER_DISHES]
if not ACTIVE_LABELS:
    raise RuntimeError('No dish input images succeeded from Stage 0 dataset, OpenRouter, or open-weights fallback.')

cutout_worker_code = r'''
import inspect, json, sys, time
from pathlib import Path
import numpy as np
import torch
from PIL import Image, ImageOps
from scipy.ndimage import binary_dilation, binary_fill_holes, gaussian_filter, label as nd_label
import rembg

payload = json.loads(Path(sys.argv[1]).read_text(encoding='utf-8'))
jobs = payload['jobs']
stats_out_path = payload.get('stats_out')
dino_repo = payload.get('grounding_dino_repo', 'IDEA-Research/grounding-dino-tiny')
dino_rev = payload.get('grounding_dino_revision')
sam_repo = payload.get('sam_repo', 'facebook/sam-vit-base')
sam_rev = payload.get('sam_revision')
plate_mode = payload.get('plate_mode', 'preserve_plate')

MODEL_ORDER = ('birefnet-general-lite', 'isnet-general-use', 'u2net')
PLATE_PROMPT_PASSES = [
    (
        'a plate. a bowl. a tray. a pan. a platter. a dish. a saucer.',
        [['a plate', 'a bowl', 'a tray', 'a pan', 'a platter', 'a dish', 'a saucer']],
    ),
    (
        'a cutting board. a wooden board. a serving board. a ceramic bowl. a metal tray. a baking pan.',
        [['a cutting board', 'a wooden board', 'a serving board', 'a ceramic bowl', 'a metal tray', 'a baking pan']],
    ),
]
rembg_sessions = {}
plate_models = {}


def get_rembg_session(model_name):
    if model_name not in rembg_sessions:
        print(f'Initializing rembg session: {model_name} (CPU)...', flush=True)
        rembg_sessions[model_name] = rembg.new_session(model_name, providers=['CPUExecutionProvider'])
    return rembg_sessions[model_name]


def get_plate_models():
    if not plate_models:
        from transformers import (
            AutoModelForZeroShotObjectDetection,
            AutoProcessor,
            SamModel,
            SamProcessor,
        )
        device = 'cuda:0' if torch.cuda.is_available() else 'cpu'
        print(f'Loading GroundingDINO@{str(dino_rev)[:8]} + SAM@{str(sam_rev)[:8]} on {device}...', flush=True)
        dino_proc = AutoProcessor.from_pretrained(dino_repo, revision=dino_rev, token=False)
        dino_mod = AutoModelForZeroShotObjectDetection.from_pretrained(
            dino_repo, revision=dino_rev, token=False
        ).eval().to(device)
        sam_proc = SamProcessor.from_pretrained(sam_repo, revision=sam_rev, token=False)
        sam_mod = SamModel.from_pretrained(
            sam_repo, revision=sam_rev, token=False
        ).eval().to(device)
        plate_models.update(
            device=device,
            dino_proc=dino_proc,
            dino_mod=dino_mod,
            sam_proc=sam_proc,
            sam_mod=sam_mod,
        )
    return plate_models


def extract_food_alpha(rgb, label_name):
    last_err = None
    for model_name in MODEL_ORDER:
        try:
            sess = get_rembg_session(model_name)
            rgba = rembg.remove(rgb, session=sess)
            arr = np.array(rgba.convert('RGBA'), dtype=np.uint8)
            alpha = arr[:, :, 3]
            raw_mask = alpha > 16
            filled = binary_fill_holes(raw_mask)
            labeled, num_features = nd_label(filled)
            if num_features > 1:
                counts = np.bincount(labeled.ravel())[1:]
                max_area = int(counts.max())
                keep = np.zeros(num_features + 1, dtype=bool)
                for cid, area in enumerate(counts, start=1):
                    if area >= max(64, int(0.005 * max_area)):
                        keep[cid] = True
                clean_mask = keep[labeled]
            else:
                clean_mask = filled
            fg_check = clean_mask & (alpha >= 128)
            ys, xs = np.where(fg_check)
            if len(ys) == 0:
                ys, xs = np.where(clean_mask)
            if len(ys) == 0 or (xs.max() - xs.min()) < 8 or (ys.max() - ys.min()) < 8:
                raise RuntimeError(f'{model_name} produced empty food foreground')
            food_alpha = np.where(clean_mask, np.where(clean_mask & (alpha <= 16), 255, alpha), 0).astype(np.uint8)
            return food_alpha, clean_mask, model_name, int(num_features)
        except Exception as exc:
            last_err = exc
            print(f'rembg {model_name} fallback on {label_name}: {type(exc).__name__}: {exc}', flush=True)
    raise RuntimeError(f'All rembg models failed for {label_name}: {last_err}')


def _corner_block_pair(h, w, patch=8):
    p_h = min(patch, max(1, h // 8))
    p_w = min(patch, max(1, w // 8))
    ins = min(max(8, int(round(0.035 * min(h, w)))), max(0, min(h, w) // 4))
    p_in = min(max(patch, int(round(0.04 * min(h, w)))), max(1, min(h, w) // 4))
    pairs = (
        ((slice(0, p_h), slice(0, p_w)), (slice(ins, ins + p_in), slice(ins, ins + p_in))),
        ((slice(0, p_h), slice(w - p_w, w)), (slice(ins, ins + p_in), slice(w - ins - p_in, w - ins))),
        ((slice(h - p_h, h), slice(0, p_w)), (slice(h - ins - p_in, h - ins), slice(ins, ins + p_in))),
        ((slice(h - p_h, h), slice(w - p_w, w)), (slice(h - ins - p_in, h - ins), slice(w - ins - p_in, w - ins))),
    )
    return pairs


def count_non_food_corners(mask, food_mask, patch=8):
    h, w = mask.shape
    non_food = mask & (~food_mask)
    hits = 0
    for (oy, ox), (iy, ix) in _corner_block_pair(h, w, patch):
        if float(non_food[oy, ox].mean()) > 0.50 or float(non_food[iy, ix].mean()) > 0.55:
            hits += 1
    return hits


def count_top_non_food_corners(mask, food_mask, patch=8):
    h, w = mask.shape
    non_food = mask & (~food_mask)
    hits = 0
    for (oy, ox), (iy, ix) in _corner_block_pair(h, w, patch)[:2]:
        if float(non_food[oy, ox].mean()) > 0.50 or float(non_food[iy, ix].mean()) > 0.55:
            hits += 1
    return hits


def _box_iou(a, b):
    ix1, iy1 = max(a[0], b[0]), max(a[1], b[1])
    ix2, iy2 = min(a[2], b[2]), min(a[3], b[3])
    iw, ih = max(0.0, ix2 - ix1), max(0.0, iy2 - iy1)
    inter = iw * ih
    if inter <= 0.0:
        return 0.0
    area_a = max(0.0, (a[2] - a[0]) * (a[3] - a[1]))
    area_b = max(0.0, (b[2] - b[0]) * (b[3] - b[1]))
    return inter / max(area_a + area_b - inter, 1e-6)


def detect_plate_mask(rgb, food_mask, label_name='dish'):
    w, h = rgb.size
    models = get_plate_models()
    device = models['device']
    dino_proc = models['dino_proc']
    dino_mod = models['dino_mod']
    sam_proc = models['sam_proc']
    sam_mod = models['sam_mod']

    fys, fxs = np.where(food_mask)
    fx1, fy1 = int(fxs.min()), int(fys.min())
    fx2, fy2 = int(fxs.max()), int(fys.max())
    fw = max(fx2 - fx1 + 1, 1)
    fh = max(fy2 - fy1 + 1, 1)
    fcx = float(fxs.mean())
    fcy = float(fys.mean())
    food_sum = max(int(food_mask.sum()), 1)
    food_ring = binary_dilation(food_mask, iterations=max(4, int(round(0.015 * min(h, w))))) & (~food_mask)

    sig_params = inspect.signature(dino_proc.post_process_grounded_object_detection).parameters
    scored_boxes = []
    for prompt_text, prompt_labels in PLATE_PROMPT_PASSES:
        dino_inputs = dino_proc(images=rgb, text=prompt_text, return_tensors='pt').to(device)
        with torch.inference_mode():
            dino_out = dino_mod(**dino_inputs)
        pp_kwargs = {'target_sizes': [(h, w)]}
        if 'threshold' in sig_params:
            pp_kwargs['threshold'] = 0.15
        elif 'box_threshold' in sig_params:
            pp_kwargs['box_threshold'] = 0.15
        if 'text_threshold' in sig_params:
            pp_kwargs['text_threshold'] = 0.13
        if 'text_labels' in sig_params:
            pp_kwargs['text_labels'] = prompt_labels
        det = dino_proc.post_process_grounded_object_detection(
            dino_out, input_ids=dino_inputs['input_ids'], **pp_kwargs
        )[0]
        raw_boxes = det['boxes'].detach().cpu().tolist()
        raw_scores = det['scores'].detach().cpu().tolist()
        for box, sc in zip(raw_boxes, raw_scores):
            x1, y1, x2, y2 = [float(v) for v in box]
            bx1, by1 = max(0, int(np.floor(x1))), max(0, int(np.floor(y1)))
            bx2, by2 = min(w, int(np.ceil(x2))), min(h, int(np.ceil(y2)))
            if bx2 - bx1 < 16 or by2 - by1 < 16:
                continue
            overlap = float(food_mask[by1:by2, bx1:bx2].sum()) / food_sum
            contains_center = (x1 <= fcx <= x2) and (y1 <= fcy <= y2)
            if overlap >= 0.25 or contains_center:
                box_area_frac = ((bx2 - bx1) * (by2 - by1)) / float(h * w)
                b_sc = float(sc) + 0.30 * overlap + (0.12 if box_area_frac > food_mask.mean() else 0.0) - 0.15 * max(0.0, box_area_frac - 0.88)
                scored_boxes.append((b_sc, [x1, y1, x2, y2]))

    scored_boxes.sort(key=lambda pair: pair[0], reverse=True)
    selected_boxes = []
    for _, cand_box in scored_boxes:
        if all(_box_iou(cand_box, kept) <= 0.90 for kept in selected_boxes):
            selected_boxes.append(cand_box)
        if len(selected_boxes) >= 6:
            break

    candidates = []

    def evaluate_sam_masks(masks_batch, iou_batch, source_tag):
        for k_idx in range(masks_batch.shape[0]):
            for m_idx in range(masks_batch.shape[1]):
                m = masks_batch[k_idx, m_idx] > 0.0
                if not m.any():
                    continue
                if float(m.mean()) > 0.92:
                    continue
                if count_non_food_corners(m, food_mask) >= 2 or count_top_non_food_corners(m, food_mask) >= 2:
                    continue
                if float((m[0, :] & (~food_mask[0, :])).mean()) > 0.50:
                    continue
                if float((m[:, 0] & (~food_mask[:, 0])).mean()) > 0.72 and float((m[:, -1] & (~food_mask[:, -1])).mean()) > 0.72:
                    continue
                iou_sc = float(iou_batch[k_idx, m_idx])
                u_raw = binary_fill_holes(m | food_mask)
                lbl_u, n_u = nd_label(u_raw)
                if n_u > 1:
                    keep_u = np.zeros(n_u + 1, dtype=bool)
                    for cid in range(1, n_u + 1):
                        comp = lbl_u == cid
                        if (comp & food_mask).any():
                            keep_u[cid] = True
                    u = keep_u[lbl_u]
                else:
                    u = u_raw
                plate_only = u & (~food_mask)
                plate_gain = float(plate_only.sum()) / float(h * w)
                union_area_frac = float(u.mean())
                u_corners = count_non_food_corners(u, food_mask)
                u_top_corners = count_top_non_food_corners(u, food_mask)
                u_top_edge = float((u[0, :] & (~food_mask[0, :])).mean())
                ring_contact = float((plate_only & food_ring).sum()) / max(int(food_ring.sum()), 1)
                raw_touch = float((binary_dilation(m, iterations=3) & food_mask).sum()) / food_sum
                u_ys, u_xs = np.where(u)
                if len(u_ys) == 0:
                    continue
                bbox_fill = float(u.sum()) / max(float((u_xs.max() - u_xs.min() + 1) * (u_ys.max() - u_ys.min() + 1)), 1.0)
                u_left_edge = float((u[:, 0] & (~food_mask[:, 0])).mean())
                u_right_edge = float((u[:, -1] & (~food_mask[:, -1])).mean())
                if (
                    u_corners >= 2
                    or u_top_corners >= 2
                    or u_top_edge > 0.55
                    or (u_left_edge > 0.72 and u_right_edge > 0.72)
                    or union_area_frac > 0.92
                    or plate_gain < 0.018
                    or ring_contact < 0.25
                    or raw_touch < 0.10
                    or bbox_fill < 0.48
                ):
                    continue
                score = (
                    0.70 * iou_sc
                    + 0.85 * ring_contact
                    + 0.70 * bbox_fill
                    + min(plate_gain, 0.38) * 2.2
                    + (0.10 if source_tag == 'grounding_dino_sam' else 0.0)
                    - 0.55 * u_corners
                    - 0.30 * max(0.0, union_area_frac - 0.85)
                )
                meta = {
                    'iou': round(iou_sc, 3),
                    'gain': round(plate_gain, 3),
                    'ring': round(ring_contact, 3),
                    'fill': round(bbox_fill, 3),
                    'area': round(union_area_frac, 3),
                }
                candidates.append((score, source_tag, u_corners, u, meta))

    def run_sam_point_prompt(pos_pts, neg_pts, source_tag):
        if not pos_pts:
            return
        pts = pos_pts + neg_pts
        lbls = [1] * len(pos_pts) + [0] * len(neg_pts)
        sam_pt_inputs = sam_proc(images=rgb, input_points=[pts], input_labels=[lbls], return_tensors='pt').to(device)
        with torch.inference_mode():
            sam_pt_out = sam_mod(**sam_pt_inputs, multimask_output=True)
        pt_post = sam_proc.image_processor.post_process_masks(
            sam_pt_out.pred_masks.cpu(),
            sam_pt_inputs['original_sizes'].cpu(),
            sam_pt_inputs['reshaped_input_sizes'].cpu(),
        )[0].numpy()
        pt_ious = sam_pt_out.iou_scores[0].detach().cpu().numpy()
        evaluate_sam_masks(pt_post, pt_ious, source_tag)

    # 1. GroundingDINO boxes -> SAM
    if selected_boxes:
        sam_inputs = sam_proc(images=rgb, input_boxes=[selected_boxes], return_tensors='pt').to(device)
        with torch.inference_mode():
            sam_out = sam_mod(**sam_inputs, multimask_output=True)
        post = sam_proc.image_processor.post_process_masks(
            sam_out.pred_masks.cpu(),
            sam_inputs['original_sizes'].cpu(),
            sam_inputs['reshaped_input_sizes'].cpu(),
        )[0].numpy()
        ious = sam_out.iou_scores[0].detach().cpu().numpy()
        evaluate_sam_masks(post, ious, 'grounding_dino_sam')

    pad_c = 4
    ins_c = min(max(12, int(round(0.035 * min(h, w)))), max(1, min(h, w) // 4))
    corner_negs = []
    for cx, cy in (
        (pad_c, pad_c), (w - 1 - pad_c, pad_c),
        (pad_c, h - 1 - pad_c), (w - 1 - pad_c, h - 1 - pad_c),
        (ins_c, ins_c), (w - 1 - ins_c, ins_c),
    ):
        ix, iy = int(np.clip(cx, 0, w - 1)), int(np.clip(cy, 0, h - 1))
        if not food_mask[iy, ix]:
            corner_negs.append([float(ix), float(iy)])

    # 2. Directional lower/lateral support-ring SAM prompts (avoids protruding top garnish)
    r_in = max(4, int(round(0.018 * min(h, w))))
    r_out = max(14, int(round(0.075 * min(h, w))))
    ring_band = binary_dilation(food_mask, iterations=r_out) & (~binary_dilation(food_mask, iterations=r_in))
    y_floor = min(h - 9, fy1 + int(round(0.25 * fh)))
    valid_ring = ring_band.copy()
    valid_ring[:max(8, y_floor), :] = False
    valid_ring[max(8, h - 8):, :] = False
    valid_ring[:, :8] = False
    valid_ring[:, max(8, w - 8):] = False
    ry, rx = np.where(valid_ring)
    if len(ry) >= 4:
        angles = np.arctan2(ry.astype(np.float64) - fcy, rx.astype(np.float64) - fcx)
        order = np.argsort(angles)
        picks = order[np.linspace(0, len(order) - 1, min(6, len(order)), dtype=int)]
        ring_pos = [[float(rx[idx]), float(ry[idx])] for idx in picks]
        run_sam_point_prompt(ring_pos, corner_negs, 'sam_ring_points')
        interior_pos = [[float(int(np.clip(fcx, 2, w - 3))), float(int(np.clip(fcy, 2, h - 3)))]]
        low_y = int(np.clip(fy1 + int(round(0.65 * fh)), 2, h - 3))
        low_xs = np.where(food_mask[low_y])[0]
        if len(low_xs) > 0:
            interior_pos.append([float(int(low_xs.mean())), float(low_y)])
        else:
            interior_pos.append([float(int(np.clip(fcx, 2, w - 3))), float(low_y)])
        run_sam_point_prompt(ring_pos + interior_pos, corner_negs, 'sam_ring_points')

    # 3. Direct vessel-surface single-click & pair-click probes around lower/lateral perimeter
    vessel_pts = []
    step_lat = max(10, int(round(0.055 * min(h, w))))
    pt_bx = int(np.clip(round(fcx), 8, w - 9))
    pt_by = int(np.clip(fy2 + max(8, int(round(0.045 * h))), 8, h - 9))
    if not food_mask[pt_by, pt_bx]:
        vessel_pts.append([float(pt_bx), float(pt_by)])

    yr = int(np.clip(fy1 + int(round(0.55 * fh)), 8, h - 9))
    rxs = np.where(food_mask[yr])[0]
    if len(rxs) > 0:
        px = min(w - 10, int(rxs.max()) + step_lat)
        while px < w - 8 and food_mask[yr, px]:
            px += 8
        if 8 <= px <= w - 9 and not food_mask[yr, px]:
            vessel_pts.append([float(px), float(yr)])

    yl = int(np.clip(fy1 + int(round(0.65 * fh)), 8, h - 9))
    lxs = np.where(food_mask[yl])[0]
    if len(lxs) > 0:
        px = max(9, int(lxs.min()) - step_lat)
        while px > 8 and food_mask[yl, px]:
            px -= 8
        if 8 <= px <= w - 9 and not food_mask[yl, px]:
            vessel_pts.append([float(px), float(yl)])

    for vp in vessel_pts:
        run_sam_point_prompt([vp], corner_negs, 'sam_vessel_points')
    if len(vessel_pts) >= 2:
        run_sam_point_prompt(vessel_pts[:2], corner_negs, 'sam_vessel_points')

    if not candidates:
        print(f'Plate candidates [{label_name}]: none passed guards (boxes={len(selected_boxes)})', flush=True)
        return None, 'food_only_fallback', 0, len(selected_boxes)

    candidates.sort(key=lambda item: item[0], reverse=True)
    top3 = [f'{src}:{round(sc, 3)}({meta})' for sc, src, _, _, meta in candidates[:3]]
    print(f'Plate candidates [{label_name}] boxes={len(selected_boxes)} top3={top3}', flush=True)
    _, method, corners_hit, best_mask, _ = candidates[0]
    return best_mask, method, int(corners_hit), len(selected_boxes)


all_stats = {}
for job in jobs:
    t0 = time.perf_counter()
    with Image.open(job['src']) as im:
        rgb = ImageOps.exif_transpose(im).convert('RGB')
    w, h = rgb.size
    rgb_arr = np.array(rgb, dtype=np.uint8)

    reviewed_path = job.get('reviewed_mask')
    if reviewed_path and Path(reviewed_path).is_file():
        with Image.open(reviewed_path) as rm:
            mask_l = ImageOps.exif_transpose(rm).convert('L').resize((w, h), Image.Resampling.BILINEAR)
        new_alpha = np.array(mask_l, dtype=np.uint8)
        clean_mask = new_alpha >= 128
        food_coverage_pct = round(float(clean_mask.mean()) * 100.0, 2)
        plate_preserved = True
        plate_method = 'reviewed_mask'
        plate_added_pct = 0.0
        corners_hit = count_non_food_corners(clean_mask, clean_mask)
        det_boxes = 0
        used_model = 'reviewed-full-plate-mask'
        comps_before = 1
    else:
        food_alpha, food_mask, used_model, comps_before = extract_food_alpha(rgb, job['label'])
        food_coverage_pct = round(float((food_alpha >= 128).mean()) * 100.0, 2)
        plate_mask = None
        plate_method = 'food_only'
        corners_hit = 0
        det_boxes = 0
        if plate_mode == 'preserve_plate':
            try:
                plate_mask, plate_method, corners_hit, det_boxes = detect_plate_mask(rgb, food_mask, job['label'])
            except Exception as exc:
                print(f'Plate segmentation warning on {job["label"]}: {type(exc).__name__}: {exc}', flush=True)
                plate_mask, plate_method = None, 'food_only_fallback'

        if plate_mask is not None:
            union_raw = binary_fill_holes(food_mask | plate_mask)
            labeled_u, n_u = nd_label(union_raw)
            if n_u > 1:
                keep_u = np.zeros(n_u + 1, dtype=bool)
                for cid in range(1, n_u + 1):
                    comp = labeled_u == cid
                    if (comp & food_mask).any():
                        keep_u[cid] = True
                clean_mask = keep_u[labeled_u]
            else:
                clean_mask = union_raw
            plate_only = clean_mask & (~food_mask)
            plate_edge = gaussian_filter(clean_mask.astype(np.float32), sigma=1.0)
            plate_alpha = np.where(clean_mask, np.clip(np.round(plate_edge * 255.0), 210, 255), 0).astype(np.uint8)
            new_alpha = np.where(food_mask, food_alpha, np.where(plate_only, plate_alpha, 0)).astype(np.uint8)
            plate_added_pct = round(float((plate_only & (new_alpha >= 128)).mean()) * 100.0, 2)
            plate_preserved = plate_added_pct > 0.0
        else:
            clean_mask = food_mask
            new_alpha = food_alpha
            plate_added_pct = 0.0
            plate_preserved = False

    fg_bin = new_alpha >= 128
    ys, xs = np.where(fg_bin)
    if len(ys) == 0:
        ys, xs = np.where(new_alpha > 16)
    if len(ys) == 0 or (xs.max() - xs.min()) < 8 or (ys.max() - ys.min()) < 8:
        raise RuntimeError(f"Cutout produced empty foreground for {job['label']}")

    rgba_arr = np.dstack([rgb_arr, new_alpha]).astype(np.uint8)
    rgba_arr[new_alpha == 0, :3] = 0
    clean_rgba = Image.fromarray(rgba_arr, mode='RGBA')
    clean_rgba.save(job['dst'])

    _, comps_after = nd_label(new_alpha > 16)
    edge_sides = []
    if bool((new_alpha[:, 0] >= 128).any()):
        edge_sides.append('left')
    if bool((new_alpha[:, -1] >= 128).any()):
        edge_sides.append('right')
    if bool((new_alpha[0, :] >= 128).any()):
        edge_sides.append('top')
    if bool((new_alpha[-1, :] >= 128).any()):
        edge_sides.append('bottom')
    bw = int(xs.max() - xs.min() + 1)
    bh = int(ys.max() - ys.min() + 1)

    job_stats = {
        'cutout_model': used_model,
        'plate_mode': plate_mode,
        'plate_preserved': bool(plate_preserved),
        'plate_method': plate_method,
        'food_coverage_pct': food_coverage_pct,
        'plate_added_pct': plate_added_pct,
        'fg_coverage_pct': round(float(fg_bin.mean()) * 100.0, 2),
        'bbox_aspect': round(bw / max(bh, 1), 3),
        'bbox': [int(xs.min()), int(ys.min()), int(xs.max() + 1), int(ys.max() + 1)],
        'edge_touch_sides': edge_sides,
        'corners_hit': int(corners_hit),
        'detected_plate_boxes': int(det_boxes),
        'components_before': int(comps_before),
        'components_after': int(comps_after),
        'elapsed_seconds': round(time.perf_counter() - t0, 2),
    }
    all_stats[job['label']] = job_stats
    print(
        f"Cutout [{job['label']}] model={used_model} plate={plate_method}({plate_preserved}) "
        f"food={food_coverage_pct}% +plate={plate_added_pct}% -> fg={job_stats['fg_coverage_pct']}% "
        f"corners={corners_hit} edges={edge_sides} ({job_stats['elapsed_seconds']}s)",
        flush=True,
    )

plate_models.clear()
rembg_sessions.clear()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

if stats_out_path:
    Path(stats_out_path).write_text(json.dumps(all_stats, indent=2), encoding='utf-8')
'''
cutout_worker = SCRATCH_ROOT / 'run_rembg_cutout.py'
cutout_worker.write_text(cutout_worker_code, encoding='utf-8')
cutout_jobs = [
    {
        'label': label,
        'src': str(image_results[label]['image_path']),
        'dst': str(Path(image_results[label]['image_path']).parent / 'cutout_rgba.png'),
        'reviewed_mask': str(image_results[label]['reviewed_mask_path']) if image_results[label].get('reviewed_mask_path') else None,
    }
    for label in ACTIVE_LABELS
]
cutout_stats_path = SCRATCH_ROOT / 'cutout_stats.json'
cutout_cfg = SCRATCH_ROOT / 'cutout_jobs.json'
cutout_cfg.write_text(json.dumps({
    'jobs': cutout_jobs,
    'stats_out': str(cutout_stats_path),
    'grounding_dino_repo': GROUNDING_DINO_REPO,
    'grounding_dino_revision': GROUNDING_DINO_REVISION,
    'sam_repo': SAM_REPO,
    'sam_revision': SAM_REVISION,
    'plate_mode': PLATE_PRESERVATION_MODE,
}), encoding='utf-8')
clean_env = {k: v for k, v in os.environ.items() if k not in ('OPENROUTER_API_KEY', 'HF_TOKEN')}
clean_env['CUDA_VISIBLE_DEVICES'] = '0'
run_streaming([PYTHON, cutout_worker, cutout_cfg], env=clean_env, prefix='rembg', timeout=600)
cutout_stats = json.loads(cutout_stats_path.read_text(encoding='utf-8'))

for job in cutout_jobs:
    label = job['label']
    entry = image_results[label]
    entry['cutout_path'] = Path(job['dst'])
    entry['cutout_metrics'] = cutout_stats[label]
    REVISIONS[f'input_{label}'] = entry.get('revision', entry['image_source'])
    REVISIONS[f'cutout_{label}'] = f"{cutout_stats[label]['cutout_model']}+{cutout_stats[label]['plate_method']}"
    jm = build_job_manifest(label, entry, REVISIONS)
    jm_path = Path(entry['image_path']).parent / 'job_manifest.json'
    jm_path.write_text(json.dumps(jm, indent=2) + '\n', encoding='utf-8')
    entry['job_manifest_path'] = jm_path
    print(f"Active dish [{label}] job_id={jm['job_id']} cutout={entry['cutout_metrics']}")
    display(DisplayImage(filename=job['src'], width=280))
    display(DisplayImage(filename=job['dst'], width=280))


## 2. Option 1 — `MoGe-2` Depth/Normal Relief + Parametric Ceramic Plate


In [ ]:
# Benchmark Option 1: MoGe-2 (Ruicheng/moge-2-vitl-normal, MIT) 2.5D/3D Relief + Parametric Ceramic Plate
if 'RUNS_3D' not in globals():
    RUNS_3D = {}

MOGE2_REPO = 'Ruicheng/moge-2-vitl-normal'
MOGE2_REVISION = 'cb0e8bbd6b1e243589717c78e750b1ba4c093acf'
MOGE_CODE_REV = '74fbce054ebed49800de42d0ad0e83495065719a'
UTILS3D_MOGE_REV = '62f09d58509485564e24d5d9f6aac9ee9ebc0c37'
REVISIONS['moge2'] = MOGE2_REVISION

if 'moge2-relief' in ENABLED_MODELS:
    print(f'[moge2-relief] Installing/loading {MOGE2_REPO}@{MOGE2_REVISION[:8]}...', flush=True)
    moge_model = None
    try:
        try:
            from moge.model import import_model_class_by_version
        except ImportError:
            _moge_env = os.environ.copy()
            for _s in ('HF_TOKEN', 'OPENROUTER_API_KEY'):
                _moge_env.pop(_s, None)
            # MoGe pins `utils3d-moge`; --no-deps keeps pip from dragging gradio>=6.0 or a
            # conflicting PyPI `utils3d` into the Kaggle torch/CUDA stack.
            run([PYTHON, '-m', 'pip', 'install', '-q', '--no-deps',
                 f'git+https://github.com/EasternJournalist/utils3d-moge.git@{UTILS3D_MOGE_REV}',
                 f'git+https://github.com/microsoft/MoGe.git@{MOGE_CODE_REV}'], env=_moge_env)
            from moge.model import import_model_class_by_version
        _MoGeV2 = import_model_class_by_version('v2')
        moge_model = _MoGeV2.from_pretrained(MOGE2_REPO, revision=MOGE2_REVISION).to('cuda').eval()
        print('[moge2-relief] Loaded MoGe-2 ViT-L Normal on CUDA.', flush=True)
    except Exception as _moge_err:
        print(f'[moge2-relief] Live MoGe-2 weights unavailable ({type(_moge_err).__name__}: {_moge_err}); using geometric relief fallback.', flush=True)

    for label in ACTIVE_LABELS:
        in_entry = image_results[label]
        key = f'{label}/moge2-relief'
        rec = {
            'input_label': label,
            'image_source': in_entry['image_source'],
            'image_model': in_entry.get('model', 'user-photo-upload'),
            'backend': 'MoGe-2 Relief + Ceramic Plate',
            'license': 'MIT',
            'appearance': 'captured_photo_uv_pbr',
            'artifact_paths': [],
        }
        try:
            t0 = time.perf_counter()
            cutout_p = Path(in_entry['cutout_path'])
            depth_map, normal_map = None, None
            if moge_model is not None:
                with Image.open(cutout_p) as _cim:
                    _rgb = ImageOps.exif_transpose(_cim).convert('RGB')
                    _arr = np.asarray(_rgb, dtype=np.float32) / 255.0
                _ten = torch.tensor(_arr, dtype=torch.float32, device='cuda').permute(2, 0, 1)
                with torch.no_grad():
                    _out = moge_model.infer(_ten)
                if 'depth' in _out and _out['depth'] is not None:
                    depth_map = _out['depth'].detach().cpu().numpy()
                if 'normal' in _out and _out['normal'] is not None:
                    normal_map = _sanitize_normal_map(_out['normal'].detach().cpu().numpy())
            raw_glb = RESULTS_DIR / label / 'moge2_relief.glb'
            mob_glb = RESULTS_DIR / label / 'moge2_relief_mobile.glb'
            res = build_moge2_relief_and_ceramic_plate_glb(
                cutout_p, raw_glb, mob_glb, depth_map=depth_map, normal_map=normal_map,
            )
            elapsed = round(time.perf_counter() - t0, 2)
            glb_meta = validate_glb(raw_glb)
            mob_meta = validate_glb(mob_glb)
            res['raw']['elapsed_seconds'] = elapsed
            res['mobile']['elapsed_seconds'] = elapsed
            rec.update(
                status='success',
                device='cuda:0' if moge_model is not None else 'cpu-fallback',
                watertight=res['raw']['watertight'],
                elapsed_seconds=elapsed,
                metrics={**res['raw'], **glb_meta},
                mobile_metrics={**res['mobile'], **mob_meta},
                mobile_glb_path=mob_glb,
                mobile_status='available',
                artifact_paths=[raw_glb, mob_glb],
            )
            print(f'[moge2-relief] {label}: {elapsed}s, faces={res["raw"]["faces"]} (mob={res["mobile"]["faces"]}), watertight={res["raw"]["watertight"]}', flush=True)
        except Exception as _exc:
            rec.update(status='failed', error=f'{type(_exc).__name__}: {_exc}')
            print(f'[moge2-relief] {label} FAILED:', _exc, flush=True)
        RUNS_3D[key] = rec

    del moge_model
    purge_hf_and_torch_caches()


## 3. Option 3 — `TripoSG` (1.5B Flow SDF DiT) + Photo UV PBR


In [ ]:
# Benchmark Option 3: VAST AI TripoSG (1.5B Rectified Flow SDF DiT, MIT) + Photo UV PBR Baker (Bypassing NC RMBG-1.4)
TRIPOSG_REPO = 'VAST-AI/TripoSG'
TRIPOSG_REVISION = '2c1c516d22d58db486a058d98d31bb6177344e06'
TRIPOSG_CODE_REV = 'fc5c40990181e2a756c4e0b1c2f4d6b5202faf8c'
REVISIONS['triposg'] = TRIPOSG_REVISION

if 'triposg' in ENABLED_MODELS:
    triposg_src = SCRATCH_ROOT / 'TripoSG-src'
    triposg_weights = SCRATCH_ROOT / 'TripoSG-weights'
    triposg_pipe = None
    triposg_setup_err = None
    _has_diso = False
    try:
        if not triposg_src.exists():
            run(['git', 'clone', '--depth', '1', 'https://github.com/VAST-AI-Research/TripoSG.git', str(triposg_src)])
            run(['git', '-C', str(triposg_src), 'fetch', '--depth', '1', 'origin', TRIPOSG_CODE_REV])
            run(['git', '-C', str(triposg_src), 'checkout', '--detach', TRIPOSG_CODE_REV])
        _tsg_env = os.environ.copy()
        for _s in ('HF_TOKEN', 'OPENROUTER_API_KEY'):
            _tsg_env.pop(_s, None)
        run([PYTHON, '-m', 'pip', 'install', '-q', '-c', str(SCRATCH_ROOT / 'kaggle_constraints.txt'),
             'peft', 'jaxtyping', 'typeguard', 'einops', 'omegaconf', 'scikit-image'], env=_tsg_env)
        # `diso` has no cp313 wheel and needs a CUDA build; it is optional (only the flash decoder
        # dereferences DiffDMC), so a failed install must not lose the shard.
        try:
            run([PYTHON, '-m', 'pip', 'install', '-q', '--no-build-isolation', 'diso'], env=_tsg_env)
        except Exception as _diso_install_err:
            print(f'[triposg] Optional diso build unavailable ({type(_diso_install_err).__name__}); using marching-cubes decoder.', flush=True)
        try:
            import diso  # noqa: F401
            _has_diso = True
        except Exception:
            _has_diso = False
        if not _has_diso:
            # `triposg.inference_utils` imports DiffDMC at module scope; the stub keeps that import
            # clean because use_flash_decoder=False never constructs it.
            import sys as _diso_sys
            import types as _diso_types
            if 'diso' not in _diso_sys.modules:
                _diso_stub = _diso_types.ModuleType('diso')

                class DiffDMC:
                    pass

                _diso_stub.DiffDMC = DiffDMC
                _diso_sys.modules['diso'] = _diso_stub
        from huggingface_hub import snapshot_download as _snap_dl
        _snap_dl(repo_id=TRIPOSG_REPO, revision=TRIPOSG_REVISION, local_dir=str(triposg_weights), token=HF_TOKEN or False)
        import sys as _sys
        for _p in (str(triposg_src), str(triposg_src / 'scripts')):
            if _p not in _sys.path:
                _sys.path.insert(0, _p)
        from triposg.pipelines.pipeline_triposg import TripoSGPipeline
        # Upstream `scripts/inference_triposg.py` casts the whole pipeline with `.to(device, dtype)`.
        # Passing `torch_dtype=` to `from_pretrained` alone left the DINOv2 image encoder in float32,
        # and because `TripoSGPipeline` derives `latents.dtype` from `image_embeds.dtype`, its features
        # reached FP16 DiT linears and every dish died with
        # `mat1 and mat2 must have the same dtype, but got Float and Half` (benchmark-a/3, 19/19).
        _candidate_pipe = TripoSGPipeline.from_pretrained(str(triposg_weights)).to('cuda', torch.float16)
        # Belt and braces: pin the component that decides the latent dtype.
        _candidate_pipe.image_encoder_dinov2.to(dtype=torch.float16)
        # A dtype probe is a diagnostic, never a gate: benchmark-a/4 lost its own setup report to
        # `TypeError: 'dict' object is not callable` while the pipeline itself was already usable.
        try:
            _triposg_dtypes = {
                _name: torch.nn.Module.parameters(_module).__next__().dtype
                for _name, _module in (
                    ('vae', _candidate_pipe.vae),
                    ('transformer', _candidate_pipe.transformer),
                    ('image_encoder_dinov2', _candidate_pipe.image_encoder_dinov2),
                )
            }
        except Exception as _dtype_probe_err:
            print(
                f'[triposg] dtype probe unavailable ({type(_dtype_probe_err).__name__}: '
                f'{_dtype_probe_err}); continuing with the pipeline-level cast.',
                flush=True,
            )
        else:
            if set(_triposg_dtypes.values()) != {torch.float16}:
                raise RuntimeError(
                    f'TripoSG component dtypes {_triposg_dtypes} must all be float16; '
                    f'latents inherit image_embeds.dtype, so a float32 encoder breaks the DiT linears.',
                )
            print(f'[triposg] dtype probe ok: {_triposg_dtypes}', flush=True)
        triposg_pipe = _candidate_pipe
        print(
            f'[triposg] Loaded TripoSGPipeline FP16 on CUDA (diso={_has_diso}, RMBG-1.4 bypassed).',
            flush=True,
        )
    except Exception as _exc:
        triposg_setup_err = f'{type(_exc).__name__}: {_exc}'
        print('[triposg] Setup failed:', triposg_setup_err, flush=True)

    for label in ACTIVE_LABELS:
        in_entry = image_results[label]
        key = f'{label}/triposg'
        rec = {
            'input_label': label,
            'image_source': in_entry['image_source'],
            'image_model': in_entry.get('model', 'user-photo-upload'),
            'backend': 'TripoSG + Photo PBR',
            'license': 'MIT',
            'appearance': 'pbr_photo_projection',
            'artifact_paths': [],
        }
        if triposg_pipe is None:
            rec.update(status='failed', error=triposg_setup_err or 'TripoSG setup failed')
            RUNS_3D[key] = rec
            continue
        try:
            t0 = time.perf_counter()
            cutout_p = Path(in_entry['cutout_path'])
            # Prepare white-background square RGB directly from plate-preserving RGBA cutout (no RMBG-1.4)
            with Image.open(cutout_p) as _im:
                _rgba = _im.convert('RGBA')
                _bg = Image.new('RGBA', _rgba.size, (255, 255, 255, 255))
                _white_rgb = Image.alpha_composite(_bg, _rgba).convert('RGB')
            gen = torch.Generator(device=triposg_pipe.device).manual_seed(42)
            sample = triposg_pipe(
                image=_white_rgb,
                generator=gen,
                num_inference_steps=50,
                guidance_scale=7.0,
                use_flash_decoder=_has_diso,
            ).samples[0]
            verts = np.asarray(sample[0], dtype=np.float32)
            faces = np.ascontiguousarray(sample[1], dtype=np.uint32)
            # TripoSG's hierarchical marching-cubes decoder emits 1.4M-7.1M faces (benchmark-a/4);
            # without simplification the raw GLB breaks the 50 MB production cap that
            # `validate_glb` (generate_model) enforces and leaves nothing for the mobile ceiling.
            raw_faces_before = int(len(faces))
            if len(faces) > MAX_RAW_FACES:
                verts, faces = _decimate_vertex_cluster(verts, faces, MAX_RAW_FACES)
            raw_glb = RESULTS_DIR / label / 'triposg.glb'
            mob_glb = RESULTS_DIR / label / 'triposg_mobile.glb'
            baked = bake_photographic_uv_pbr_glb(verts, faces, cutout_p, raw_glb, mob_glb, texture_size=1536)
            elapsed = round(time.perf_counter() - t0, 2)
            glb_meta = validate_glb(raw_glb)
            mob_meta = validate_glb(mob_glb) if baked.get('mobile') else {}
            baked['raw']['elapsed_seconds'] = elapsed
            rec.update(
                status='success',
                device='cuda:0',
                watertight=baked['raw'].get('watertight'),
                elapsed_seconds=elapsed,
                metrics={**baked['raw'], **glb_meta, 'raw_faces_before_decimation': raw_faces_before},
                mobile_metrics={**baked['mobile'], **mob_meta} if baked.get('mobile') else None,
                mobile_glb_path=mob_glb if baked.get('mobile') else None,
                mobile_status='available' if baked.get('mobile') else 'failed',
                **({'mobile_error': baked['mobile_error']} if baked.get('mobile_error') else {}),
                artifact_paths=[raw_glb, mob_glb] if baked.get('mobile') else [raw_glb],
            )
            print(
                f'[triposg] {label}: {elapsed}s, faces={baked["raw"]["faces"]} '
                f'(was {raw_faces_before}, mob={(baked["mobile"] or {}).get("faces")})',
                flush=True,
            )
        except Exception as _exc:
            rec.update(status='failed', error=f'{type(_exc).__name__}: {_exc}')
            print(f'[triposg] {label} FAILED:', _exc, flush=True)
        RUNS_3D[key] = rec

    del triposg_pipe
    purge_hf_and_torch_caches((triposg_weights, triposg_src))


## 4. Option 4 — `Hunyuan3D-2` (DiT Shape + Texture/Photo UV PBR)


In [ ]:
# Benchmark Option 4: Tencent Hunyuan3D-2 (1.1B SDF DiT, Community R&D Non-EU/UK/KR) + Texture/PBR
HUNYUAN2_REPO = 'tencent/Hunyuan3D-2'
HUNYUAN2_REVISION = '9cd649ba6913f7a852e3286bad86bfa9a2d83dcf'
HUNYUAN2_CODE_REV = 'f8db63096c8282cb27354314d896feba5ba6ff8a'
REVISIONS['hunyuan3d2'] = HUNYUAN2_REVISION

if 'hunyuan3d2' in ENABLED_MODELS:
    hy2_src = SCRATCH_ROOT / 'Hunyuan3D-2-src'
    hy2_shape = None
    hy2_paint = None
    hy2_err = None
    try:
        if not hy2_src.exists():
            run(['git', 'clone', '--depth', '1', 'https://github.com/Tencent-Hunyuan/Hunyuan3D-2.git', str(hy2_src)])
            run(['git', '-C', str(hy2_src), 'fetch', '--depth', '1', 'origin', HUNYUAN2_CODE_REV])
            run(['git', '-C', str(hy2_src), 'checkout', '--detach', HUNYUAN2_CODE_REV])
        import sys as _sys
        if str(hy2_src) not in _sys.path:
            _sys.path.insert(0, str(hy2_src))
        _hy_env = os.environ.copy()
        for _s in ('HF_TOKEN', 'OPENROUTER_API_KEY'):
            _hy_env.pop(_s, None)
        run([PYTHON, '-m', 'pip', 'install', '-q', '-c', str(SCRATCH_ROOT / 'kaggle_constraints.txt'),
             'ninja', 'pybind11'], env=_hy_env)
        from hy3dgen.shapegen import Hunyuan3DDiTFlowMatchingPipeline
        hy2_shape = Hunyuan3DDiTFlowMatchingPipeline.from_pretrained(
            HUNYUAN2_REPO, revision=HUNYUAN2_REVISION, subfolder='hunyuan3d-dit-v2-0-fast',
            torch_dtype=torch.float16, token=HF_TOKEN or False,
        )
        hy2_shape.to('cuda')
        print('[hunyuan3d2] Loaded Hunyuan3D-2 (v2-0-fast) shape pipeline on CUDA.', flush=True)
    except Exception as _exc:
        hy2_err = f'{type(_exc).__name__}: {_exc}'
        print('[hunyuan3d2] Setup failed:', hy2_err, flush=True)

    # Pass 1: shape only. `mc_algo='mc'` uses skimage marching cubes (no diso / CUDA extension), and
    # the photographic UV PBR baker supplies captured-photo colour immediately.
    for label in ACTIVE_LABELS:
        in_entry = image_results[label]
        key = f'{label}/hunyuan3d2'
        rec = {
            'input_label': label,
            'image_source': in_entry['image_source'],
            'image_model': in_entry.get('model', 'user-photo-upload'),
            'backend': 'Hunyuan3D-2 / 2.1',
            'license': 'Tencent Community R&D (Non-EU/UK/KR)',
            'appearance': 'pbr_photo_projection',
            'artifact_paths': [],
        }
        if hy2_shape is None:
            rec.update(status='failed', error=hy2_err or 'Hunyuan3D-2 setup failed')
            RUNS_3D[key] = rec
            continue
        try:
            t0 = time.perf_counter()
            cutout_p = Path(in_entry['cutout_path'])
            with Image.open(cutout_p) as _im:
                _rgba = _im.convert('RGBA')
            mesh = hy2_shape(
                image=_rgba, octree_resolution=256, mc_algo='mc', num_inference_steps=30,
            )[0]
            verts = np.asarray(mesh.vertices, dtype=np.float32)
            faces = np.ascontiguousarray(mesh.faces, dtype=np.uint32)
            np.savez(RESULTS_DIR / label / 'hunyuan3d2_mesh.npz', vertices=verts, faces=faces)
            raw_glb = RESULTS_DIR / label / 'hunyuan3d2.glb'
            mob_glb = RESULTS_DIR / label / 'hunyuan3d2_mobile.glb'
            baked = bake_photographic_uv_pbr_glb(verts, faces, cutout_p, raw_glb, mob_glb, texture_size=1536)
            elapsed = round(time.perf_counter() - t0, 2)
            glb_meta = validate_glb(raw_glb)
            mob_meta = validate_glb(mob_glb) if baked.get('mobile') else {}
            baked['raw']['elapsed_seconds'] = elapsed
            rec.update(
                status='success',
                device='cuda:0',
                watertight=bool(check_watertight_edges(faces)[0]),
                appearance='pbr_photo_projection',
                elapsed_seconds=elapsed,
                metrics={**baked['raw'], **glb_meta},
                mobile_metrics={**baked['mobile'], **mob_meta} if baked.get('mobile') else None,
                mobile_glb_path=mob_glb if baked.get('mobile') else None,
                mobile_status='available' if baked.get('mobile') else 'failed',
                **({'mobile_error': baked['mobile_error']} if baked.get('mobile_error') else {}),
                artifact_paths=[raw_glb, mob_glb] if baked.get('mobile') else [raw_glb],
            )
            print(f'[hunyuan3d2] {label}: {elapsed}s, faces={len(faces)}, photo_uv_pbr=True', flush=True)
        except Exception as _exc:
            rec.update(status='failed', error=f'{type(_exc).__name__}: {_exc}')
            print(f'[hunyuan3d2] {label} FAILED:', _exc, flush=True)
        RUNS_3D[key] = rec

    # Free the shape pipeline before the optional texture stage so both never co-reside in 15GB T4 VRAM.
    if 'hy2_shape' in globals():
        del hy2_shape
    hy2_shape = None
    import gc as _hy_gc
    _hy_gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    print('[hunyuan3d2] Released shape pipeline VRAM before the optional paint pass.', flush=True)

    # Pass 2 (optional): native Hunyuan3DPaintPipeline over the meshes saved in pass 1.
    _hy2_paint_labels = [
        label for label in ACTIVE_LABELS
        if (RUNS_3D.get(f'{label}/hunyuan3d2') or {}).get('status') == 'success'
        and (RESULTS_DIR / label / 'hunyuan3d2_mesh.npz').is_file()
    ]
    if not _hy2_paint_labels:
        print('[hunyuan3d2] Skipping native paint pass: no successful shape meshes.', flush=True)
    else:
        try:
            _hy_env2 = os.environ.copy()
            for _s in ('HF_TOKEN', 'OPENROUTER_API_KEY'):
                _hy_env2.pop(_s, None)
            run([PYTHON, '-m', 'pip', 'install', '-q', '--no-build-isolation',
                 'xatlas', str(hy2_src / 'hy3dgen' / 'texgen' / 'custom_rasterizer')], env=_hy_env2)
            from hy3dgen.texgen import Hunyuan3DPaintPipeline
            hy2_paint = Hunyuan3DPaintPipeline.from_pretrained(HUNYUAN2_REPO)
            print('[hunyuan3d2] Loaded Hunyuan3DPaintPipeline for second-pass texture.', flush=True)
        except Exception as _pe:
            hy2_paint = None
            print(f'[hunyuan3d2] Native paint unavailable on T4 ({type(_pe).__name__}: {_pe}); keeping Photographic UV PBR output.', flush=True)

        if hy2_paint is not None:
            import trimesh as _hy_tm
            for label in _hy2_paint_labels:
                rec = RUNS_3D.get(f'{label}/hunyuan3d2') or {}
                mesh_npz = RESULTS_DIR / label / 'hunyuan3d2_mesh.npz'
                try:
                    t_paint = time.perf_counter()
                    with np.load(mesh_npz) as _saved:
                        _verts, _faces = _saved['vertices'], _saved['faces']
                    cutout_p = Path(image_results[label]['cutout_path'])
                    with Image.open(cutout_p) as _im:
                        _rgba = _im.convert('RGBA')
                    painted = hy2_paint(_hy_tm.Trimesh(vertices=_verts, faces=_faces, process=False), image=_rgba)
                    raw_glb = RESULTS_DIR / label / 'hunyuan3d2.glb'
                    painted.export(str(raw_glb))
                    glb_meta = validate_glb(raw_glb)
                    rec['metrics'] = {
                        **(rec.get('metrics') or {}),
                        **{k: glb_meta.get(k) for k in ('vertex_count', 'has_texture', 'has_vertex_color', 'bytes')},
                    }
                    rec['appearance'] = 'native_paint_glb'
                    rec['texture_seconds'] = round(time.perf_counter() - t_paint, 2)
                    print(f'[hunyuan3d2] {label}: native paint pass done in {rec["texture_seconds"]}s', flush=True)
                except Exception as _pr_err:
                    print(f'[hunyuan3d2] {label} native paint pass skipped ({type(_pr_err).__name__}: {_pr_err})', flush=True)
            if 'hy2_paint' in globals():
                del hy2_paint
            hy2_paint = None

    purge_hf_and_torch_caches((hy2_src,))


## 4b. Option 4b — `Hunyuan3D-2.1` (DiT Shape + Photo UV PBR)


In [ ]:
# Benchmark Option 4b: Tencent Hunyuan3D-2.1 (3.3B MoE DiT Shape, Community R&D Non-EU/UK/KR) + Photo UV PBR
HUNYUAN21_REPO = 'tencent/Hunyuan3D-2.1'
HUNYUAN21_REVISION = '0b94677654c57bb9a6b6845cd7b704ccf551d327'
HUNYUAN21_CODE_REV = '82920d643c0dc2f7bfd7255f45f62d386edfe60c'
REVISIONS['hunyuan3d21'] = HUNYUAN21_REVISION

if 'hunyuan3d21' in ENABLED_MODELS:
    hy21_src = SCRATCH_ROOT / 'Hunyuan3D-2.1-src'
    hy21_shape = None
    hy21_err = None
    try:
        if not hy21_src.exists():
            run(['git', 'clone', '--depth', '1', 'https://github.com/Tencent-Hunyuan/Hunyuan3D-2.1.git', str(hy21_src)])
            run(['git', '-C', str(hy21_src), 'fetch', '--depth', '1', 'origin', HUNYUAN21_CODE_REV])
            run(['git', '-C', str(hy21_src), 'checkout', '--detach', HUNYUAN21_CODE_REV])
        import sys as _sys
        if str(hy21_src) not in _sys.path:
            _sys.path.insert(0, str(hy21_src))
        _hy21_shape_dir = str(hy21_src / 'hy3dshape')
        if _hy21_shape_dir not in _sys.path:
            _sys.path.insert(0, _hy21_shape_dir)
        _hy_env = os.environ.copy()
        for _s in ('HF_TOKEN', 'OPENROUTER_API_KEY'):
            _hy_env.pop(_s, None)
        run([PYTHON, '-m', 'pip', 'install', '-q', '-c', str(SCRATCH_ROOT / 'kaggle_constraints.txt'),
             'ninja', 'pybind11'], env=_hy_env)
        from huggingface_hub import snapshot_download
        snapshot_download(
            HUNYUAN21_REPO,
            revision=HUNYUAN21_REVISION,
            allow_patterns=['hunyuan3d-dit-v2-1/*'],
            token=HF_TOKEN or False,
        )
        from hy3dshape.pipelines import Hunyuan3DDiTFlowMatchingPipeline
        hy21_shape = Hunyuan3DDiTFlowMatchingPipeline.from_pretrained(
            HUNYUAN21_REPO, subfolder='hunyuan3d-dit-v2-1',
            torch_dtype=torch.float16,
        )
        hy21_shape.to('cuda')
        print('[hunyuan3d21] Loaded Hunyuan3D-2.1 (v2-1) shape pipeline on CUDA.', flush=True)
    except Exception as _exc:
        hy21_err = f'{type(_exc).__name__}: {_exc}'
        print('[hunyuan3d21] Setup failed:', hy21_err, flush=True)

    for label in ACTIVE_LABELS:
        in_entry = image_results[label]
        key = f'{label}/hunyuan3d21'
        rec = {
            'input_label': label,
            'image_source': in_entry['image_source'],
            'image_model': in_entry.get('model', 'user-photo-upload'),
            'backend': 'Hunyuan3D-2.1',
            'license': 'Tencent Community R&D (Non-EU/UK/KR)',
            'appearance': 'pbr_photo_projection',
            'artifact_paths': [],
        }
        if hy21_shape is None:
            rec.update(status='failed', error=hy21_err or 'Hunyuan3D-2.1 setup failed')
            RUNS_3D[key] = rec
            continue
        try:
            t0 = time.perf_counter()
            cutout_p = Path(in_entry['cutout_path'])
            with Image.open(cutout_p) as _im:
                _rgba = _im.convert('RGBA')
            mesh = hy21_shape(
                image=_rgba, octree_resolution=256, mc_algo='mc', num_inference_steps=30,
            )[0]
            verts = np.asarray(mesh.vertices, dtype=np.float32)
            faces = np.ascontiguousarray(mesh.faces, dtype=np.uint32)
            np.savez(RESULTS_DIR / label / 'hunyuan3d21_mesh.npz', vertices=verts, faces=faces)
            raw_glb = RESULTS_DIR / label / 'hunyuan3d21.glb'
            mob_glb = RESULTS_DIR / label / 'hunyuan3d21_mobile.glb'
            baked = bake_photographic_uv_pbr_glb(verts, faces, cutout_p, raw_glb, mob_glb, texture_size=1536)
            elapsed = round(time.perf_counter() - t0, 2)
            glb_meta = validate_glb(raw_glb)
            mob_meta = validate_glb(mob_glb) if baked.get('mobile') else {}
            baked['raw']['elapsed_seconds'] = elapsed
            rec.update(
                status='success',
                device='cuda:0',
                watertight=bool(check_watertight_edges(faces)[0]),
                appearance='pbr_photo_projection',
                elapsed_seconds=elapsed,
                metrics={**baked['raw'], **glb_meta},
                mobile_metrics={**baked['mobile'], **mob_meta} if baked.get('mobile') else None,
                mobile_glb_path=mob_glb if baked.get('mobile') else None,
                mobile_status='available' if baked.get('mobile') else 'failed',
                **({'mobile_error': baked['mobile_error']} if baked.get('mobile_error') else {}),
                artifact_paths=[raw_glb, mob_glb] if baked.get('mobile') else [raw_glb],
            )
            print(f'[hunyuan3d21] {label}: {elapsed}s, faces={len(faces)}, photo_uv_pbr=True', flush=True)
        except Exception as _exc:
            rec.update(status='failed', error=f'{type(_exc).__name__}: {_exc}')
            print(f'[hunyuan3d21] {label} FAILED:', _exc, flush=True)
        RUNS_3D[key] = rec

    if 'hy21_shape' in globals():
        del hy21_shape
    hy21_shape = None
    import gc as _hy21_gc
    _hy21_gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    purge_hf_and_torch_caches((hy21_src,))


## 5. Option 5 — `Hi3DGen` (Normal-Bridging 3D + Photo UV PBR)


In [ ]:
# Benchmark Option 5: Stable-X Hi3DGen (Normal-Bridging 3D, MIT) + Photo UV PBR Baker
HI3D_TRELLIS_NORMAL_REPO = 'Stable-X/trellis-normal-v0-1'
HI3D_TRELLIS_NORMAL_REV = 'fdfac1d0e0cb'
HI3D_YOSO_NORMAL_REPO = 'Stable-X/yoso-normal-v1-8-1'
HI3D_YOSO_NORMAL_REV = '4dd7d000d76f'
HI3D_CODE_REV = 'c29f668ecec44b197275e9bf77f823c0c8a21076'
REVISIONS['hi3dgen'] = HI3D_TRELLIS_NORMAL_REV
# Child-process spconv check: report pccm's editable-install markers, then run one submanifold
# sparse convolution on the GPU Hi3DGen will use.
HI3D_SPCONV_SMOKE = '''
import importlib.util, pathlib
_root = pathlib.Path(importlib.util.find_spec('cumm').origin).parent.parent
print('pccm editable markers:', [n for n in ('setup.py', '.gitignore') if (_root / n).exists()])
import torch
import spconv.pytorch as spconv
dev = 'cuda' if torch.cuda.is_available() else 'cpu'
idx = torch.tensor([[0, 0, 0, 0], [0, 1, 1, 1], [0, 1, 2, 1]], dtype=torch.int32, device=dev)
x = spconv.SparseConvTensor(torch.ones(3, 4, device=dev), idx, [4, 4, 4], 1)
conv = spconv.SubMConv3d(4, 8, 3, bias=False, algo=spconv.ConvAlgo.Native).to(dev)
y = conv(x)
assert tuple(y.features.shape) == (3, 8), tuple(y.features.shape)
print('spconv smoke ok:', dev, tuple(y.features.shape))
'''

def _install_xformers_sdpa_shim():
    """Return True when native `xformers.ops` exists, else register a pure-PyTorch SDPA stand-in."""
    import sys as _x_sys
    import types as _x_types
    try:
        import xformers.ops as _x_ops  # noqa: F401
        print('[hi3dgen] Native xformers.ops is available; using it unchanged.', flush=True)
        return True
    except Exception:
        pass

    import torch as _x_torch
    import torch.nn.functional as _x_F

    class _SeqLensBlocks:
        """Minimal `xformers.ops.fmha.BlockDiagonalMask` replacement (seqlens -> block slices)."""

        def __init__(self, q_seqlen, kv_seqlen=None):
            self.q_seqlen = [int(s) for s in q_seqlen]
            self.kv_seqlen = [int(s) for s in (kv_seqlen if kv_seqlen is not None else q_seqlen)]

        @staticmethod
        def _blocks(seqlens):
            blocks, start = [], 0
            for length in seqlens:
                blocks.append((start, start + length))
                start += length
            return blocks

        @property
        def q_blocks(self):
            return self._blocks(self.q_seqlen)

        @property
        def kv_blocks(self):
            return self._blocks(self.kv_seqlen)

        @classmethod
        def from_seqlens(cls, q_seqlen, kv_seqlen=None):
            return cls(q_seqlen, kv_seqlen)

    def _x_memory_efficient_attention(q, k, v, attn_bias=None, *args, **kwargs):
        # xformers layout is [batch, seq, heads, channels]; torch SDPA wants [batch, heads, seq, channels].
        # Real xformers returns a tensor [B, M, H, C] (not a tuple). Callers index [0] to drop batch=1.
        if attn_bias is None or not hasattr(attn_bias, 'q_seqlen'):
            out = _x_F.scaled_dot_product_attention(
                q.transpose(1, 2), k.transpose(1, 2), v.transpose(1, 2),
            )
            return out.transpose(1, 2)
        q_blocks, kv_blocks = attn_bias.q_blocks, attn_bias.kv_blocks
        ragged = q.shape[1] == sum(attn_bias.q_seqlen) and k.shape[1] == sum(attn_bias.kv_seqlen)
        if ragged and len(q_blocks) == len(kv_blocks):
            # Pure block-diagonal (varlen) attention: each query block only sees its own key block.
            outs = []
            for (qs, qe), (ks, ke) in zip(q_blocks, kv_blocks):
                outs.append(_x_F.scaled_dot_product_attention(
                    q[:, qs:qe].transpose(1, 2),
                    k[:, ks:ke].transpose(1, 2),
                    v[:, ks:ke].transpose(1, 2),
                ).transpose(1, 2))
            return _x_torch.cat(outs, dim=1)
        # Rare mixed-shape fallback: build the additive block mask explicitly.
        mask = _x_torch.full((q.shape[1], k.shape[1]), float('-inf'), device=q.device, dtype=q.dtype)
        for (qs, qe), (ks, ke) in zip(q_blocks, kv_blocks):
            mask[qs:qe, ks:ke] = 0.0
        out = _x_F.scaled_dot_product_attention(
            q.transpose(1, 2), k.transpose(1, 2), v.transpose(1, 2), attn_mask=mask,
        )
        return out.transpose(1, 2)

    _x_fmha = _x_types.ModuleType('xformers.ops.fmha')
    _x_fmha.BlockDiagonalMask = _SeqLensBlocks
    _x_ops = _x_types.ModuleType('xformers.ops')
    _x_ops.__path__ = []
    _x_ops.fmha = _x_fmha
    _x_ops.memory_efficient_attention = _x_memory_efficient_attention
    _x_pkg = _x_types.ModuleType('xformers')
    _x_pkg.__path__ = []
    _x_pkg.ops = _x_ops
    _x_sys.modules['xformers'] = _x_pkg
    _x_sys.modules['xformers.ops'] = _x_ops
    _x_sys.modules['xformers.ops.fmha'] = _x_fmha
    print('[hi3dgen] xformers has no cp313 wheel; registered pure-PyTorch SDPA attention shim.', flush=True)
    return False


if 'hi3dgen' in ENABLED_MODELS:
    hi3d_src = SCRATCH_ROOT / 'Stable3DGen-src'
    hi3d_weights = SCRATCH_ROOT / 'Stable3DGen-weights'
    hi3d_pipe = None
    normal_pred = None
    hi3d_err = None
    # A model whose dependency cannot work on this image is `skipped`, never 19 failed dishes.
    hi3d_unavailable = False
    try:
        os.environ['SPCONV_ALGO'] = 'native'
        # The cp313 manylinux `cumm`/`spconv` wheels already ship compiled `core_cc` extensions and
        # their own CUDA 12.6 runtime. `pccm.utils.project_is_editable()` nevertheless reports them
        # as source checkouts whenever `site-packages/setup.py` or `site-packages/.gitignore`
        # exists, and the import then JIT-rebuilds `cumm` with `ninja` against headers the binary
        # wheel does not ship (benchmark-a/3 and /5: `tensorview/pybind_utils.h: No such file`).
        # These are the packages' own switches for skipping that rebuild; set before any import.
        os.environ['CUMM_DISABLE_JIT'] = '1'
        os.environ['SPCONV_DISABLE_JIT'] = '1'
        _xformers_native = _install_xformers_sdpa_shim()
        os.environ['SPARSE_ATTN_BACKEND'] = 'xformers'
        if not _xformers_native:
            os.environ['ATTN_BACKEND'] = 'sdpa'
        else:
            os.environ.setdefault('ATTN_BACKEND', 'sdpa')
        if not hi3d_src.exists():
            run(['git', 'clone', '--depth', '1', 'https://github.com/Stable-X/Stable3DGen.git', str(hi3d_src)])
            run(['git', '-C', str(hi3d_src), 'fetch', '--depth', '1', 'origin', HI3D_CODE_REV])
            run(['git', '-C', str(hi3d_src), 'checkout', '--detach', HI3D_CODE_REV])
        _h_env = os.environ.copy()
        for _s in ('HF_TOKEN', 'OPENROUTER_API_KEY'):
            _h_env.pop(_s, None)
        # spconv 2.3.8 ships cp313 manylinux wheels for cu126 and cu124; try cu126 first.
        _spconv_installed = None
        for _spconv_pkg in ('spconv-cu126', 'spconv-cu124'):
            try:
                run([PYTHON, '-m', 'pip', 'install', '-q', '-c', str(SCRATCH_ROOT / 'kaggle_constraints.txt'),
                     _spconv_pkg, 'easydict', 'kornia', 'timm'], env=_h_env)
                _spconv_installed = _spconv_pkg
                break
            except Exception as _spconv_err:
                print(f'[hi3dgen] {_spconv_pkg} install failed ({type(_spconv_err).__name__}); trying next.', flush=True)
        if _spconv_installed is None:
            run([PYTHON, '-m', 'pip', 'install', '-q', '-c', str(SCRATCH_ROOT / 'kaggle_constraints.txt'),
                 'easydict', 'kornia', 'timm'], env=_h_env)
            print('[hi3dgen] No spconv wheel matched; continuing without sparse-conv acceleration.', flush=True)
        else:
            print(f'[hi3dgen] Installed {_spconv_installed}.', flush=True)
            # Run one real sparse convolution in a bounded child process before loading weights, so
            # an unusable wheel is one measured, actionable shard skip instead of per-dish failures.
            try:
                _probe = subprocess.run(
                    [PYTHON, '-c', HI3D_SPCONV_SMOKE], capture_output=True, text=True, env=_h_env,
                    timeout=600,
                )
            except subprocess.TimeoutExpired:
                _probe = subprocess.CompletedProcess(
                    [PYTHON, '-c', 'spconv smoke'], 124, '', 'spconv smoke timed out after 600s',
                )
            if _probe.returncode != 0:
                _probe_lines = [
                    line.strip() for line in f'{_probe.stderr}\n{_probe.stdout}'.splitlines()
                    if line.strip()
                ]
                _probe_tail = _probe_lines[-1] if _probe_lines else 'no output'
                print(
                    f'[hi3dgen] {_spconv_installed} unusable on this CUDA image; last line: '
                    f'{_probe_tail}',
                    flush=True,
                )
                run([PYTHON, '-m', 'pip', 'uninstall', '-y', '-q', _spconv_installed], env=_h_env)
                raise RuntimeError(
                    f'spconv_unavailable:{_spconv_installed}:{_probe_tail[:160]}',
                )
            for _probe_line in _probe.stdout.splitlines():
                if _probe_line.strip():
                    print(f'[hi3dgen] {_probe_line.strip()}', flush=True)
        from huggingface_hub import snapshot_download as _snap_dl
        _tn_dir = hi3d_weights / 'trellis-normal-v0-1'
        _snap_dl(repo_id=HI3D_TRELLIS_NORMAL_REPO, local_dir=str(_tn_dir), token=HF_TOKEN or False)
        import sys as _sys
        if str(hi3d_src) not in _sys.path:
            _sys.path.insert(0, str(hi3d_src))
        from hi3dgen.pipelines import Hi3DGenPipeline
        hi3d_pipe = Hi3DGenPipeline.from_pretrained(str(_tn_dir))
        hi3d_pipe.cuda()
        normal_pred = torch.hub.load('hugoycj/StableNormal', 'StableNormal_turbo', trust_repo=True, yoso_version='yoso-normal-v1-8-1', local_cache_dir=str(hi3d_weights))
        print('[hi3dgen] Loaded Hi3DGenPipeline + StableNormal_turbo on CUDA.', flush=True)
    except Exception as _exc:
        hi3d_err = f'{type(_exc).__name__}: {_exc}'
        # Classify the measured environment, not just the probe's own marker: an spconv/cumm import
        # that still rebuilds `cumm` dies in `ninja` on the wheel's missing `tensorview` headers
        # (benchmark-a/3, 19/19 `CalledProcessError`). The no-wheel fallback above can still reach
        # that same import, so those failures stay one shard skip with the measured reason instead
        # of per-dish failures. A genuine Hi3DGen setup error (bad API, missing unrelated module)
        # keeps its `failed` status.
        hi3d_unavailable = str(_exc).startswith('spconv_unavailable:') or any(
            _marker in hi3d_err for _marker in ('spconv', 'cumm', 'tensorview', 'ninja')
        )
        print('[hi3dgen] Setup failed:', hi3d_err, flush=True)

    for label in ACTIVE_LABELS:
        in_entry = image_results[label]
        key = f'{label}/hi3dgen'
        rec = {
            'input_label': label,
            'image_source': in_entry['image_source'],
            'image_model': in_entry.get('model', 'user-photo-upload'),
            'backend': 'Hi3DGen Normal-Bridge',
            'license': 'MIT',
            'appearance': 'pbr_photo_projection',
            'artifact_paths': [],
        }
        if hi3d_pipe is None or normal_pred is None:
            if hi3d_unavailable:
                rec.update(
                    status='skipped',
                    error=f'Skipped ({hi3d_err or "Hi3DGen dependencies unavailable"})',
                )
            else:
                rec.update(status='failed', error=hi3d_err or 'Hi3DGen setup failed')
            RUNS_3D[key] = rec
            continue
        try:
            t0 = time.perf_counter()
            cutout_p = Path(in_entry['cutout_path'])
            with Image.open(cutout_p) as _im:
                _rgba = _im.convert('RGBA')
            _prep = hi3d_pipe.preprocess_image(_rgba, resolution=768)
            _norm_img = normal_pred(_prep, resolution=768, match_input_resolution=True, data_type='object')
            _outs = hi3d_pipe.run(
                _norm_img,
                seed=42,
                formats=['mesh'],
                preprocess_image=False,
                sparse_structure_sampler_params={'steps': 25, 'cfg_strength': 3.0},
                slat_sampler_params={'steps': 6, 'cfg_strength': 3.0},
            )
            _tm_mesh = _outs['mesh'][0].to_trimesh(transform_pose=True)
            verts = np.asarray(_tm_mesh.vertices, dtype=np.float32)
            faces = np.ascontiguousarray(_tm_mesh.faces, dtype=np.uint32)
            raw_glb = RESULTS_DIR / label / 'hi3dgen.glb'
            mob_glb = RESULTS_DIR / label / 'hi3dgen_mobile.glb'
            baked = bake_photographic_uv_pbr_glb(verts, faces, cutout_p, raw_glb, mob_glb, texture_size=1536)
            elapsed = round(time.perf_counter() - t0, 2)
            glb_meta = validate_glb(raw_glb)
            mob_meta = validate_glb(mob_glb) if baked.get('mobile') else {}
            baked['raw']['elapsed_seconds'] = elapsed
            rec.update(
                status='success',
                device='cuda:0',
                watertight=baked['raw'].get('watertight'),
                elapsed_seconds=elapsed,
                metrics={**baked['raw'], **glb_meta},
                mobile_metrics={**baked['mobile'], **mob_meta} if baked.get('mobile') else None,
                mobile_glb_path=mob_glb if baked.get('mobile') else None,
                mobile_status='available' if baked.get('mobile') else 'failed',
                **({'mobile_error': baked['mobile_error']} if baked.get('mobile_error') else {}),
                artifact_paths=[raw_glb, mob_glb] if baked.get('mobile') else [raw_glb],
            )
            print(f'[hi3dgen] {label}: {elapsed}s, faces={baked["raw"]["faces"]} (mob={baked["mobile"]["faces"]})', flush=True)
        except Exception as _exc:
            rec.update(status='failed', error=f'{type(_exc).__name__}: {_exc}')
            print(f'[hi3dgen] {label} FAILED:', _exc, flush=True)
        RUNS_3D[key] = rec

    del hi3d_pipe, normal_pred
    purge_hf_and_torch_caches((hi3d_weights, hi3d_src))


## 5b. Option 5b — `SPAR3D` (Point-Aware Reconstruction 3D + Native PBR)


In [ ]:
# Benchmark Option 5b: Stability AI SPAR3D (Point-Aware Reconstruction 3D, Stability Community)
SPAR3D_REPO = 'stabilityai/stable-point-aware-3d'
SPAR3D_REVISION = '5699918cb34f55cd7d828493d2725f3038313761'
SPAR3D_CODE_REV = 'fdc311b16809e6a8adc2f5a3407ebb3db1a95bd1'
REVISIONS['spar3d'] = SPAR3D_REVISION

if 'spar3d' in ENABLED_MODELS:
    spar3d_src = SCRATCH_ROOT / 'SPAR3D-src'
    spar3d_weights = SCRATCH_ROOT / 'SPAR3D-weights'
    spar3d_model = None
    spar3d_err = None
    try:
        if not spar3d_src.exists():
            run(['git', 'clone', '--depth', '1', 'https://github.com/Stability-AI/stable-point-aware-3d.git', str(spar3d_src)])
            run(['git', '-C', str(spar3d_src), 'fetch', '--depth', '1', 'origin', SPAR3D_CODE_REV])
            run(['git', '-C', str(spar3d_src), 'checkout', '--detach', SPAR3D_CODE_REV])
        import sys as _sys
        if str(spar3d_src) not in _sys.path:
            _sys.path.insert(0, str(spar3d_src))
        _spar_env = os.environ.copy()
        for _s in ('HF_TOKEN', 'OPENROUTER_API_KEY'):
            _spar_env.pop(_s, None)
        _spar_env['TORCH_CUDA_ARCH_LIST'] = '7.5'
        # If their setup pins c++17 and torch requires newer, rewrite it before building
        for _setup_path in (spar3d_src / 'texture_baker' / 'setup.py', spar3d_src / 'uv_unwrapper' / 'setup.py'):
            if _setup_path.exists():
                _setup_text = _setup_path.read_text(encoding='utf-8')
                if 'c++17' in _setup_text:
                    _setup_path.write_text(_setup_text.replace('c++17', 'c++20'), encoding='utf-8')
        run([PYTHON, '-m', 'pip', 'install', '-q', '-c', str(SCRATCH_ROOT / 'kaggle_constraints.txt'),
             'einops', 'jaxtyping', 'omegaconf', 'loralib'], env=_spar_env)
        # Avoid heavy non-commercial RMBG-1.4 or transparent-background installs; provide stub for Remover
        import types as _types
        if 'transparent_background' not in _sys.modules:
            _tb_stub = _types.ModuleType('transparent_background')
            class Remover:
                def __init__(self, *args, **kwargs): pass
            _tb_stub.Remover = Remover
            _sys.modules['transparent_background'] = _tb_stub
        # Install AlphaCLIP and CLIP
        run([PYTHON, '-m', 'pip', 'install', '-q', '--no-deps',
             'git+https://github.com/openai/CLIP.git',
             'git+https://github.com/SunzeY/AlphaCLIP.git'], env=_spar_env)
        # Build custom CUDA extensions: texture baker and uv unwrapper
        run([PYTHON, '-m', 'pip', 'install', '-q', '--no-build-isolation',
             str(spar3d_src / 'texture_baker'), str(spar3d_src / 'uv_unwrapper')], env=_spar_env)
        from huggingface_hub import snapshot_download
        snapshot_download(
            SPAR3D_REPO,
            revision=SPAR3D_REVISION,
            local_dir=str(spar3d_weights),
            token=HF_TOKEN or False,
        )
        from spar3d.system import SPAR3D
        spar3d_model = SPAR3D.from_pretrained(
            str(spar3d_weights),
            config_name='config.yaml',
            weight_name='model.safetensors',
            low_vram_mode=False,
        )
        spar3d_model.to('cuda')
        spar3d_model.eval()
        print('[spar3d] Loaded SPAR3D on CUDA.', flush=True)
    except Exception as _exc:
        spar3d_err = f'{type(_exc).__name__}: {_exc}'
        print('[spar3d] Setup failed:', spar3d_err, flush=True)

    for label in ACTIVE_LABELS:
        in_entry = image_results[label]
        key = f'{label}/spar3d'
        rec = {
            'input_label': label,
            'image_source': in_entry['image_source'],
            'image_model': in_entry.get('model', 'user-photo-upload'),
            'backend': 'SPAR3D',
            'license': 'Stability AI Community',
            'appearance': 'pbr_texture',
            'artifact_paths': [],
        }
        if spar3d_model is None:
            rec.update(status='skipped', error=f'Skipped ({spar3d_err or "SPAR3D setup failed"})')
            RUNS_3D[key] = rec
            continue
        try:
            t0 = time.perf_counter()
            cutout_p = Path(in_entry['cutout_path'])
            with Image.open(cutout_p) as _im:
                _rgba = _im.convert('RGBA')
            with torch.no_grad():
                with torch.autocast(device_type='cuda', dtype=torch.float16):
                    _out_mesh, _ = spar3d_model.run_image(
                        _rgba,
                        bake_resolution=1024,
                        return_points=False,
                    )
            if isinstance(_out_mesh, list):
                _out_mesh = _out_mesh[0]
            verts = np.asarray(_out_mesh.vertices, dtype=np.float32)
            faces = np.ascontiguousarray(_out_mesh.faces, dtype=np.uint32)
            raw_glb = RESULTS_DIR / label / 'spar3d.glb'
            mob_glb = RESULTS_DIR / label / 'spar3d_mobile.glb'
            _out_mesh.export(str(raw_glb))
            glb_meta = validate_glb(raw_glb)
            mob_meta = {}
            mob_err = None
            if len(faces) <= MAX_MOBILE_FACES:
                shutil.copyfile(raw_glb, mob_glb)
                mob_meta = validate_glb(mob_glb)
            else:
                try:
                    mob_mesh = _out_mesh.simplify_quadric_decimation(MAX_MOBILE_FACES)
                    mob_mesh.export(str(mob_glb))
                    mob_meta = validate_glb(mob_glb)
                except Exception as _m_err:
                    try:
                        vm, fm = _decimate_vertex_cluster(verts, faces, MAX_MOBILE_FACES)
                        if len(fm) <= MAX_MOBILE_FACES:
                            bake_photographic_uv_pbr_glb(vm, fm, cutout_p, mob_glb, mobile_glb_path=None, texture_size=1024)
                            mob_meta = validate_glb(mob_glb)
                        else:
                            mob_err = f'Decimation could not reach <= {MAX_MOBILE_FACES} faces'
                    except Exception as _fb_err:
                        mob_err = f'{type(_fb_err).__name__}: {_fb_err}'
                        mob_glb.unlink(missing_ok=True)
            elapsed = round(time.perf_counter() - t0, 2)
            watertight = bool(check_watertight_edges(faces)[0])
            raw_meta = {
                'vertices': len(verts),
                'faces': len(faces),
                'watertight': watertight,
                'bytes': raw_glb.stat().st_size,
                'sha256': sha256_file(raw_glb),
                'elapsed_seconds': elapsed,
                **glb_meta,
            }
            mob_available = mob_glb.is_file() and mob_meta.get('face_count', len(faces)) <= MAX_MOBILE_FACES
            rec.update(
                status='success',
                device='cuda:0',
                watertight=watertight,
                appearance='pbr_texture',
                elapsed_seconds=elapsed,
                metrics=raw_meta,
                mobile_metrics={**mob_meta} if mob_available else None,
                mobile_glb_path=mob_glb if mob_available else None,
                mobile_status='available' if mob_available else 'failed',
                **({'mobile_error': mob_err} if mob_err else {}),
                artifact_paths=[raw_glb, mob_glb] if mob_available else [raw_glb],
            )
            print(f'[spar3d] {label}: {elapsed}s, faces={len(faces)}, pbr_texture=True', flush=True)
        except Exception as _exc:
            rec.update(status='failed', error=f'{type(_exc).__name__}: {_exc}')
            print(f'[spar3d] {label} FAILED:', _exc, flush=True)
        RUNS_3D[key] = rec

    if 'spar3d_model' in globals():
        del spar3d_model
    spar3d_model = None
    purge_hf_and_torch_caches((spar3d_weights, spar3d_src))


## 6. Option 2 — `TRELLIS.2-4B` Native CUDA Build & Generation (when enabled & `HF_TOKEN` attached)


In [ ]:
TRELLIS_SOURCE = SCRATCH_ROOT / 'TRELLIS.2-native'
TRELLIS_REVISIONS = {
    'trellis': '75fbf0183001ed9876c8dbb35de6b68552ee08bd',
    'flex_gemm': '6dd94a859c26ee8246888502eada3dd8ad85532e',
    'cumesh': '12289e1062f0603f2f0d0771b02e1395d247f26f',
    'nvdiffrast': '253ac4fcea7de5f396371124af597e6cc957bfae',
    'nvdiffrec': 'b296927cc7fd01c2ac1087c8065c4d7248f72da4',
}
REVISIONS.update(TRELLIS_REVISIONS)
BUILD_ENV = os.environ.copy()
for secret_name in ('OPENROUTER_API_KEY', 'HF_TOKEN'):
    BUILD_ENV.pop(secret_name, None)
cuda_caps = sorted({torch.cuda.get_device_capability(idx) for idx in range(GPU_COUNT)})
BUILD_ENV['TORCH_CUDA_ARCH_LIST'] = ';'.join(f'{maj}.{min_}' for maj, min_ in cuda_caps)
BUILD_ENV['MAX_JOBS'] = '2'
if Path('/usr/local/cuda').exists():
    BUILD_ENV['CUDA_HOME'] = '/usr/local/cuda'

NVDIFFREC_AVAILABLE = False
TRELLIS_BUILD_OK = False
RUN_TRELLIS_BUILD = 'trellis2-4b' in ENABLED_MODELS
TRELLIS_SKIP_REASON = None

if not RUN_TRELLIS_BUILD:
    TRELLIS_SKIP_REASON = 'disabled_in_shard'
    print('Skipping TRELLIS.2 native CUDA build: trellis2-4b is disabled in this shard.')
elif not DINOV3_OK:
    TRELLIS_SKIP_REASON = DINOV3_NOTE
    print(f'Skipping TRELLIS.2 native CUDA build: {DINOV3_NOTE}. Attach an authorized HF_TOKEN in Kaggle Secrets (or the private peekaplate-hf-token dataset) to enable TRELLIS.2-4B.')
elif not NVCC_AVAILABLE:
    TRELLIS_SKIP_REASON = 'nvcc_unavailable'
    print('Skipping TRELLIS.2 native CUDA build: nvcc compiler unavailable.')
elif len(cuda_caps) != 1:
    TRELLIS_SKIP_REASON = f'heterogeneous_cuda_caps_{cuda_caps}'
    print(f'Skipping TRELLIS.2 native CUDA build: heterogeneous CUDA compute capabilities detected: {cuda_caps}')
elif (not any(_cuda_rel in subprocess.check_output(['nvcc', '--version'], text=True) for _cuda_rel in ('release 12.', 'release 13.'))) or (not str(torch.version.cuda).startswith(('12.', '13.'))):
    TRELLIS_SKIP_REASON = 'requires_cuda_12_or_13_toolchain'
    print(f'Skipping TRELLIS.2 native CUDA build: FlexGEMM/CuMesh/nvdiffrast pin CUDA 12.x/13.x (detected CUDA {torch.version.cuda}).')
else:
    try:
        run([PYTHON, '-m', 'pip', 'install',
             'wrapt', 'setuptools<81', 'wheel', 'ninja==1.11.1.4',
             'huggingface_hub==0.36.0', 'safetensors==0.6.2', 'transformers==4.57.3',
             'imageio==2.37.2', 'imageio-ffmpeg==0.6.0', 'tqdm==4.67.1', 'easydict==1.13',
             'opencv-python-headless==4.12.0.88', 'trimesh==4.10.1',
             'kornia==0.8.2', 'timm==1.0.22', 'zstandard==0.25.0',
             'rembg[cpu]==2.0.67', 'pyrender==0.1.45',
             'git+https://github.com/EasternJournalist/utils3d.git@9a4eb15e4021b67b12c460c7057d642626897ec8'],
            env=BUILD_ENV)

        def checkout(repo, folder, rev):
            created = not folder.exists()
            if created:
                run(['git', 'clone', '--no-checkout', repo, folder])
            current = subprocess.check_output(['git', '-C', str(folder), 'rev-parse', 'HEAD'], text=True).strip()
            if created or not (folder / '.git' / 'index').exists() or current != rev:
                run(['git', '-C', folder, 'fetch', '--depth', '1', 'origin', rev])
                run(['git', '-C', folder, 'checkout', '--detach', rev])
            run(['git', '-C', folder, 'submodule', 'update', '--init', '--recursive'])

        checkout('https://github.com/microsoft/TRELLIS.2.git', TRELLIS_SOURCE, TRELLIS_REVISIONS['trellis'])
        dependencies = SCRATCH_ROOT / 'trellis_native_dependencies'
        dependencies.mkdir(exist_ok=True)

        for name, repository in (
            ('flex_gemm', 'https://github.com/JeffreyXiang/FlexGEMM.git'),
            ('cumesh', 'https://github.com/JeffreyXiang/CuMesh.git'),
            ('nvdiffrast', 'https://github.com/NVlabs/nvdiffrast.git'),
            ('nvdiffrec', 'https://github.com/JeffreyXiang/nvdiffrec.git'),
        ):
            directory = dependencies / name
            checkout(repository, directory, TRELLIS_REVISIONS[name])
            try:
                run([PYTHON, '-m', 'pip', 'install', '--no-build-isolation', '--no-deps', str(directory)], env=BUILD_ENV)
                if name == 'nvdiffrec':
                    NVDIFFREC_AVAILABLE = True
            except Exception as err:
                if name == 'nvdiffrec':
                    print('Optional nvdiffrec_render build failed; continuing GLB export without HF Space shader previews:', err)
                    NVDIFFREC_AVAILABLE = False
                else:
                    raise

        run([PYTHON, '-m', 'pip', 'install', '--no-build-isolation', '--no-deps', TRELLIS_SOURCE / 'o-voxel'], env=BUILD_ENV)
        TRELLIS_BUILD_OK = True
        print('TRELLIS.2 native build complete. nvdiffrec_available =', NVDIFFREC_AVAILABLE)
    except Exception as _trellis_build_exc:
        TRELLIS_BUILD_OK = False
        NVDIFFREC_AVAILABLE = False
        TRELLIS_SKIP_REASON = f'cuda_build_failed:{type(_trellis_build_exc).__name__}'
        print('[trellis2-4b] Native CUDA build failed (non-fatal):', TRELLIS_SKIP_REASON, flush=True)


In [ ]:
(SCRATCH_ROOT / 'trellis_native_compat.py').write_text('"""Experimental, inference-only PyTorch adapters for official TRELLIS.2 weights.\n\nThe submanifold-convolution parameter names and layout match Microsoft\'s\nFlexGEMM adapter: [out_channels, kernel_x, kernel_y, kernel_z, in_channels].\nThese operations prioritize T4 compatibility over throughput.\n"""\n\nimport importlib\nimport itertools\nimport math\nfrom types import SimpleNamespace\n\nimport torch\nimport torch.nn as nn\nimport torch.nn.functional as F\n\n\nclass CoordinateLookup:\n    def __init__(self, coords, spatial_shape=None):\n        if coords.ndim != 2 or coords.shape[1] != 4 or len(coords) == 0:\n            raise ValueError(\'Expected nonempty [N, 4] batch/xyz coordinates.\')\n        self.coords = coords.long()\n        self.spatial_shape = tuple(int(v) for v in (\n            spatial_shape if spatial_shape is not None else self.coords[:, 1:].amax(0) + 1\n        ))\n        self.batch_size = int(self.coords[:, 0].max()) + 1\n        self.keys, self.order = self.encode(self.coords).sort()\n        if len(self.keys) > 1 and bool((self.keys[1:] == self.keys[:-1]).any()):\n            raise ValueError(\'Duplicate sparse coordinates are unsupported.\')\n\n    def encode(self, coords):\n        width, height, depth = self.spatial_shape\n        return ((coords[:, 0] * width + coords[:, 1]) * height + coords[:, 2]) * depth + coords[:, 3]\n\n    def find(self, query):\n        query = query.long()\n        limits = query.new_tensor((self.batch_size, *self.spatial_shape))\n        valid = ((query >= 0) & (query < limits)).all(1)\n        keys = self.encode(query)\n        positions = torch.searchsorted(self.keys, keys).clamp_max(len(self.keys) - 1)\n        valid = valid & (self.keys[positions] == keys)\n        return torch.where(valid, self.order[positions], positions.new_full(positions.shape, -1))\n\n\ndef move_to_device(value, device):\n    """Move Tensors, SparseTensors, and VarLenTensors across devices without keeping source caches."""\n    if value is None:\n        return None\n    if hasattr(value, \'feats\') and hasattr(value, \'coords\'):\n        kwargs = {\'feats\': value.feats.to(device), \'coords\': value.coords.to(device)}\n        if hasattr(value, \'_shape\'):\n            kwargs[\'shape\'] = value._shape\n        if getattr(value, \'_scale\', None) is not None:\n            kwargs[\'scale\'] = value._scale\n        return type(value)(**kwargs)\n    if hasattr(value, \'feats\') and hasattr(value, \'layout\'):\n        return type(value)(feats=value.feats.to(device), layout=list(value.layout))\n    if isinstance(value, torch.Tensor):\n        return value.to(device)\n    if isinstance(value, list):\n        return [move_to_device(v, device) for v in value]\n    if isinstance(value, tuple):\n        return tuple(move_to_device(v, device) for v in value)\n    return value\n\n\ndef _manual_cast(value, dtype):\n    if dtype is None or not hasattr(value, \'dtype\'):\n        return value\n    if value.dtype == dtype:\n        return value\n    if hasattr(value, \'type\'):\n        return value.type(dtype)\n    return value.to(dtype)\n\n\nclass DualGpuShardedFlowModel(nn.Module):\n    """Pipeline-parallel 2-GPU wrapper splitting TRELLIS flow transformer blocks evenly across GPUs."""\n\n    def __init__(self, model, dev0=\'cuda:0\', dev1=\'cuda:1\'):\n        super().__init__()\n        self.model = model\n        self.dev0 = torch.device(dev0)\n        self.dev1 = torch.device(dev1)\n        blocks = getattr(model, \'blocks\', None)\n        self.split = (len(blocks) // 2) if blocks is not None else 0\n        self._rope_phases_dev1 = None\n        try:\n            self._active_dev0 = next(model.parameters()).device\n        except StopIteration:\n            self._active_dev0 = torch.device(\'cpu\')\n\n    @property\n    def in_channels(self):\n        return self.model.in_channels\n\n    @property\n    def resolution(self):\n        return self.model.resolution\n\n    @property\n    def dtype(self):\n        return self.model.dtype\n\n    @property\n    def device(self):\n        return self._active_dev0\n\n    def convert_to(self, dtype):\n        self.model.convert_to(dtype)\n\n    def _place_sharded(self):\n        m = self.model\n        for attr in (\'t_embedder\', \'adaLN_modulation\', \'pos_embedder\', \'input_layer\'):\n            sub = getattr(m, attr, None)\n            if isinstance(sub, nn.Module):\n                sub.to(self.dev0)\n        for buf_name in (\'pos_emb\', \'rope_phases\'):\n            buf = getattr(m, buf_name, None)\n            if isinstance(buf, torch.Tensor):\n                m.register_buffer(buf_name, buf.to(self.dev0))\n        if getattr(m, \'rope_phases\', None) is not None:\n            self._rope_phases_dev1 = m.rope_phases.to(self.dev1)\n        else:\n            self._rope_phases_dev1 = None\n        for idx, block in enumerate(m.blocks):\n            block.to(self.dev0 if idx < self.split else self.dev1)\n        if isinstance(getattr(m, \'out_layer\', None), nn.Module):\n            m.out_layer.to(self.dev1)\n        self._active_dev0 = self.dev0\n        return self\n\n    def cpu(self):\n        self.model.cpu()\n        self._rope_phases_dev1 = None\n        self._active_dev0 = torch.device(\'cpu\')\n        if torch.cuda.is_available():\n            torch.cuda.empty_cache()\n        return self\n\n    def to(self, *args, **kwargs):\n        device = kwargs.get(\'device\')\n        dtype = kwargs.get(\'dtype\')\n        for arg in args:\n            if isinstance(arg, torch.dtype):\n                dtype = arg\n            elif isinstance(arg, (torch.device, str)):\n                device = arg\n        if dtype is not None:\n            self.model.to(dtype=dtype)\n        if device is not None:\n            target = torch.device(device)\n            if target.type == \'cpu\':\n                return self.cpu()\n            return self._place_sharded()\n        return self\n\n    def forward(self, x, t, cond, concat_cond=None, **kwargs):\n        m = self.model\n        out_device = x.device\n        out_dtype = x.dtype\n        dev0 = self.dev0 if self._active_dev0.type == \'cuda\' or self.dev0.type != \'cuda\' else self._active_dev0\n        dev1 = self.dev1 if self._active_dev0.type == \'cuda\' or self.dev1.type != \'cuda\' else self._active_dev0\n\n        if hasattr(x, \'feats\') and hasattr(x, \'coords\'):\n            def _run_stage0():\n                nonlocal x, t, cond, concat_cond\n                if x.device != dev0:\n                    x = move_to_device(x, dev0)\n                t = move_to_device(t, dev0)\n                if concat_cond is not None:\n                    if concat_cond.device != dev0:\n                        concat_cond = move_to_device(concat_cond, dev0)\n                    try:\n                        from trellis2.modules import sparse as sp\n                        x_in = sp.sparse_cat([x, concat_cond], dim=-1)\n                    except ImportError:\n                        x_in = x.replace(torch.cat([x.feats, concat_cond.feats], dim=-1))\n                else:\n                    x_in = x\n                if isinstance(cond, list):\n                    cond_on_d0 = move_to_device(cond, dev0)\n                    from trellis2.modules import sparse as sp\n                    c_in = sp.VarLenTensor.from_tensor_list(cond_on_d0)\n                else:\n                    c_in = move_to_device(cond, dev0)\n                h0 = m.input_layer(x_in)\n                h0 = _manual_cast(h0, m.dtype)\n                t0 = m.t_embedder(t)\n                if getattr(m, \'share_mod\', False):\n                    t0 = m.adaLN_modulation(t0)\n                t0 = _manual_cast(t0, m.dtype)\n                c0 = _manual_cast(c_in, m.dtype)\n                if getattr(m, \'pe_mode\', None) == \'ape\':\n                    pe = m.pos_embedder(h0.coords[:, 1:])\n                    h0 = h0 + _manual_cast(pe, m.dtype)\n                for block in m.blocks[:self.split]:\n                    h0 = block(h0, t0, c0)\n                return h0, t0, c0\n\n            if dev0.type == \'cuda\':\n                with torch.cuda.device(dev0):\n                    h, t_emb, c_cur = _run_stage0()\n            else:\n                h, t_emb, c_cur = _run_stage0()\n\n            def _run_stage1():\n                h1 = move_to_device(h, dev1)\n                t1 = move_to_device(t_emb, dev1)\n                c1 = move_to_device(c_cur, dev1)\n                for block in m.blocks[self.split:]:\n                    h1 = block(h1, t1, c1)\n                h1 = _manual_cast(h1, out_dtype)\n                h1 = h1.replace(F.layer_norm(h1.feats, h1.feats.shape[-1:]))\n                return m.out_layer(h1)\n\n            if dev1.type == \'cuda\':\n                with torch.cuda.device(dev1):\n                    h_out = _run_stage1()\n            else:\n                h_out = _run_stage1()\n            return move_to_device(h_out, out_device)\n\n        def _run_dense0():\n            xd = x.to(dev0) if x.device != dev0 else x\n            td = t.to(dev0) if t.device != dev0 else t\n            cd = move_to_device(cond, dev0)\n            hd = xd.view(*xd.shape[:2], -1).permute(0, 2, 1).contiguous()\n            hd = m.input_layer(hd)\n            if getattr(m, \'pe_mode\', None) == \'ape\' and getattr(m, \'pos_emb\', None) is not None:\n                hd = hd + m.pos_emb[None]\n            te = m.t_embedder(td)\n            if getattr(m, \'share_mod\', False):\n                te = m.adaLN_modulation(te)\n            te = _manual_cast(te, m.dtype)\n            hd = _manual_cast(hd, m.dtype)\n            cd = _manual_cast(cd, m.dtype)\n            rope0 = getattr(m, \'rope_phases\', None)\n            if rope0 is not None and rope0.device != dev0:\n                rope0 = rope0.to(dev0)\n            for block in m.blocks[:self.split]:\n                hd = block(hd, te, cd, rope0)\n            return hd, te, cd\n\n        if dev0.type == \'cuda\':\n            with torch.cuda.device(dev0):\n                h, t_emb, c_cur = _run_dense0()\n        else:\n            h, t_emb, c_cur = _run_dense0()\n\n        def _run_dense1():\n            h1 = h.to(dev1) if h.device != dev1 else h\n            t1 = t_emb.to(dev1) if t_emb.device != dev1 else t_emb\n            c1 = move_to_device(c_cur, dev1)\n            rope1 = self._rope_phases_dev1\n            if rope1 is None and getattr(m, \'rope_phases\', None) is not None:\n                rope1 = m.rope_phases.to(dev1)\n            for block in m.blocks[self.split:]:\n                h1 = block(h1, t1, c1, rope1)\n            h1 = _manual_cast(h1, out_dtype)\n            h1 = F.layer_norm(h1, h1.shape[-1:])\n            h1 = m.out_layer(h1)\n            return h1.permute(0, 2, 1).view(h1.shape[0], h1.shape[2], *[m.resolution] * 3).contiguous()\n\n        if dev1.type == \'cuda\':\n            with torch.cuda.device(dev1):\n                h_out = _run_dense1()\n        else:\n            h_out = _run_dense1()\n        return h_out.to(out_device)\n\n\ndef submanifold_convolution(feats, coords, weight, bias=None, dilation=(1, 1, 1),\n                            neighbor_map=None, chunk_size=8192):\n    # 1. Keep exactly the checkpoint\'s convolution orientation and sparse sites.\n    out_channels, size_x, size_y, size_z, in_channels = weight.shape\n    if feats.shape != (len(coords), in_channels):\n        raise ValueError(\'Sparse feature channels do not match convolution weights.\')\n    kernel_shape = (size_x, size_y, size_z)\n    if any(size % 2 == 0 for size in kernel_shape):\n        raise ValueError(\'This adapter supports odd, centered submanifold kernels.\')\n    offsets = list(itertools.product(*(range(size) for size in kernel_shape)))\n    if neighbor_map is None:\n        lookup = CoordinateLookup(coords)\n        neighbor_map = torch.empty((len(coords), len(offsets)), dtype=torch.int32, device=coords.device)\n        for index, offset in enumerate(offsets):\n            query = coords.clone().long()\n            delta = query.new_tensor([(offset[axis] - kernel_shape[axis] // 2) * dilation[axis]\n                                      for axis in range(3)])\n            query[:, 1:] += delta\n            neighbor_map[:, index] = lookup.find(query).int()\n\n    # 2. Bound temporary activation memory and accumulate convolution in FP32.\n    output = torch.empty((len(coords), out_channels), dtype=feats.dtype, device=feats.device)\n    for start in range(0, len(coords), chunk_size):\n        stop = min(start + chunk_size, len(coords))\n        accumulator = torch.zeros((stop - start, out_channels), dtype=torch.float32, device=feats.device)\n        for index, (ix, iy, iz) in enumerate(offsets):\n            neighbors = neighbor_map[start:stop, index].long()\n            valid = neighbors >= 0\n            gathered = feats[neighbors.clamp_min(0)].float()\n            gathered = gathered * valid[:, None]\n            accumulator.addmm_(gathered, weight[:, ix, iy, iz, :].float().t())\n        if bias is not None:\n            accumulator += bias.float()\n        output[start:stop] = accumulator.to(feats.dtype)\n    return output, neighbor_map\n\n\ndef sparse_grid_sample(feats, coords, shape, grid, mode=\'trilinear\', chunk_size=32768):\n    # 3. Sample voxel-centered attributes without the Triton interpolation kernel.\n    if mode not in (\'nearest\', \'trilinear\'):\n        raise ValueError(f\'Unknown sampling mode: {mode}\')\n    if grid.ndim != 3 or grid.shape[-1] != 3:\n        raise ValueError(\'Expected grid shape [batch, query_count, 3].\')\n    lookup = CoordinateLookup(coords, shape[-3:])\n    batch_size, query_count, _ = grid.shape\n    batches = torch.arange(batch_size, device=grid.device).repeat_interleave(query_count)\n    queries = grid.reshape(-1, 3).float()\n    output = torch.empty((len(queries), feats.shape[1]), dtype=feats.dtype, device=feats.device)\n    for start in range(0, len(queries), chunk_size):\n        stop = min(start + chunk_size, len(queries))\n        points = queries[start:stop]\n        batch_column = batches[start:stop, None]\n        if mode == \'nearest\':\n            indices = lookup.find(torch.cat([batch_column, points.trunc().long()], 1))\n            output[start:stop] = feats[indices.clamp_min(0)] * (indices >= 0)[:, None]\n            continue\n        lower = (points - 0.5).floor().long()\n        value_sum = torch.zeros((len(points), feats.shape[1]), dtype=torch.float32, device=feats.device)\n        weight_sum = torch.zeros((len(points), 1), dtype=torch.float32, device=feats.device)\n        for offset in itertools.product((0, 1), repeat=3):\n            neighbor = lower + lower.new_tensor(offset)\n            indices = lookup.find(torch.cat([batch_column, neighbor], 1))\n            weights = (1 - (neighbor.float() + 0.5 - points).abs()).clamp_min(0).prod(1, keepdim=True)\n            weights *= (indices >= 0)[:, None]\n            value_sum += weights * feats[indices.clamp_min(0)].float()\n            weight_sum += weights\n        output[start:stop] = (value_sum / weight_sum.clamp_min(1e-12)).to(feats.dtype)\n    return output.reshape(batch_size, query_count, -1)\n\n\ndef sparse_attention(*args, **kwargs):\n    # 4. Preserve each batch\'s independent sequence and the public return type.\n    names = {1: (\'qkv\',), 2: (\'q\', \'kv\'), 3: (\'q\', \'k\', \'v\')}\n    count = len(args) + len(kwargs)\n    if count not in names:\n        raise ValueError(\'Expected qkv, q/kv, or q/k/v.\')\n    values = list(args) + [kwargs[name] for name in names[count][len(args):]]\n\n    def unpack(value):\n        if hasattr(value, \'feats\'):\n            return value.feats, value.layout\n        batch_size, length = value.shape[:2]\n        return value.flatten(0, 1), [slice(i * length, (i + 1) * length) for i in range(batch_size)]\n\n    source = values[0] if hasattr(values[0], \'feats\') else None\n    if count == 1:\n        tensor, q_layout = unpack(values[0])\n        query, key, value = tensor.unbind(1)\n        kv_layout = q_layout\n    elif count == 2:\n        query, q_layout = unpack(values[0])\n        tensor, kv_layout = unpack(values[1])\n        key, value = tensor.unbind(1)\n    else:\n        query, q_layout = unpack(values[0])\n        key, kv_layout = unpack(values[1])\n        value, value_layout = unpack(values[2])\n        if [(s.start, s.stop) for s in value_layout] != [(s.start, s.stop) for s in kv_layout]:\n            raise ValueError(\'Key and value batch layouts differ.\')\n    if len(q_layout) != len(kv_layout):\n        raise ValueError(\'Query and key/value batch sizes differ.\')\n    output = query.new_empty((len(query), query.shape[1], value.shape[-1]))\n    for query_slice, key_slice in zip(q_layout, kv_layout):\n        keys = key[key_slice].transpose(0, 1).unsqueeze(0)\n        vals = value[key_slice].transpose(0, 1).unsqueeze(0)\n        kv_len = max(1, key_slice.stop - key_slice.start)\n        q_chunk = min(1024, max(256, 8_000_000 // kv_len))\n        for start in range(query_slice.start, query_slice.stop, q_chunk):\n            stop = min(start + q_chunk, query_slice.stop)\n            queries = query[start:stop].transpose(0, 1).unsqueeze(0)\n            result = F.scaled_dot_product_attention(queries, keys, vals, dropout_p=0.0)\n            output[start:stop] = result.squeeze(0).transpose(0, 1)\n    if source is not None:\n        return source.replace(output)\n    return output.reshape(values[0].shape[0], values[0].shape[1], *output.shape[1:])\n\n\ndef install_adapters():\n    # 5. Register a weight-compatible backend before constructing any decoder.\n    from trellis2.modules import sparse as sp\n    from trellis2.modules.sparse import config\n    config.set_conv_backend(\'none\')\n    conv_module = importlib.import_module(\'trellis2.modules.sparse.conv.conv\')\n\n    def initialize(module, in_channels, out_channels, kernel_size, stride=1,\n                   dilation=1, padding=None, bias=True, indice_key=None):\n        if stride != 1 or padding is not None:\n            raise NotImplementedError(\'Only the official submanifold-convolution path is supported.\')\n        sizes = tuple(kernel_size) if isinstance(kernel_size, (tuple, list)) else (kernel_size,) * 3\n        module.dilation = tuple(dilation) if isinstance(dilation, (tuple, list)) else (dilation,) * 3\n        module.in_channels, module.out_channels = in_channels, out_channels\n        module.weight = nn.Parameter(torch.empty(out_channels, *sizes, in_channels))\n        nn.init.kaiming_uniform_(module.weight, a=math.sqrt(5))\n        if bias:\n            module.bias = nn.Parameter(torch.zeros(out_channels))\n        else:\n            module.register_parameter(\'bias\', None)\n\n    def forward(module, value):\n        key = (\'native_torch_neighbors\', tuple(module.weight.shape[1:4]), module.dilation)\n        cached = value.get_spatial_cache(key)\n        features, neighbors = submanifold_convolution(\n            value.feats, value.coords, module.weight, module.bias, module.dilation, cached\n        )\n        if cached is None:\n            value.register_spatial_cache(key, neighbors)\n        return value.replace(features)\n\n    def unsupported(*args, **kwargs):\n        raise NotImplementedError(\'Inverse sparse convolution is not part of this inference adapter.\')\n\n    conv_module._backends[\'none\'] = SimpleNamespace(\n        sparse_conv3d_init=initialize, sparse_conv3d_forward=forward,\n        sparse_inverse_conv3d_init=unsupported, sparse_inverse_conv3d_forward=unsupported,\n    )\n    for name in (\'trellis2.modules.sparse.attention\',\n                 \'trellis2.modules.sparse.attention.full_attn\',\n                 \'trellis2.modules.sparse.attention.modules\'):\n        module = importlib.import_module(name)\n        module.sparse_scaled_dot_product_attention = sparse_attention\n    sp.sparse_scaled_dot_product_attention = sparse_attention\n    mesh_module = importlib.import_module(\'trellis2.representations.mesh.base\')\n    mesh_module.grid_sample_3d = sparse_grid_sample\n    export_module = importlib.import_module(\'o_voxel.postprocess\')\n    export_module.grid_sample_3d = sparse_grid_sample\n', encoding='utf-8')
(SCRATCH_ROOT / 'trellis_native_runner.py').write_text('"""Direct Microsoft TRELLIS.2-4B inference with stage-by-stage lazy model loading."""\n\nimport argparse\nimport gc\nimport json\nimport os\nfrom pathlib import Path\nimport sys\nimport time\nimport traceback\nimport types\n\n# 1. Select supported attention before importing TRELLIS modules.\nos.environ[\'OPENCV_IO_ENABLE_OPENEXR\'] = \'1\'\nos.environ[\'PYTORCH_CUDA_ALLOC_CONF\'] = \'expandable_segments:True\'\nos.environ[\'ATTN_BACKEND\'] = \'sdpa\'\nos.environ[\'SPARSE_CONV_BACKEND\'] = \'none\'\nos.environ[\'SPARSE_ATTN_BACKEND\'] = \'xformers\'  # Replaced by our native SDPA adapter before inference.\n\nimport numpy as np\nfrom PIL import Image\nimport torch\nfrom huggingface_hub import HfApi, hf_hub_download, snapshot_download\nfrom safetensors.torch import load_file\n\n\ndef write_json(path, value):\n    path.write_text(json.dumps(value, indent=2), encoding=\'utf-8\')\n\n\ndef _rss_gib():\n    try:\n        for line in Path(\'/proc/self/status\').read_text(encoding=\'utf-8\').splitlines():\n            if line.startswith(\'VmRSS:\'):\n                return int(line.split()[1]) / (1024 * 1024)\n    except OSError:\n        pass\n    return 0.0\n\n\ndef _cuda_cleanup():\n    gc.collect()\n    if torch.cuda.is_available():\n        for dev_idx in range(torch.cuda.device_count()):\n            with torch.cuda.device(dev_idx):\n                torch.cuda.empty_cache()\n\n\ndef _log_stage(stage, t0, extra=\'\'):\n    gpu0 = (torch.cuda.memory_allocated(0) / 2**30) if torch.cuda.is_available() else 0.0\n    gpu1 = (torch.cuda.memory_allocated(1) / 2**30) if torch.cuda.device_count() > 1 else 0.0\n    suffix = f\' {extra}\' if extra else \'\'\n    print(\n        f\'[trellis:stage] {stage} elapsed={time.perf_counter() - t0:.1f}s \'\n        f\'rss={_rss_gib():.2f}GiB gpu0={gpu0:.2f}GiB gpu1={gpu1:.2f}GiB{suffix}\',\n        flush=True,\n    )\n\n\ndef _release_model(model_obj):\n    if model_obj is not None and hasattr(model_obj, \'cpu\'):\n        model_obj.cpu()\n    _cuda_cleanup()\n    return None\n\n\ndef main(config_path):\n    config = json.loads(Path(config_path).read_text(encoding=\'utf-8\'))\n    output = Path(config[\'output_dir\'])\n    output.mkdir(parents=True, exist_ok=True)\n    status_path = output / \'run_status.json\'\n    status = {\'status\': \'running\', \'started_at\': time.time(), \'model\': \'microsoft/TRELLIS.2-4B\'}\n    write_json(status_path, status)\n    try:\n        if not torch.cuda.is_available():\n            raise RuntimeError(\'A CUDA GPU is required for model decoding and GLB export.\')\n        torch.cuda.set_device(0)\n        sys.path.insert(0, config[\'source_dir\'])\n        from trellis_native_compat import DualGpuShardedFlowModel, install_adapters, move_to_device\n        install_adapters()\n        from trellis2 import models\n        from trellis2.modules import image_feature_extractor\n        from trellis2.pipelines import Trellis2ImageTo3DPipeline, samplers\n        from trellis2.representations import MeshWithVoxel\n        import o_voxel\n\n        # 2. Pin the model repository and prepare lazy role loader.\n        api = HfApi()\n        token = os.environ.get(\'HF_TOKEN\') or None\n        pinned = config.get(\'checkpoint_revisions\') or {}\n        model_repo = \'microsoft/TRELLIS.2-4B\'\n        revision = config.get(\'model_revision\') or pinned.get(model_repo) or api.model_info(model_repo, token=token).sha\n        pipeline_path = hf_hub_download(model_repo, \'pipeline.json\', revision=revision, token=token)\n        specification = json.loads(Path(pipeline_path).read_text(encoding=\'utf-8\'))[\'args\']\n        revisions = {model_repo: revision}\n        use_bf16 = (\n            torch.cuda.is_available()\n            and torch.cuda.get_device_capability(0)[0] >= 8\n            and bool(config.get(\'auto_bf16\', True))\n        )\n        dtype_used = \'bfloat16\' if use_bf16 else \'float32\'\n        dual_gpu = torch.cuda.device_count() >= 2 and bool(config.get(\'use_dual_gpu_sharding\', True))\n        requested_pipeline = config.get(\'pipeline_type\', \'512\')\n        if requested_pipeline not in (\'512\', \'1024\', \'1024_cascade\', \'1536_cascade\'):\n            raise ValueError(f\'Invalid pipeline type: {requested_pipeline}\')\n        max_num_tokens = int(config.get(\'max_num_tokens\', 49152))\n        encoder_device = 1 if torch.cuda.device_count() > 1 else 0\n        checkpoint_audit = {}\n        allowed_generated_buffers = (\'pos_emb\', \'rope_phases\')\n\n        def load_role(role):\n            _cuda_cleanup()\n            reference = specification[\'models\'][role]\n            if reference.startswith(\'ckpts/\'):\n                repository, prefix = model_repo, reference\n            else:\n                owner, name, prefix = reference.split(\'/\', 2)\n                repository = owner + \'/\' + name\n            if repository not in revisions:\n                revisions[repository] = pinned.get(repository) or api.model_info(repository, token=token).sha\n            model_json = hf_hub_download(repository, prefix + \'.json\', revision=revisions[repository], token=token)\n            model_weights = hf_hub_download(repository, prefix + \'.safetensors\', revision=revisions[repository], token=token)\n            model_spec = json.loads(Path(model_json).read_text(encoding=\'utf-8\'))\n            arguments = model_spec[\'args\'].copy()\n            if \'flow_model\' in role and not use_bf16:\n                # T4 (sm_75) lacks native BF16 tensor cores; Ampere+ (sm_80+) keeps native BF16.\n                arguments[\'dtype\'] = \'float32\'\n            print(\n                f\'Loading official checkpoint: {role} (dtype={dtype_used}, dual_gpu_sharded={dual_gpu and ("flow_model" in role)})\',\n                flush=True,\n            )\n            model = getattr(models, model_spec[\'name\'])(**arguments).eval()\n            weights = load_file(model_weights, device=\'cpu\')\n            incompatible = model.load_state_dict(weights, strict=False)\n            missing = [key for key in incompatible.missing_keys\n                       if key.rsplit(\'.\', 1)[-1] not in allowed_generated_buffers]\n            if missing or incompatible.unexpected_keys:\n                raise RuntimeError(f\'Checkpoint mismatch for {role}: missing={missing}; \'\n                                   f\'unexpected={incompatible.unexpected_keys}\')\n            checkpoint_audit[role] = {\n                \'repository\': repository, \'revision\': revisions[repository], \'prefix\': prefix,\n                \'generated_buffers_not_stored\': list(incompatible.missing_keys),\n            }\n            if \'flow_model\' in role:\n                if not use_bf16:\n                    model.convert_to(torch.float32)\n                if dual_gpu:\n                    model = DualGpuShardedFlowModel(model, dev0=\'cuda:0\', dev1=\'cuda:1\')\n            del weights\n            gc.collect()\n            return model.cpu()\n\n        def make_sampler(name):\n            entry = specification[name]\n            return getattr(samplers, entry[\'name\'])(**entry[\'args\'])\n\n        pipeline = Trellis2ImageTo3DPipeline(\n            models={},\n            sparse_structure_sampler=make_sampler(\'sparse_structure_sampler\'),\n            shape_slat_sampler=make_sampler(\'shape_slat_sampler\'),\n            tex_slat_sampler=make_sampler(\'tex_slat_sampler\'),\n            sparse_structure_sampler_params=specification[\'sparse_structure_sampler\'][\'params\'],\n            shape_slat_sampler_params=specification[\'shape_slat_sampler\'][\'params\'],\n            tex_slat_sampler_params=specification[\'tex_slat_sampler\'][\'params\'],\n            shape_slat_normalization=specification[\'shape_slat_normalization\'],\n            tex_slat_normalization=specification[\'tex_slat_normalization\'],\n            image_cond_model=None, rembg_model=None, low_vram=True,\n            default_pipeline_type=requested_pipeline,\n        )\n        pipeline.to(torch.device(\'cuda:0\'))\n\n        if dual_gpu:\n            @torch.no_grad()\n            def decode_tex_slat_gpu1(self, slat, subs):\n                dev1 = torch.device(\'cuda:1\')\n                dec = self.models[\'tex_slat_decoder\']\n                with torch.cuda.device(0):\n                    torch.cuda.empty_cache()\n                with torch.cuda.device(dev1):\n                    if self.low_vram:\n                        dec.to(dev1)\n                    try:\n                        slat_d1 = move_to_device(slat, dev1)\n                        subs_d1 = move_to_device(subs, dev1)\n                        ret_d1 = dec(slat_d1, guide_subs=subs_d1) * 0.5 + 0.5\n                        ret = move_to_device(ret_d1, self.device)\n                    finally:\n                        if self.low_vram:\n                            dec.cpu()\n                        torch.cuda.empty_cache()\n                return ret\n\n            pipeline.decode_tex_slat = types.MethodType(decode_tex_slat_gpu1, pipeline)\n\n        for idx in range(torch.cuda.device_count()):\n            torch.cuda.reset_peak_memory_stats(idx)\n        image = Image.open(config[\'prepared_image\']).convert(\'RGB\')\n        started = time.perf_counter()\n        ss_params = config.get(\'samplers\', {}).get(\'sparse_structure\', {\'steps\': config[\'sampling_steps\']})\n        sh_params = config.get(\'samplers\', {}).get(\'shape\', {\'steps\': config[\'sampling_steps\']})\n        tx_params = config.get(\'samplers\', {}).get(\'material\', {\'steps\': config[\'sampling_steps\']})\n\n        @torch.no_grad()\n        def run_pipeline_staged(pipeline_type):\n            # Stage 1/6: Load DINOv3, encode conditions, then free CPU/GPU memory immediately.\n            t_stage = time.perf_counter()\n            encoder_arguments = specification[\'image_cond_model\'][\'args\'].copy()\n            encoder_repo = encoder_arguments[\'model_name\']\n            revisions[encoder_repo] = pinned.get(encoder_repo) or api.model_info(encoder_repo, token=token).sha\n            encoder_directory = snapshot_download(\n                encoder_repo, revision=revisions[encoder_repo], token=token,\n                allow_patterns=[\'config.json\', \'*.safetensors\', \'*.safetensors.index.json\'],\n            )\n            encoder_arguments[\'model_name\'] = encoder_directory\n            encoder_class = getattr(image_feature_extractor, specification[\'image_cond_model\'][\'name\'])\n            encoder = encoder_class(**encoder_arguments)\n            encoder.model.float().eval()\n\n            def encode_cond(resolution):\n                encoder.image_size = resolution\n                with torch.cuda.device(encoder_device):\n                    encoder.to(torch.device(f\'cuda:{encoder_device}\'))\n                    try:\n                        feats = encoder([image]) if use_bf16 else encoder([image]).float()\n                    finally:\n                        encoder.cpu()\n                        torch.cuda.empty_cache()\n                feats = feats.to(pipeline.device)\n                if not torch.isfinite(feats).all():\n                    raise RuntimeError(\'DINO image features contain NaN/Inf.\')\n                return {\'cond\': feats, \'neg_cond\': torch.zeros_like(feats)}\n\n            cond_512 = encode_cond(512)\n            cond_1024 = encode_cond(1024) if pipeline_type != \'512\' else None\n            encoder = _release_model(encoder)\n            gc.collect()\n            _log_stage(\'1/6 dinov3_cond\', t_stage, f\'pipeline={pipeline_type}\')\n\n            # Stage 2/6: Sparse structure flow + decoder, freeing each before next load.\n            t_stage = time.perf_counter()\n            torch.manual_seed(config[\'seed\'])\n            ss_res = {\'512\': 32, \'1024\': 64, \'1024_cascade\': 32, \'1536_cascade\': 32}[pipeline_type]\n            ss_flow = load_role(\'sparse_structure_flow_model\')\n            reso = ss_flow.resolution\n            in_channels = ss_flow.in_channels\n            noise = torch.randn(1, in_channels, reso, reso, reso).to(pipeline.device)\n            ss_merged = {**pipeline.sparse_structure_sampler_params, **ss_params}\n            ss_flow.to(pipeline.device)\n            z_s = pipeline.sparse_structure_sampler.sample(\n                ss_flow, noise, **cond_512, **ss_merged, verbose=False,\n            ).samples\n            ss_flow = _release_model(ss_flow)\n            gc.collect()\n            ss_dec = load_role(\'sparse_structure_decoder\')\n            ss_dec.to(pipeline.device)\n            decoded = ss_dec(z_s) > 0\n            ss_dec = _release_model(ss_dec)\n            del z_s, noise\n            gc.collect()\n            if ss_res != decoded.shape[2]:\n                ratio = decoded.shape[2] // ss_res\n                decoded = torch.nn.functional.max_pool3d(decoded.float(), ratio, ratio, 0) > 0.5\n            coords = torch.argwhere(decoded)[:, [0, 2, 3, 4]].int()\n            del decoded\n            _cuda_cleanup()\n            _log_stage(\'2/6 sparse_structure\', t_stage, f\'coords={len(coords)}\')\n\n            # Stage 3/6: Shape SLat flow (single 512 or cascade 512->1024), releasing after sampling.\n            t_stage = time.perf_counter()\n            if pipeline_type == \'512\':\n                sh_flow_512 = load_role(\'shape_slat_flow_model_512\')\n                shape_slat = pipeline.sample_shape_slat(cond_512, sh_flow_512, coords, sh_params)\n                sh_flow_512 = _release_model(sh_flow_512)\n                gc.collect()\n                res = 512\n            elif pipeline_type == \'1024\':\n                sh_flow_1024 = load_role(\'shape_slat_flow_model_1024\')\n                shape_slat = pipeline.sample_shape_slat(cond_1024, sh_flow_1024, coords, sh_params)\n                sh_flow_1024 = _release_model(sh_flow_1024)\n                gc.collect()\n                res = 1024\n            else:\n                sh_flow_512 = load_role(\'shape_slat_flow_model_512\')\n                shape_slat = pipeline.sample_shape_slat(cond_512, sh_flow_512, coords, sh_params)\n                sh_flow_512 = _release_model(sh_flow_512)\n                gc.collect()\n                shape_dec = load_role(\'shape_slat_decoder\')\n                shape_dec.to(pipeline.device)\n                shape_dec.low_vram = True\n                hr_coords = shape_dec.upsample(shape_slat, upsample_times=4)\n                shape_dec.low_vram = False\n                shape_dec = _release_model(shape_dec)\n                gc.collect()\n                base_res = 1024 if pipeline_type == \'1024_cascade\' else 1536\n                hr_resolution = base_res\n                while True:\n                    quant_coords = torch.cat([\n                        hr_coords[:, :1],\n                        ((hr_coords[:, 1:] + 0.5) / 512 * (hr_resolution // 16)).int(),\n                    ], dim=1)\n                    coords_hr = quant_coords.unique(dim=0)\n                    if coords_hr.shape[0] < max_num_tokens or hr_resolution == 1024:\n                        if hr_resolution != base_res:\n                            print(f\'[trellis] Reduced HR resolution to {hr_resolution} ({coords_hr.shape[0]} tokens).\', flush=True)\n                        break\n                    hr_resolution -= 128\n                sh_flow_1024 = load_role(\'shape_slat_flow_model_1024\')\n                shape_slat = pipeline.sample_shape_slat(cond_1024, sh_flow_1024, coords_hr, sh_params)\n                sh_flow_1024 = _release_model(sh_flow_1024)\n                gc.collect()\n                res = hr_resolution\n            _log_stage(\'3/6 shape_slat\', t_stage, f\'tokens={len(shape_slat.coords)} res={res}\')\n\n            # Stage 4/6: Material SLat flow, then drop conditioning features.\n            t_stage = time.perf_counter()\n            tex_role = \'tex_slat_flow_model_512\' if res == 512 else \'tex_slat_flow_model_1024\'\n            tex_cond = cond_512 if res == 512 else cond_1024\n            tx_flow = load_role(tex_role)\n            tex_slat = pipeline.sample_tex_slat(tex_cond, tx_flow, shape_slat, tx_params)\n            tx_flow = _release_model(tx_flow)\n            del cond_512, cond_1024, tex_cond\n            _cuda_cleanup()\n            _log_stage(\'4/6 tex_slat\', t_stage, f\'tokens={len(tex_slat.coords)}\')\n\n            # Stage 5/6: Decode shape then texture (one decoder resident at a time).\n            t_stage = time.perf_counter()\n            sh_dec = load_role(\'shape_slat_decoder\')\n            pipeline.models[\'shape_slat_decoder\'] = sh_dec\n            meshes, subs = pipeline.decode_shape_slat(shape_slat, res)\n            pipeline.models.clear()\n            sh_dec = _release_model(sh_dec)\n            gc.collect()\n            tx_dec = load_role(\'tex_slat_decoder\')\n            pipeline.models[\'tex_slat_decoder\'] = tx_dec\n            tex_voxels = pipeline.decode_tex_slat(tex_slat, subs)\n            pipeline.models.clear()\n            tx_dec = _release_model(tx_dec)\n            gc.collect()\n            m, v = meshes[0], tex_voxels[0]\n            m.fill_holes()\n            built_mesh = MeshWithVoxel(\n                m.vertices, m.faces,\n                origin=[-0.5, -0.5, -0.5],\n                voxel_size=1 / res,\n                coords=v.coords[:, 1:],\n                attrs=v.feats,\n                voxel_shape=torch.Size([*v.shape, *v.spatial_shape]),\n                layout=pipeline.pbr_attr_layout,\n            )\n            del meshes, subs, tex_voxels, m, v\n            _cuda_cleanup()\n            write_json(output / \'model_revision.json\', {\'revisions\': revisions, \'checkpoints\': checkpoint_audit})\n            _log_stage(\n                \'5/6 decode_latent\',\n                t_stage,\n                f\'raw_verts={len(built_mesh.vertices)} raw_faces={len(built_mesh.faces)}\',\n            )\n            return built_mesh, (shape_slat, tex_slat, res)\n\n        fallback_ladder = {\n            \'1536_cascade\': (\'1024_cascade\', \'512\'),\n            \'1024_cascade\': (\'512\',),\n            \'1024\': (\'512\',),\n            \'512\': (),\n        }[requested_pipeline]\n        actual_pipeline = requested_pipeline\n        fallback_reason = None\n        try:\n            mesh, latents = run_pipeline_staged(requested_pipeline)\n        except torch.OutOfMemoryError as oom_exc:\n            if not fallback_ladder:\n                raise\n            mesh = None\n            latents = None\n            last_oom = oom_exc\n            for cand_pipe in fallback_ladder:\n                print(f\'[trellis] {actual_pipeline} hit CUDA OOM ({last_oom}); falling back to {cand_pipe}...\', flush=True)\n                pipeline.models.clear()\n                _cuda_cleanup()\n                actual_pipeline = cand_pipe\n                fallback_reason = f\'oom_{requested_pipeline}\'\n                try:\n                    mesh, latents = run_pipeline_staged(cand_pipe)\n                    break\n                except torch.OutOfMemoryError as next_oom:\n                    last_oom = next_oom\n            if mesh is None:\n                raise last_oom\n\n        for label, tensor in ((\'vertices\', mesh.vertices), (\'attributes\', mesh.attrs),\n                              (\'shape latent\', latents[0].feats), (\'material latent\', latents[1].feats)):\n            if not torch.isfinite(tensor).all():\n                raise RuntimeError(f\'Generated {label} contains NaN/Inf; no result will be declared successful.\')\n        if len(mesh.vertices) == 0 or len(mesh.faces) == 0:\n            raise RuntimeError(\'The model generated an empty mesh.\')\n        generation_seconds = time.perf_counter() - started\n        np.savez_compressed(output / \'raw_mesh_and_materials.npz\',\n                            vertices=mesh.vertices.cpu().numpy(), faces=mesh.faces.cpu().numpy(),\n                            coords=mesh.coords.cpu().numpy(), attributes=mesh.attrs.float().cpu().numpy(),\n                            voxel_size=np.array(mesh.voxel_size), origin=mesh.origin.cpu().numpy(),\n                            voxel_shape=np.array(list(mesh.voxel_shape)))\n        layout_dict = {k: [v.start, v.stop, v.step] for k, v in mesh.layout.items()}\n        metadata = {\n            \'attribute_layout\': layout_dict,\n            \'voxel_size\': float(mesh.voxel_size.item() if hasattr(mesh.voxel_size, \'item\') else mesh.voxel_size),\n            \'voxel_shape\': list(mesh.voxel_shape),\n            \'origin\': mesh.origin.cpu().tolist() if hasattr(mesh.origin, \'cpu\') else list(mesh.origin),\n        }\n        (output / \'raw_mesh_metadata.json\').write_text(json.dumps(metadata, indent=2), encoding=\'utf-8\')\n        if config.get(\'save_latents\', False):\n            torch.save({\'shape_features\': latents[0].feats.cpu(), \'material_features\': latents[1].feats.cpu(),\n                        \'coords\': latents[0].coords.cpu(), \'resolution\': latents[2]}, output / \'generated_latents.pt\')\n        del latents\n        pipeline.models.clear()\n        _cuda_cleanup()\n\n        # 6. Export watertight PBR GLB (HF Space simplify/remesh + fallback ladder) and mobile <=35k GLB.\n        started = time.perf_counter()\n        if hasattr(mesh, \'simplify\'):\n            mesh.simplify(16777216)\n        req_remesh = bool(config.get(\'remesh\', True))\n        req_tex = int(config.get(\'texture_size\', 2048))\n        req_dec = int(config.get(\'decimation_target\', 300000))\n        remesh_band = float(config.get(\'remesh_band\', 1))\n        remesh_project = float(config.get(\'remesh_project\', 0.0))\n\n        export_attempts = [(req_remesh, req_tex, req_dec, None)]\n        if req_dec > 150000:\n            export_attempts.append((req_remesh, req_tex, 150000, \'export_downgraded_150k\'))\n        if req_tex > 1024 or req_dec > 100000:\n            export_attempts.append((req_remesh, 1024, min(req_dec, 100000), \'export_downgraded_1024\'))\n        if req_remesh:\n            export_attempts.append((False, 1024, min(req_dec, 100000), \'export_fallback_no_remesh\'))\n\n        asset = None\n        remesh_used = req_remesh\n        tex_used = req_tex\n        export_fallback_reason = None\n        last_export_err = None\n        for use_remesh, use_tex, use_dec, fb_tag in export_attempts:\n            try:\n                asset = o_voxel.postprocess.to_glb(\n                    vertices=mesh.vertices, faces=mesh.faces, attr_volume=mesh.attrs,\n                    coords=mesh.coords, attr_layout=mesh.layout, voxel_size=mesh.voxel_size,\n                    aabb=[[-0.5, -0.5, -0.5], [0.5, 0.5, 0.5]],\n                    decimation_target=use_dec, texture_size=use_tex,\n                    remesh=use_remesh, remesh_band=remesh_band, remesh_project=remesh_project,\n                    use_tqdm=False,\n                )\n                remesh_used = use_remesh\n                tex_used = use_tex\n                export_fallback_reason = fb_tag\n                break\n            except Exception as exp_err:\n                last_export_err = exp_err\n                print(f\'[trellis] export attempt (remesh={use_remesh}, tex={use_tex}) failed: {exp_err}\', flush=True)\n                _cuda_cleanup()\n        if asset is None:\n            raise last_export_err\n\n        target = output / \'generated_original.glb\'\n        asset.export(str(target), extension_webp=True)\n        with target.open(\'rb\') as stream:\n            if stream.read(4) != b\'glTF\' or target.stat().st_size <= 20:\n                raise RuntimeError(\'Export did not create a valid GLB container.\')\n\n        mob_verts = None\n        mob_faces = None\n        mob_target = config.get(\'mobile_decimation_target\')\n        if mob_target:\n            mob_tex = int(config.get(\'mobile_texture_size\', 1024))\n            mob_asset = None\n            mobile_status = \'failed\'\n            for use_remesh in ((True, False) if remesh_used else (False,)):\n                try:\n                    mob_asset = o_voxel.postprocess.to_glb(\n                        vertices=mesh.vertices, faces=mesh.faces, attr_volume=mesh.attrs,\n                        coords=mesh.coords, attr_layout=mesh.layout, voxel_size=mesh.voxel_size,\n                        aabb=[[-0.5, -0.5, -0.5], [0.5, 0.5, 0.5]],\n                        decimation_target=int(mob_target), texture_size=mob_tex,\n                        remesh=use_remesh, remesh_band=remesh_band, remesh_project=remesh_project,\n                        use_tqdm=False,\n                    )\n                    break\n                except Exception as mob_err:\n                    print(f\'[trellis] mobile export (remesh={use_remesh}) warning: {mob_err}\', flush=True)\n                    _cuda_cleanup()\n            if mob_asset is not None:\n                mob_path = output / \'generated_mobile.glb\'\n                try:\n                    mob_asset.export(str(mob_path), extension_webp=True)\n                    with mob_path.open(\'rb\') as stream:\n                        if stream.read(4) != b\'glTF\' or mob_path.stat().st_size <= 20:\n                            raise RuntimeError(\'Mobile export did not create a valid GLB container.\')\n                    mob_verts = int(len(mob_asset.vertices))\n                    mob_faces = int(len(mob_asset.faces))\n                    mobile_status = \'success\'\n                except Exception as mob_err:\n                    mob_path.unlink(missing_ok=True)\n                    print(f\'[trellis] mobile serialization warning: {type(mob_err).__name__}\', flush=True)\n        export_seconds = time.perf_counter() - started\n        _log_stage(\n            \'6/6 export_glb\',\n            started,\n            f\'verts={len(asset.vertices)} faces={len(asset.faces)} mob_faces={mob_faces} remesh={remesh_used} tex={tex_used}\',\n        )\n        peak_gpu0 = torch.cuda.max_memory_allocated(0) / 2**30\n        peak_gpu1 = (torch.cuda.max_memory_allocated(1) / 2**30) if torch.cuda.device_count() > 1 else 0.0\n        status.update(\n            mobile_status=locals().get(\'mobile_status\', \'not_run\'),\n            status=\'success\',\n            generation_seconds=round(generation_seconds, 2),\n            export_seconds=round(export_seconds, 2),\n            elapsed_seconds=round(generation_seconds + export_seconds, 2),\n            output=str(target),\n            peak_gpu0_gib=round(peak_gpu0, 3),\n            peak_gpu1_gib=round(peak_gpu1, 3),\n            total_peak_vram_gib=round(peak_gpu0 + peak_gpu1, 3),\n            encoder_gpu=encoder_device,\n            dtype_used=dtype_used,\n            dual_gpu_sharded=bool(dual_gpu),\n            pipeline_type=requested_pipeline,\n            actual_pipeline_type=actual_pipeline,\n            fallback_reason=fallback_reason,\n            remesh_used=bool(remesh_used),\n            texture_size_used=int(tex_used),\n            export_fallback_reason=export_fallback_reason,\n            vertices=int(len(asset.vertices)),\n            faces=int(len(asset.faces)),\n            mobile_vertices=mob_verts,\n            mobile_faces=mob_faces,\n            ended_at=time.time(),\n        )\n        write_json(status_path, status)\n        print(\'Saved:\', target, flush=True)\n    except Exception as error:\n        status.update(status=\'failed\', error_type=type(error).__name__, ended_at=time.time())\n        write_json(status_path, status)\n        traceback.print_exc()\n        raise\n\n\nif __name__ == \'__main__\':\n    parser = argparse.ArgumentParser()\n    parser.add_argument(\'config\')\n    main(parser.parse_args().config)\n', encoding='utf-8')
(SCRATCH_ROOT / 'trellis_prepare_full_plate.py').write_text('"""Preserve the complete plate cutout and pad it for image-to-3D conditioning."""\n\nimport argparse\nimport json\nimport math\nfrom pathlib import Path\n\nimport numpy as np\nfrom PIL import Image, ImageOps\n\n\ndef prepare_full_plate(source, destination, margin=0.08, max_size=1024, reviewed_mask=None,\n                       crop_to_alpha=False, require_alpha=False):\n    # 1. Retain the entire subject, including the plate rim and food (never run rembg here).\n    if not 0 <= margin <= 0.5 or max_size < 32:\n        raise ValueError(\'Use a margin from 0 to 0.5 and an image size of at least 32 pixels.\')\n    destination = Path(destination)\n    destination.mkdir(parents=True, exist_ok=True)\n    with Image.open(source) as opened:\n        image = ImageOps.exif_transpose(opened).convert(\'RGBA\')\n    original_size = image.size\n    image.save(destination / \'original_input.png\')\n\n    # An optional hand-reviewed mask must include BOTH the complete plate and the food.\n    if reviewed_mask:\n        with Image.open(reviewed_mask) as opened:\n            mask = ImageOps.exif_transpose(opened).convert(\'L\')\n        if mask.size != image.size:\n            raise ValueError(\'The reviewed full-plate mask must match the original image dimensions.\')\n        alpha = np.asarray(image.getchannel(\'A\'), dtype=np.uint16)\n        alpha = ((alpha * np.asarray(mask, dtype=np.uint16) + 127) // 255).astype(np.uint8)\n        image.putalpha(Image.fromarray(alpha))\n\n    alpha_arr = np.asarray(image.getchannel(\'A\'), dtype=np.uint8)\n    if int(alpha_arr.max()) == 0:\n        raise ValueError(\'The source or reviewed mask is fully transparent. Supply the original full plate image.\')\n    if require_alpha and int(alpha_arr.min()) == 255:\n        raise ValueError(\'Input image has no transparent background mask (all alpha == 255). Pass the plate-preserving cutout_rgba.png or a reviewed mask.\')\n\n    # Optionally crop tightly around the non-zero alpha bbox (preserving translucent rims) before padding.\n    alpha_bbox = None\n    foreground_cropped = False\n    if crop_to_alpha:\n        ys, xs = np.where(alpha_arr > 0)\n        x0, y0, x1, y1 = int(xs.min()), int(ys.min()), int(xs.max()) + 1, int(ys.max()) + 1\n        alpha_bbox = [x0, y0, x1, y1]\n        if (x0, y0, x1, y1) != (0, 0, image.width, image.height):\n            image = image.crop((x0, y0, x1, y1))\n        foreground_cropped = True\n\n    # 2. Fit the prepared subject inside a padded square; premultiply onto black for TRELLIS.2 conditioning.\n    longest_side = max(image.size)\n    border = math.ceil(longest_side * margin)\n    square_size = longest_side + 2 * border\n    scale = min(1, max_size / square_size)\n    resized_size = tuple(max(1, round(value * scale)) for value in image.size)\n    square_size = min(square_size, max_size)\n    if resized_size != image.size:\n        image = image.resize(resized_size, Image.Resampling.LANCZOS)\n    left = (square_size - image.width) // 2\n    top = (square_size - image.height) // 2\n    square = Image.new(\'RGBA\', (square_size, square_size), (0, 0, 0, 0))\n    square.paste(image, (left, top))\n    square.save(destination / \'full_plate_rgba.png\')\n\n    prepared = Image.alpha_composite(Image.new(\'RGBA\', square.size, (0, 0, 0, 255)), square).convert(\'RGB\')\n    prepared.save(destination / \'prepared_image.png\')\n    report = {\n        \'source\': str(source), \'original_size\': list(original_size),\n        \'prepared_size\': list(prepared.size), \'resized_image_size\': list(resized_size),\n        \'image_offset\': [left, top], \'margin\': margin,\n        \'automatic_background_removal\': False, \'foreground_cropped\': foreground_cropped,\n        \'alpha_bbox\': alpha_bbox, \'require_alpha\': bool(require_alpha),\n        \'reviewed_full_plate_mask\': str(reviewed_mask) if reviewed_mask else None,\n        \'existing_alpha_preserved\': True,\n        \'note\': \'Preserves reviewed/cutout RGBA including translucent plate rims; crops to alpha bbox when crop_to_alpha=True.\',\n    }\n    (destination / \'image_preparation.json\').write_text(json.dumps(report, indent=2), encoding=\'utf-8\')\n    return report\n\n\nif __name__ == \'__main__\':\n    parser = argparse.ArgumentParser()\n    parser.add_argument(\'source\')\n    parser.add_argument(\'destination\')\n    parser.add_argument(\'--margin\', type=float, default=0.08)\n    parser.add_argument(\'--max-size\', type=int, default=1024)\n    parser.add_argument(\'--reviewed-mask\')\n    parser.add_argument(\'--crop-to-alpha\', action=\'store_true\')\n    parser.add_argument(\'--require-alpha\', action=\'store_true\')\n    arguments = parser.parse_args()\n    print(json.dumps(prepare_full_plate(arguments.source, arguments.destination, arguments.margin,\n                                        arguments.max_size, arguments.reviewed_mask,\n                                        crop_to_alpha=arguments.crop_to_alpha,\n                                        require_alpha=arguments.require_alpha), indent=2))\n', encoding='utf-8')
(SCRATCH_ROOT / 'trellis_native_space_tools.py').write_text('"""HF Space render modes and repeatable GLB export from saved mesh/material data."""\n\nimport argparse\nimport base64\nimport html\nimport importlib\nimport json\nimport os\nfrom pathlib import Path\nimport sys\nimport time\nimport traceback\n\nfrom PIL import Image, ImageDraw\n\n\nMODES = (\n    (\'normal\', \'Normal\'),\n    (\'clay\', \'Clay render\'),\n    (\'base_color\', \'Base color\'),\n    (\'shaded_forest\', \'HDRI forest\'),\n    (\'shaded_sunset\', \'HDRI sunset\'),\n    (\'shaded_courtyard\', \'HDRI courtyard\'),\n)\n\n\ndef write_gallery(frames, destination):\n    # 1. Keep the six Space render modes usable offline, with the same eight view angles.\n    lengths = {len(frames[key]) for key, _ in MODES}\n    if len(lengths) != 1 or not lengths or min(lengths) < 1:\n        raise ValueError(\'Every Space render mode must have the same nonzero number of views.\')\n    view_count = lengths.pop()\n    images = {\n        key: [\'data:image/png;base64,\' + base64.b64encode(Path(path).read_bytes()).decode()\n              for path in frames[key]]\n        for key, _ in MODES\n    }\n    buttons = \'\'.join(\n        f\'<button type="button" data-mode="{key}" aria-pressed="false">{html.escape(label)}</button>\'\n        for key, label in MODES\n    )\n    document = \'\'\'<!doctype html><html lang="en"><head><meta charset="utf-8">\n    <meta name="viewport" content="width=device-width,initial-scale=1">\n    <title>Full plate — TRELLIS.2 preview modes</title>\n    <style>\n    body{font:16px system-ui,sans-serif;background:#161b24;color:#f4f6fa;margin:0;padding:20px}\n    main{max-width:1000px;margin:auto}h1{font-size:22px}.modes{display:flex;gap:8px;flex-wrap:wrap}\n    button{padding:10px 14px;border:1px solid #737c8d;border-radius:8px;background:#273143;color:inherit;cursor:pointer}\n    button[aria-pressed=true]{background:#245c9b;border-color:#9fcaff}\n    img{display:block;width:100%;height:min(65vh,600px);object-fit:contain;background:#000;margin:16px 0}\n    input{width:min(100%,600px)}p{color:#c7ceda;line-height:1.5}\n    </style></head><body><main><h1>Full plate and food</h1>\n    <div class="modes">__BUTTONS__</div><img id="preview" alt="Generated full plate">\n    <label for="angle">View angle: <span id="view-number"></span> / __COUNT__</label><br>\n    <input id="angle" type="range" min="0" max="__MAX__" value="__DEFAULT__" step="1">\n    <p>All six modes show the same generated object. Normal shows surface direction; clay shows\n    shape; base color shows its color; the HDRI modes show three lighting environments.</p>\n    <script>\n    const frames=__IMAGES__;\n    const angle=document.getElementById(\'angle\');\n    let mode=\'shaded_forest\';\n    function refresh(){\n        document.getElementById(\'preview\').src=frames[mode][Number(angle.value)];\n        document.getElementById(\'view-number\').textContent=Number(angle.value)+1;\n        document.querySelectorAll(\'[data-mode]\').forEach(button=>{\n            button.setAttribute(\'aria-pressed\',String(button.dataset.mode===mode));\n        });\n    }\n    document.querySelectorAll(\'[data-mode]\').forEach(button=>button.addEventListener(\'click\',()=>{\n        mode=button.dataset.mode;refresh();\n    }));\n    angle.addEventListener(\'input\',refresh);refresh();\n    </script></main></body></html>\'\'\'\n    document = document.replace(\'__BUTTONS__\', buttons).replace(\'__COUNT__\', str(view_count))\n    document = document.replace(\'__MAX__\', str(view_count - 1)).replace(\'__DEFAULT__\', str(min(3, view_count - 1)))\n    document = document.replace(\'__IMAGES__\', json.dumps(images))\n    destination = Path(destination)\n    destination.write_text(document, encoding=\'utf-8\')\n    return destination\n\n\ndef load_saved_mesh(output, torch, device=None):\n    import numpy as np\n    from trellis2.representations import MeshWithVoxel\n\n    if device is None:\n        device = torch.device(\'cuda\' if torch.cuda.is_available() else \'cpu\')\n    elif isinstance(device, str):\n        device = torch.device(device)\n\n    metadata = json.loads((output / \'raw_mesh_metadata.json\').read_text(encoding=\'utf-8\'))\n    layout = {key: slice(*value) for key, value in metadata[\'attribute_layout\'].items()}\n    with np.load(output / \'raw_mesh_and_materials.npz\', allow_pickle=False) as saved:\n        if \'voxel_shape\' in saved:\n            voxel_shape = torch.Size(saved[\'voxel_shape\'].tolist())\n        else:\n            voxel_shape = torch.Size(metadata[\'voxel_shape\'])\n        mesh = MeshWithVoxel(\n            vertices=torch.from_numpy(saved[\'vertices\']).to(device),\n            faces=torch.from_numpy(saved[\'faces\']).to(device),\n            origin=saved[\'origin\'].tolist(),\n            voxel_size=float(saved[\'voxel_size\'].item() if hasattr(saved[\'voxel_size\'], \'item\') else saved[\'voxel_size\']),\n            coords=torch.from_numpy(saved[\'coords\']).to(device),\n            attrs=torch.from_numpy(saved[\'attributes\']).to(device),\n            voxel_shape=voxel_shape, layout=layout,\n        )\n    if not len(mesh.vertices) or not len(mesh.faces):\n        raise ValueError(\'Saved mesh is empty.\')\n    if not torch.isfinite(mesh.vertices).all() or not torch.isfinite(mesh.attrs).all():\n        raise ValueError(\'Saved mesh/materials contain NaN or Inf.\')\n    return mesh\n\n\ndef render_modes(mesh, configuration, directory, torch):\n    import cv2\n    import numpy as np\n    from trellis2.renderers import EnvMap\n    from trellis2.utils import render_utils\n\n    resolution = int(configuration[\'preview_resolution\'])\n    supersampling = int(configuration[\'preview_ssaa\'])\n    if resolution not in (256, 512, 1024) or supersampling not in (1, 2):\n        raise ValueError(\'Preview resolution must be 256, 512 or 1024; SSAA must be 1 or 2.\')\n    # 2. Use the same official HDRIs, PBR renderer and camera positions as the HF Space.\n    environment_maps = {}\n    for name in (\'forest\', \'sunset\', \'courtyard\'):\n        path = Path(configuration[\'source_dir\']) / \'assets\' / \'hdri\' / (name + \'.exr\')\n        image = cv2.imread(str(path), cv2.IMREAD_UNCHANGED)\n        if image is None:\n            raise RuntimeError(f\'The official lighting asset could not be read: {path}\')\n        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)\n        environment_maps[name] = EnvMap(torch.tensor(image, dtype=torch.float32, device=\'cuda\'))\n    if len(mesh.faces) > 16777216:\n        mesh.simplify(16777216)  # The same nvdiffrast face limit as the Space; saved geometry is unchanged.\n    yaws = (np.linspace(0, 2 * np.pi, 8, endpoint=False) - 16 / 180 * np.pi).tolist()\n    pitches = [20 / 180 * np.pi] * 8\n    extrinsics, intrinsics = render_utils.yaw_pitch_r_fov_to_extrinsics_intrinsics(yaws, pitches, 2, 36)\n    renderer = render_utils.get_renderer(mesh, resolution=resolution, ssaa=supersampling, peel_layers=8)\n    frames = {key: [] for key, _ in MODES}\n    for view, (extrinsic, intrinsic) in enumerate(zip(extrinsics, intrinsics)):\n        print(f\'Rendering all six modes: angle {view + 1}/8\', flush=True)\n        torch.manual_seed(42 + view)\n        result = renderer.render(mesh, extrinsic, intrinsic, envmap=environment_maps)\n        for key, _ in MODES:\n            image = result[key]\n            if not torch.isfinite(image).all():\n                raise RuntimeError(f\'The {key} preview contains NaN/Inf.\')\n            if image.dim() == 2:\n                image = image[None].repeat(3, 1, 1)\n            pixels = np.clip(image.detach().cpu().numpy().transpose(1, 2, 0) * 255, 0, 255).astype(np.uint8)\n            target = directory / f\'{key}_{view:02d}.png\'\n            Image.fromarray(pixels).save(target)\n            frames[key].append(target)\n        del result\n    gallery = write_gallery(frames, directory / \'all_hf_modes.html\')\n    # Keep a compact comparison in notebook output instead of embedding all 48 full images.\n    sheet = Image.new(\'RGB\', (3 * 320, 2 * 350), \'#161b24\')\n    draw = ImageDraw.Draw(sheet)\n    for index, (key, label) in enumerate(MODES):\n        x, y = (index % 3) * 320, (index // 3) * 350\n        with Image.open(frames[key][3]) as image:\n            sheet.paste(image.resize((320, 320), Image.Resampling.LANCZOS), (x, y + 30))\n        draw.text((x + 10, y + 8), label, fill=\'white\')\n    sheet.save(directory / \'all_modes_comparison.jpg\', quality=90)\n    return {\'gallery\': str(gallery), \'modes\': [key for key, _ in MODES], \'views_per_mode\': 8,\n            \'preview_resolution\': resolution, \'preview_ssaa\': supersampling,\n            \'rendered_from\': \'saved raw mesh and material voxels\', \'geometry_changed\': False}\n\n\ndef export_variant(mesh, configuration, directory):\n    faces = int(configuration[\'decimation_target\'])\n    texture = int(configuration[\'texture_size\'])\n    if not 100000 <= faces <= 500000 or faces % 10000:\n        raise ValueError(\'Use the Space face-count range: 100000–500000, in steps of 10000.\')\n    if texture not in (1024, 2048, 3072, 4096):\n        raise ValueError(\'Use a Space texture size: 1024, 2048, 3072 or 4096.\')\n    import o_voxel\n    # 3. Re-export from unmodified saved geometry; never chain variants or regenerate the food.\n    asset = o_voxel.postprocess.to_glb(\n        vertices=mesh.vertices, faces=mesh.faces, attr_volume=mesh.attrs,\n        coords=mesh.coords, attr_layout=mesh.layout, voxel_size=mesh.voxel_size,\n        aabb=[[-0.5, -0.5, -0.5], [0.5, 0.5, 0.5]],\n        decimation_target=faces, texture_size=texture,\n        remesh=bool(configuration[\'remesh\']), remesh_band=1, remesh_project=0,\n        use_tqdm=True,\n    )\n    target = directory / f\'full_plate_faces{faces}_texture{texture}_remesh{int(configuration["remesh"])}.glb\'\n    asset.export(str(target), extension_webp=True)\n    with target.open(\'rb\') as stream:\n        if stream.read(4) != b\'glTF\' or target.stat().st_size <= 20:\n            raise RuntimeError(\'Variant export did not create a valid GLB container.\')\n    return {\'output\': str(target), \'decimation_target\': faces, \'texture_size\': texture,\n            \'remesh\': bool(configuration[\'remesh\'])}\n\n\ndef main(config_path, action):\n    configuration = json.loads(Path(config_path).read_text(encoding=\'utf-8\'))\n    output = Path(configuration[\'output_dir\'])\n    directory = Path(configuration[\'result_dir\'])\n    directory.mkdir(parents=True, exist_ok=True)\n    status = {\'status\': \'running\', \'action\': action, \'started_at\': time.time()}\n    status_path = directory / \'status.json\'\n    status_path.write_text(json.dumps(status, indent=2), encoding=\'utf-8\')\n    try:\n        os.environ[\'OPENCV_IO_ENABLE_OPENEXR\'] = \'1\'\n        os.environ[\'ATTN_BACKEND\'] = \'sdpa\'\n        os.environ[\'SPARSE_CONV_BACKEND\'] = \'none\'\n        os.environ[\'SPARSE_ATTN_BACKEND\'] = \'xformers\'\n        os.environ[\'PYTORCH_CUDA_ALLOC_CONF\'] = \'expandable_segments:True\'\n        sys.path.insert(0, configuration[\'source_dir\'])\n        import torch\n        from trellis_native_compat import install_adapters\n        install_adapters()\n        # PBR renderer imports this function inside render(); patch its original module too.\n        sampling = importlib.import_module(\'flex_gemm.ops.grid_sample\')\n        from trellis_native_compat import sparse_grid_sample\n        sampling.grid_sample_3d = sparse_grid_sample\n        if not torch.cuda.is_available():\n            raise RuntimeError(\'Rendering and native GLB export require the Kaggle CUDA session.\')\n        torch.cuda.set_device(0)\n        device = torch.device(\'cuda:0\')\n        with torch.inference_mode():\n            mesh = load_saved_mesh(output, torch, device=device)\n            if action == \'render\':\n                result = render_modes(mesh, configuration, directory, torch)\n            else:\n                result = export_variant(mesh, configuration, directory)\n        status.update(result)\n        status.update(status=\'success\', ended_at=time.time())\n        print(json.dumps(status, indent=2), flush=True)\n    except Exception as error:\n        status.update(status=\'failed\', message=str(error), error_type=type(error).__name__, ended_at=time.time())\n        traceback.print_exc()\n        raise\n    finally:\n        status_path.write_text(json.dumps(status, indent=2), encoding=\'utf-8\')\n\n\nif __name__ == \'__main__\':\n    parser = argparse.ArgumentParser()\n    parser.add_argument(\'action\', choices=(\'render\', \'export\'))\n    parser.add_argument(\'config\')\n    arguments = parser.parse_args()\n    main(arguments.config, arguments.action)\n', encoding='utf-8')
(SCRATCH_ROOT / 'test_trellis_native_compat.py').write_text('"""Small numerical checks; run on CPU locally and CUDA inside the notebook."""\n\nimport itertools\nimport json\nimport os\nfrom pathlib import Path\nimport sys\nimport tempfile\nimport unittest\nfrom types import SimpleNamespace\nfrom unittest.mock import patch\n\nimport numpy as np\nfrom PIL import Image\n\ntest_packages = Path(__file__).with_name(\'trellis-native-test-packages\')\nif test_packages.exists():\n    sys.path.insert(0, str(test_packages))\n\nimport torch\nimport torch.nn.functional as F\nfrom trellis_native_compat import CoordinateLookup, submanifold_convolution, sparse_attention, sparse_grid_sample\nfrom trellis_prepare_full_plate import prepare_full_plate\nfrom trellis_native_space_tools import load_saved_mesh, write_gallery, export_variant, MODES\n\n\nclass MeshWithVoxel:\n    """Test double for metadata I/O; native rendering still requires the official class."""\n    def __init__(self, **values):\n        self.__dict__.update(values)\n        self.origin = torch.tensor(values[\'origin\'], dtype=torch.float32)\n\n\nclass Sequence:\n    def __init__(self, feats, lengths=None, layout=None):\n        self.feats = feats\n        self._cache = {}\n        if layout is not None:\n            self.layout = list(layout)\n        else:\n            self.layout = []\n            start = 0\n            for length in (lengths or [len(feats)]):\n                self.layout.append(slice(start, start + length))\n                start += length\n\n    @property\n    def device(self):\n        return self.feats.device\n\n    @property\n    def dtype(self):\n        return self.feats.dtype\n\n    def type(self, dt):\n        return self.replace(self.feats.to(dt))\n\n    def replace(self, feats):\n        return Sequence(feats, layout=self.layout)\n\n\nclass NativeAdaptersTest(unittest.TestCase):\n    def setUp(self):\n        torch.manual_seed(42)\n        self.device = os.environ.get(\'TRELLIS_TEST_DEVICE\', \'cpu\')\n\n    def test_coordinate_boundaries_do_not_alias_another_batch(self):\n        coords = torch.tensor([[0, 0, 0, 0], [0, 1, 2, 2], [1, 0, 0, 0]], device=self.device)\n        lookup = CoordinateLookup(coords, (2, 3, 3))\n        query = torch.tensor([[0, 2, 0, 0], [0, -1, 0, 0], [1, 0, 0, 0], [0, 1, 2, 2]], device=self.device)\n        self.assertEqual(lookup.find(query).tolist(), [-1, -1, 2, 1])\n\n    def test_move_to_device_clears_sparse_and_varlen_caches(self):\n        from trellis_native_compat import move_to_device\n        class DummySparse:\n            def __init__(self, feats, coords, shape=None, scale=None):\n                self.feats = feats\n                self.coords = coords\n                self._shape = shape\n                self._scale = scale\n                self._spatial_cache = {}\n        sp_in = DummySparse(torch.ones(2, 3), torch.zeros(2, 4, dtype=torch.int32), shape=(1, 3), scale=(1, 1, 1))\n        sp_in._spatial_cache[\'stale\'] = \'cached\'\n        sp_out = move_to_device(sp_in, \'cpu\')\n        self.assertEqual(sp_out._spatial_cache, {})\n        self.assertEqual(sp_out._shape, (1, 3))\n        seq = Sequence(torch.randn(4, 2), [2, 2])\n        seq._cache[\'stale\'] = torch.tensor([1])\n        seq_out = move_to_device(seq, \'cpu\')\n        self.assertEqual(seq_out._cache, {})\n\n    def test_dual_gpu_sharded_flow_matches_unsharded_dense_and_sparse(self):\n        from trellis_native_compat import DualGpuShardedFlowModel\n        class DummyDenseBlock(torch.nn.Module):\n            def __init__(self, c):\n                super().__init__()\n                self.proj = torch.nn.Linear(c, c)\n            def forward(self, h, t_emb, cond, rope_phases=None):\n                return self.proj(h) + t_emb[:, None, :] + cond.mean(dim=1, keepdim=True) + (rope_phases.mean() if rope_phases is not None else 0.0)\n\n        class DummyDenseFlow(torch.nn.Module):\n            def __init__(self):\n                super().__init__()\n                self.resolution = 2\n                self.in_channels = 4\n                self.dtype = torch.float32\n                self.pe_mode = \'rope\'\n                self.share_mod = False\n                self.input_layer = torch.nn.Linear(4, 8)\n                self.t_embedder = torch.nn.Linear(1, 8)\n                self.blocks = torch.nn.ModuleList([DummyDenseBlock(8) for _ in range(4)])\n                self.out_layer = torch.nn.Linear(8, 4)\n                self.register_buffer(\'rope_phases\', torch.ones(8, 8) * 0.25)\n            def forward(self, x, t, cond):\n                h = self.input_layer(x.view(*x.shape[:2], -1).permute(0, 2, 1).contiguous())\n                te = self.t_embedder(t)\n                for b in self.blocks:\n                    h = b(h, te, cond, self.rope_phases)\n                h = torch.nn.functional.layer_norm(h, h.shape[-1:])\n                return self.out_layer(h).permute(0, 2, 1).view(x.shape[0], 4, 2, 2, 2).contiguous()\n\n        dense_m = DummyDenseFlow().eval()\n        dev0 = \'cuda:0\' if torch.cuda.device_count() >= 2 else \'cpu\'\n        dev1 = \'cuda:1\' if torch.cuda.device_count() >= 2 else \'cpu\'\n        xd = torch.randn(1, 4, 2, 2, 2, device=dev0)\n        td = torch.tensor([[0.5]], device=dev0)\n        cd = torch.randn(1, 3, 8, device=dev0)\n        with torch.no_grad():\n            expected_d = dense_m.to(dev0)(xd, td, cd)\n            dense_m.cpu()\n            sharded_d = DualGpuShardedFlowModel(dense_m, dev0=dev0, dev1=dev1).to(dev0)\n            actual_d = sharded_d(xd, td, cd)\n            torch.testing.assert_close(actual_d, expected_d, rtol=1e-5, atol=1e-5)\n\n        class DummySparseTensor:\n            def __init__(self, feats, coords, layout=None, shape=None, scale=None):\n                self.feats = feats\n                self.coords = coords\n                self.layout = layout if layout is not None else [slice(0, len(feats))]\n                self._shape = shape\n                self._scale = scale\n                self._spatial_cache = {}\n            @property\n            def device(self):\n                return self.feats.device\n            @property\n            def dtype(self):\n                return self.feats.dtype\n            def type(self, dt):\n                return self.replace(self.feats.to(dt))\n            def replace(self, feats):\n                return DummySparseTensor(feats, self.coords, list(self.layout), self._shape, self._scale)\n\n        class DummySparseLin(torch.nn.Module):\n            def __init__(self, c_in, c_out):\n                super().__init__()\n                self.lin = torch.nn.Linear(c_in, c_out)\n            def forward(self, sp):\n                return sp.replace(self.lin(sp.feats))\n\n        class DummySparseBlock(torch.nn.Module):\n            def __init__(self, c):\n                super().__init__()\n                self.proj = torch.nn.Linear(c, c)\n            def forward(self, h, t_emb, cond):\n                return h.replace(self.proj(h.feats) + t_emb[0:1, :] + cond.feats.mean(dim=0, keepdim=True))\n\n        class DummySparseFlow(torch.nn.Module):\n            def __init__(self):\n                super().__init__()\n                self.resolution = 4\n                self.in_channels = 6\n                self.dtype = torch.float32\n                self.pe_mode = \'rope\'\n                self.share_mod = False\n                self.input_layer = DummySparseLin(6, 8)\n                self.t_embedder = torch.nn.Linear(1, 8)\n                self.blocks = torch.nn.ModuleList([DummySparseBlock(8) for _ in range(4)])\n                self.out_layer = DummySparseLin(8, 2)\n            def forward(self, x, t, cond, concat_cond=None):\n                xin = x.replace(torch.cat([x.feats, concat_cond.feats], dim=-1)) if concat_cond is not None else x\n                h = self.input_layer(xin)\n                te = self.t_embedder(t)\n                for b in self.blocks:\n                    h = b(h, te, cond)\n                h = h.replace(torch.nn.functional.layer_norm(h.feats, h.feats.shape[-1:]))\n                return self.out_layer(h)\n\n        sp_m = DummySparseFlow().eval()\n        coords = torch.tensor([[0, 0, 0, 0], [0, 1, 1, 1], [0, 2, 0, 1]], dtype=torch.int32, device=dev0)\n        xs = DummySparseTensor(torch.randn(3, 2, device=dev0), coords)\n        ccs = DummySparseTensor(torch.randn(3, 4, device=dev0), coords)\n        cond_s = Sequence(torch.randn(5, 8, device=dev0), [5])\n        with torch.no_grad():\n            expected_s = sp_m.to(dev0)(xs, td, cond_s, concat_cond=ccs)\n            sp_m.cpu()\n            sharded_s = DualGpuShardedFlowModel(sp_m, dev0=dev0, dev1=dev1).to(dev0)\n            actual_s = sharded_s(xs, td, cond_s, concat_cond=ccs)\n            torch.testing.assert_close(actual_s.feats, expected_s.feats, rtol=1e-5, atol=1e-5)\n\n    def test_sparse_convolution_matches_dense_with_holes_and_dilation(self):\n        coords = torch.tensor(list(itertools.product(range(2), range(4), range(3), range(5))), device=self.device)\n        coords = coords[torch.randperm(len(coords), device=self.device)[:67]]\n        features = torch.randn(len(coords), 3, device=self.device)\n        weight = torch.randn(4, 3, 3, 3, 3, device=self.device)\n        bias = torch.randn(4, device=self.device)\n        dense = torch.zeros(2, 3, 4, 3, 5, device=self.device)\n        dense[coords[:, 0], :, coords[:, 1], coords[:, 2], coords[:, 3]] = features\n        for dilation in (1, 2):\n            expected = F.conv3d(dense, weight, bias, padding=dilation, dilation=dilation)\n            expected = expected[coords[:, 0], :, coords[:, 1], coords[:, 2], coords[:, 3]]\n            actual, cache = submanifold_convolution(\n                features, coords, weight.permute(0, 2, 3, 4, 1).contiguous(), bias,\n                (dilation,) * 3, chunk_size=7,\n            )\n            torch.testing.assert_close(actual, expected, rtol=2e-5, atol=2e-5)\n            cached, _ = submanifold_convolution(\n                features, coords, weight.permute(0, 2, 3, 4, 1).contiguous(), bias,\n                (dilation,) * 3, neighbor_map=cache, chunk_size=11,\n            )\n            torch.testing.assert_close(actual, cached)\n\n    def test_attention_varlen_self_and_cross_match_independent_sequences(self):\n        query = torch.randn(8, 2, 4, device=self.device)\n        key = torch.randn(10, 2, 4, device=self.device)\n        value = torch.randn(10, 2, 6, device=self.device)\n        q = Sequence(query, [3, 5])\n        k = Sequence(key, [6, 4])\n        v = Sequence(value, [6, 4])\n        actual = sparse_attention(q=q, k=k, v=v)\n        for qs, ks in zip(q.layout, k.layout):\n            expected = F.scaled_dot_product_attention(\n                query[qs].transpose(0, 1)[None], key[ks].transpose(0, 1)[None], value[ks].transpose(0, 1)[None]\n            )[0].transpose(0, 1)\n            torch.testing.assert_close(actual.feats[qs], expected)\n        qkv = Sequence(torch.randn(8, 3, 2, 4, device=self.device), [3, 5])\n        self.assertEqual(sparse_attention(qkv).feats.shape, (8, 2, 4))\n\n    def test_attention_mixed_dense_sparse_and_query_chunking(self):\n        q = torch.randn(2, 270, 2, 4, device=self.device)\n        kv = Sequence(torch.randn(13, 2, 2, 4, device=self.device), [6, 7])\n        actual = sparse_attention(q, kv)\n        for index, key_slice in enumerate(kv.layout):\n            key, value = kv.feats[key_slice].unbind(1)\n            expected = F.scaled_dot_product_attention(\n                q[index].transpose(0, 1)[None], key.transpose(0, 1)[None], value.transpose(0, 1)[None]\n            )[0].transpose(0, 1)\n            torch.testing.assert_close(actual[index], expected)\n        sparse_q = Sequence(torch.randn(9, 2, 4, device=self.device), [4, 5])\n        dense_kv = torch.randn(2, 7, 2, 2, 4, device=self.device)\n        actual = sparse_attention(sparse_q, dense_kv)\n        for index, qs in enumerate(sparse_q.layout):\n            key, value = dense_kv[index].unbind(1)\n            expected = F.scaled_dot_product_attention(\n                sparse_q.feats[qs].transpose(0, 1)[None], key.transpose(0, 1)[None], value.transpose(0, 1)[None]\n            )[0].transpose(0, 1)\n            torch.testing.assert_close(actual.feats[qs], expected)\n\n    def test_voxel_centers_interpolation_missing_sites_and_outside(self):\n        coords = torch.tensor(list(itertools.product(range(2), range(2), range(2), range(2))), device=self.device)\n        features = (coords[:, 1:].float().sum(1) + 10 * coords[:, 0]).unsqueeze(1)\n        queries = torch.tensor([[[0.5, 0.5, 0.5], [1, 1, 1], [5, 5, 5]],\n                                [[0.5, 0.5, 0.5], [1, 1, 1], [5, 5, 5]]], device=self.device)\n        output = sparse_grid_sample(features, coords, (2, 1, 2, 2, 2), queries, chunk_size=2)\n        expected = torch.tensor([[[0], [1.5], [0]], [[10], [11.5], [0]]], dtype=torch.float32, device=self.device)\n        torch.testing.assert_close(output, expected)\n        output = sparse_grid_sample(features[:1] + 7, coords[:1], (1, 1, 2, 2, 2), queries[:1])\n        torch.testing.assert_close(output[0, :2], torch.full((2, 1), 7.0, device=self.device))\n\n    def test_nearest_sampling_truncates_negative_fractional_queries(self):\n        coords = torch.tensor([[0, 0, 0, 0]], device=self.device)\n        features = torch.tensor([[3.5, -1.25]], device=self.device)\n        queries = torch.tensor([[[-0.25, 0.25, 0.25], [-1.25, 0.25, 0.25], [0.75, 0.25, 0.25]]], device=self.device)\n        output = sparse_grid_sample(features, coords, (1, 2, 2, 2, 2), queries, mode=\'nearest\')\n        expected = torch.tensor([[[3.5, -1.25], [0.0, 0.0], [3.5, -1.25]]], device=self.device)\n        torch.testing.assert_close(output, expected)\n\n\nclass FullPlatePreparationTest(unittest.TestCase):\n    def setUp(self):\n        self.temp_dir = tempfile.TemporaryDirectory(dir=Path(__file__).resolve().parent)\n        self.dest = Path(self.temp_dir.name)\n\n    def tearDown(self):\n        self.temp_dir.cleanup()\n\n    def test_full_plate_padding_and_preservation(self):\n        source_path = self.dest / \'test_plate.png\'\n        image = Image.new(\'RGB\', (200, 150), color=(180, 140, 100))\n        image.save(source_path)\n\n        report = prepare_full_plate(source_path, self.dest, margin=0.1, max_size=512)\n        self.assertFalse(report[\'foreground_cropped\'])\n        self.assertFalse(report[\'automatic_background_removal\'])\n        self.assertTrue(report[\'existing_alpha_preserved\'])\n        self.assertEqual(report[\'original_size\'], [200, 150])\n        self.assertEqual(report[\'prepared_size\'], [240, 240])\n        self.assertTrue((self.dest / \'prepared_image.png\').is_file())\n        self.assertTrue((self.dest / \'full_plate_rgba.png\').is_file())\n        self.assertTrue((self.dest / \'image_preparation.json\').is_file())\n\n    def test_transparent_input_rejected(self):\n        source_path = self.dest / \'transparent.png\'\n        Image.new(\'RGBA\', (64, 64), (0, 0, 0, 0)).save(source_path)\n        with self.assertRaises(ValueError):\n            prepare_full_plate(source_path, self.dest)\n\n    def test_mask_dimension_mismatch(self):\n        source_path = self.dest / \'source.png\'\n        mask_path = self.dest / \'mask.png\'\n        Image.new(\'RGB\', (100, 100), (255, 255, 255)).save(source_path)\n        Image.new(\'L\', (50, 50), 255).save(mask_path)\n        with self.assertRaises(ValueError):\n            prepare_full_plate(source_path, self.dest, reviewed_mask=mask_path)\n\n\nclass SpaceToolsMeshContractTest(unittest.TestCase):\n    def setUp(self):\n        self.temp_dir = tempfile.TemporaryDirectory(dir=Path(__file__).resolve().parent)\n        self.dir = Path(self.temp_dir.name)\n\n    def tearDown(self):\n        self.temp_dir.cleanup()\n\n    def test_mesh_save_load_roundtrip(self):\n        vertices = torch.tensor([[0.0, 0.0, 0.0], [1.0, 0.0, 0.0], [0.0, 1.0, 0.0]], dtype=torch.float32)\n        faces = torch.tensor([[0, 1, 2]], dtype=torch.int32)\n        coords = torch.tensor([[0, 0, 0, 0], [0, 1, 1, 1]], dtype=torch.int32)\n        attrs = torch.tensor([[0.5, 0.2, 0.1, 0.8], [0.1, 0.9, 0.3, 0.4]], dtype=torch.float32)\n        origin = [-0.5, -0.5, -0.5]\n        voxel_size = 0.01\n        voxel_shape = torch.Size([64, 64, 64])\n        layout = {\'color\': slice(0, 3), \'roughness\': slice(3, 4)}\n\n        np.savez_compressed(\n            self.dir / \'raw_mesh_and_materials.npz\',\n            vertices=vertices.numpy(), faces=faces.numpy(),\n            coords=coords.numpy(), attributes=attrs.numpy(),\n            voxel_size=np.array(voxel_size), origin=np.array(origin),\n            voxel_shape=np.array(list(voxel_shape)),\n        )\n        layout_dict = {k: [v.start if v.start is not None else 0, v.stop] for k, v in layout.items()}\n        metadata = {\n            \'attribute_layout\': layout_dict,\n            \'voxel_size\': voxel_size,\n            \'voxel_shape\': list(voxel_shape),\n            \'origin\': origin,\n        }\n        (self.dir / \'raw_mesh_metadata.json\').write_text(json.dumps(metadata, indent=2), encoding=\'utf-8\')\n\n        with patch.dict(sys.modules, {\'trellis2.representations\': SimpleNamespace(MeshWithVoxel=MeshWithVoxel)}):\n            loaded_mesh = load_saved_mesh(self.dir, torch, device=torch.device(\'cpu\'))\n        torch.testing.assert_close(loaded_mesh.vertices, vertices)\n        torch.testing.assert_close(loaded_mesh.faces, faces)\n        torch.testing.assert_close(loaded_mesh.coords, coords)\n        torch.testing.assert_close(loaded_mesh.attrs, attrs)\n        self.assertEqual(loaded_mesh.origin.tolist(), origin)\n        self.assertEqual(loaded_mesh.voxel_size, voxel_size)\n        self.assertEqual(loaded_mesh.voxel_shape, voxel_shape)\n        self.assertEqual(loaded_mesh.layout, layout)\n\n    def test_write_gallery_html(self):\n        frames = {}\n        for key, _ in MODES:\n            frames[key] = []\n            for i in range(8):\n                p = self.dir / f\'{key}_{i:02d}.png\'\n                Image.new(\'RGB\', (16, 16), color=(i * 20, 100, 150)).save(p)\n                frames[key].append(p)\n        gallery_path = self.dir / \'all_hf_modes.html\'\n        result = write_gallery(frames, gallery_path)\n        self.assertTrue(result.is_file())\n        content = result.read_text(encoding=\'utf-8\')\n        for key, label in MODES:\n            self.assertIn(label, content)\n            self.assertIn(f\'data-mode="{key}"\', content)\n        for placeholder in (\'__MAX__\', \'__IMAGES__\', \'__BUTTONS__\', \'__COUNT__\', \'__DEFAULT__\'):\n            self.assertNotIn(placeholder, content)\n\n    def test_variant_parameter_validation(self):\n        mesh = MeshWithVoxel(\n            vertices=torch.tensor([[0.0, 0.0, 0.0], [1.0, 0.0, 0.0], [0.0, 1.0, 0.0]], dtype=torch.float32),\n            faces=torch.tensor([[0, 1, 2]], dtype=torch.int32),\n            origin=[0.0, 0.0, 0.0], voxel_size=0.01,\n            coords=torch.zeros((1, 4), dtype=torch.int32),\n            attrs=torch.zeros((1, 4), dtype=torch.float32),\n            voxel_shape=torch.Size([32, 32, 32]),\n        )\n        invalid_configs = [\n            {\'decimation_target\': 90000, \'texture_size\': 1024, \'remesh\': False},\n            {\'decimation_target\': 510000, \'texture_size\': 1024, \'remesh\': False},\n            {\'decimation_target\': 105000, \'texture_size\': 1024, \'remesh\': False},\n            {\'decimation_target\': 100000, \'texture_size\': 512, \'remesh\': False},\n            {\'decimation_target\': 100000, \'texture_size\': 2000, \'remesh\': False},\n        ]\n        for cfg in invalid_configs:\n            with self.subTest(cfg=cfg):\n                with self.assertRaises(ValueError):\n                    export_variant(mesh, cfg, self.dir)\n\n\nif __name__ == \'__main__\':\n    unittest.main(verbosity=2)\n', encoding='utf-8')
(SCRATCH_ROOT / 'test_trellis_full_plate.py').write_text('"""CPU checks for complete-plate input preservation and the six-mode gallery."""\n\nimport base64\nimport json\nfrom pathlib import Path\nimport re\nimport tempfile\nimport unittest\n\nimport numpy as np\nfrom PIL import Image, ImageDraw\n\nfrom trellis_prepare_full_plate import prepare_full_plate\nfrom trellis_native_space_tools import MODES, write_gallery\n\n\nclass FullPlateTest(unittest.TestCase):\n    def setUp(self):\n        self.temporary = tempfile.TemporaryDirectory(dir=Path(__file__).resolve().parent)\n        self.addCleanup(self.temporary.cleanup)\n        self.root = Path(self.temporary.name)\n\n    def test_opaque_image_retains_every_pixel_including_rim_at_all_four_edges(self):\n        image = Image.new(\'RGB\', (96, 64), \'#534132\')\n        draw = ImageDraw.Draw(image)\n        draw.ellipse((0, 0, 95, 63), fill=\'white\')\n        draw.ellipse((30, 20, 65, 43), fill=\'green\')\n        source = self.root / \'plate.png\'\n        image.save(source)\n        output = self.root / \'prepared\'\n        report = prepare_full_plate(source, output)\n        left, top = report[\'image_offset\']\n        with Image.open(output / \'prepared_image.png\') as prepared:\n            # Every original pixel survives, including background and the complete white rim.\n            np.testing.assert_array_equal(np.array(prepared)[top:top + 64, left:left + 96], np.array(image))\n            self.assertGreater(left, 0)\n            self.assertGreater(top, 0)\n        self.assertFalse(report[\'automatic_background_removal\'])\n        self.assertFalse(report[\'foreground_cropped\'])\n\n    def test_translucent_plate_rim_is_not_discarded_by_an_alpha_threshold(self):\n        image = Image.new(\'RGBA\', (70, 50), (220, 220, 255, 40))\n        ImageDraw.Draw(image).ellipse((20, 12, 49, 37), fill=(80, 150, 30, 255))\n        source = self.root / \'transparent_plate.png\'\n        image.save(source)\n        output = self.root / \'prepared\'\n        report = prepare_full_plate(source, output)\n        left, top = report[\'image_offset\']\n        with Image.open(output / \'full_plate_rgba.png\') as prepared:\n            np.testing.assert_array_equal(np.array(prepared)[top:top + 50, left:left + 70], np.array(image))\n\n    def test_large_image_scales_the_whole_frame_and_retains_all_corner_markers(self):\n        image = Image.new(\'RGB\', (1600, 1000), \'white\')\n        draw = ImageDraw.Draw(image)\n        markers = [((0, 0, 199, 199), \'red\'), ((1400, 0, 1599, 199), \'green\'),\n                   ((0, 800, 199, 999), \'blue\'), ((1400, 800, 1599, 999), \'yellow\')]\n        for box, color in markers:\n            draw.rectangle(box, fill=color)\n        source = self.root / \'large.png\'\n        image.save(source)\n        output = self.root / \'prepared\'\n        report = prepare_full_plate(source, output)\n        width, height = report[\'resized_image_size\']\n        left, top = report[\'image_offset\']\n        self.assertEqual(report[\'prepared_size\'], [1024, 1024])\n        self.assertAlmostEqual(width / height, 1.6, places=2)\n        with Image.open(output / \'prepared_image.png\') as prepared:\n            actual = np.array(prepared)\n            samples = [actual[top + 10, left + 10], actual[top + 10, left + width - 11],\n                       actual[top + height - 11, left + 10], actual[top + height - 11, left + width - 11]]\n        np.testing.assert_array_equal(samples, [[255, 0, 0], [0, 128, 0], [0, 0, 255], [255, 255, 0]])\n\n    def test_empty_and_mismatched_masks_are_rejected(self):\n        source = self.root / \'plate.png\'\n        Image.new(\'RGBA\', (80, 60), \'white\').save(source)\n        mask = self.root / \'mask.png\'\n        Image.new(\'L\', (20, 20), 255).save(mask)\n        with self.assertRaisesRegex(ValueError, \'dimensions\'):\n            prepare_full_plate(source, self.root / \'bad_size\', reviewed_mask=mask)\n        Image.new(\'L\', (80, 60), 0).save(mask)\n        with self.assertRaisesRegex(ValueError, \'fully transparent\'):\n            prepare_full_plate(source, self.root / \'empty\', reviewed_mask=mask)\n\n    def test_gallery_contains_all_six_modes_and_all_eight_views_offline(self):\n        frames = {}\n        for mode_index, (key, _) in enumerate(MODES):\n            frames[key] = []\n            for view in range(8):\n                path = self.root / f\'{key}_{view}.png\'\n                Image.new(\'RGB\', (8, 8), (mode_index * 40, view * 30, 128)).save(path)\n                frames[key].append(path)\n        document = write_gallery(frames, self.root / \'gallery.html\').read_text(encoding=\'utf-8\')\n        images = json.loads(re.search(r\'const frames=(.*);\', document).group(1))\n        self.assertEqual(set(images), {\'normal\', \'clay\', \'base_color\', \'shaded_forest\', \'shaded_sunset\', \'shaded_courtyard\'})\n        self.assertTrue(all(len(views) == 8 for views in images.values()))\n        self.assertEqual(document.count(\'data-mode="\'), 6)\n        self.assertNotIn(\'<script src=\', document)\n        for key, _ in MODES:\n            for view, encoded in enumerate(images[key]):\n                self.assertEqual(base64.b64decode(encoded.split(\',\', 1)[1]), frames[key][view].read_bytes())\n\n    def test_gallery_rejects_incomplete_mode_angles(self):\n        frames = {key: [self.root / \'unused.png\'] for key, _ in MODES}\n        frames[\'clay\'] = []\n        with self.assertRaisesRegex(ValueError, \'same nonzero\'):\n            write_gallery(frames, self.root / \'incomplete.html\')\n\n    def test_crop_to_alpha_centers_foreground_and_preserves_translucent_rim(self):\n        # 400x200 wide image with off-center translucent glass rim + opaque center food\n        image = Image.new(\'RGBA\', (400, 200), (90, 90, 90, 0))\n        draw = ImageDraw.Draw(image)\n        draw.ellipse((200, 40, 299, 139), fill=(210, 220, 240, 40))\n        draw.ellipse((230, 70, 269, 109), fill=(180, 70, 30, 255))\n        source = self.root / \'wide_dish.png\'\n        image.save(source)\n        output = self.root / \'cropped\'\n        report = prepare_full_plate(source, output, margin=0.1, max_size=512, crop_to_alpha=True, require_alpha=True)\n        self.assertTrue(report[\'foreground_cropped\'])\n        self.assertEqual(report[\'alpha_bbox\'], [200, 40, 300, 140])\n        self.assertEqual(report[\'prepared_size\'], [120, 120])\n        with Image.open(output / \'full_plate_rgba.png\') as rgba_out:\n            arr = np.array(rgba_out)\n            # Translucent rim pixel survives intact inside centered square; outer margin stays transparent\n            self.assertEqual(arr[15, 60, 3], 40)\n            self.assertEqual(arr[0, 0, 3], 0)\n        with Image.open(output / \'prepared_image.png\') as rgb_out:\n            # Outer transparent padding composited to pure black for TRELLIS.2 encoder\n            self.assertEqual(rgb_out.getpixel((0, 0)), (0, 0, 0))\n\n    def test_require_alpha_rejects_unmasked_opaque_image(self):\n        source = self.root / \'opaque_table.png\'\n        Image.new(\'RGB\', (120, 90), (80, 60, 40)).save(source)\n        with self.assertRaisesRegex(ValueError, \'no transparent background mask\'):\n            prepare_full_plate(source, self.root / \'reject_opaque\', crop_to_alpha=True, require_alpha=True)\n\n\nif __name__ == \'__main__\':\n    unittest.main(verbosity=2)\n', encoding='utf-8')
RUNTIME_ENV = BUILD_ENV.copy()
RUNTIME_ENV['PYTHONPATH'] = (
    (str(TRELLIS_SOURCE) + os.pathsep if TRELLIS_SOURCE.exists() else '') + str(SCRATCH_ROOT)
)
RUNTIME_ENV['ATTN_BACKEND'] = 'sdpa'
RUNTIME_ENV['SPARSE_CONV_BACKEND'] = 'none'
RUNTIME_ENV['TRELLIS_TEST_DEVICE'] = 'cuda:0' if TRELLIS_BUILD_OK else 'cpu'
run([PYTHON, SCRATCH_ROOT / 'test_trellis_full_plate.py'], env=RUNTIME_ENV)
run([PYTHON, SCRATCH_ROOT / 'test_trellis_native_compat.py'], env=RUNTIME_ENV)


In [ ]:
def _default_trellis_pipeline_type():
    import os as _os
    override = _os.environ.get('TRELLIS_PIPELINE_TYPE')
    if override:
        return override
    try:
        import torch as _t
        if _t.cuda.is_available() and _t.cuda.get_device_capability(0)[0] >= 8:
            return '1536_cascade'
    except Exception:
        pass
    return '512'


if 'RUNS_3D' not in globals():
    RUNS_3D = {}

SEED = 42
MARGIN = 0.0
MAX_SIZE = 1024
SETTINGS = {
    'seed': SEED,
    'pipeline_type': _default_trellis_pipeline_type(),
    'auto_bf16': True,
    'max_num_tokens': 49152,
    'sampling_steps': 12,
    'samplers': {
        'sparse_structure': {'steps': 12, 'guidance_strength': 7.5, 'guidance_rescale': 0.7, 'rescale_t': 5.0},
        'shape': {'steps': 12, 'guidance_strength': 7.5, 'guidance_rescale': 0.5, 'rescale_t': 3.0},
        'material': {'steps': 12, 'guidance_strength': 1.0, 'guidance_rescale': 0.0, 'rescale_t': 3.0},
    },
    'texture_size': 2048,
    'decimation_target': 300000,
    'remesh': True,
    'remesh_band': 1,
    'remesh_project': 0.0,
    'mobile_decimation_target': 34000,
    'mobile_texture_size': 1024,
    'crop_to_alpha': True,
    'require_alpha': True,
    'use_dual_gpu_sharding': True,
    'save_latents': False,
}

import io as _io
import re as _re
import signal as _signal
import sys as _sys
import threading as _threading
from tempfile import mkdtemp

TRELLIS_WORKERS = {}


def _sanitize_trellis_line(line):
    return _re.sub(r'hf_[A-Za-z0-9]{6,}', '[REDACTED_TOKEN]', line)


def _run_logged_bounded(cmd, env, log_stream, timeout, label):
    process = subprocess.Popen(
        [str(v) for v in cmd],
        env=env,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
        start_new_session=True,
    )
    tail = []
    reader = None
    stream = getattr(process, 'stdout', None)
    if isinstance(stream, _io.IOBase):
        def _pump():
            try:
                for raw_line in stream:
                    log_stream.write(raw_line)
                    log_stream.flush()
                    cleaned = _sanitize_trellis_line(raw_line.rstrip())
                    if cleaned:
                        tail.append(cleaned)
                        if len(tail) > 10:
                            del tail[:-10]
                        if cleaned.startswith(('[trellis:stage]', '[trellis]', 'Loading official checkpoint:')):
                            print(cleaned, file=_sys.stdout, flush=True)
            finally:
                try:
                    stream.close()
                except OSError:
                    pass
        reader = _threading.Thread(target=_pump, daemon=True)
        reader.start()
    try:
        rc = process.wait(timeout=timeout)
    except subprocess.TimeoutExpired:
        try:
            os.killpg(process.pid, _signal.SIGKILL)
        except OSError:
            process.kill()
        try:
            process.wait(timeout=10)
        except subprocess.SubprocessError:
            pass
        if reader is not None:
            reader.join(timeout=2)
        raise DemoError(f'{label} timed out after {timeout}s.') from None
    if reader is not None:
        reader.join(timeout=2)
    if rc and tail:
        for tline in tail[-10:]:
            print(f'[trellis:tail] {tline}', file=_sys.stdout, flush=True)
    return rc


def trellis_generation(folder, source_image, runtime, checkpoint_revisions):
    folder.mkdir(parents=True, exist_ok=True)
    prepare = [
        PYTHON, SCRATCH_ROOT / 'trellis_prepare_full_plate.py', source_image,
        folder, '--margin', str(MARGIN), '--max-size', str(MAX_SIZE),
    ]
    if SETTINGS.get('crop_to_alpha', True):
        prepare.append('--crop-to-alpha')
    if SETTINGS.get('require_alpha', True):
        prepare.append('--require-alpha')
    with (folder / 'native_generation.log').open('w', encoding='utf-8') as log:
        prep_rc = _run_logged_bounded(prepare, RUNTIME_ENV, log, timeout=120, label='Whole-plate preparation')
        if prep_rc:
            raise DemoError(f'Whole-plate preparation exited with code {prep_rc}.')
        config = {
            **SETTINGS,
            'source_dir': str(TRELLIS_SOURCE),
            'output_dir': str(folder),
            'prepared_image': str(folder / 'prepared_image.png'),
            'model_revision': checkpoint_revisions.get(TRELLIS_MODEL_REPO, TRELLIS_MODEL_REVISION),
            'checkpoint_revisions': dict(checkpoint_revisions),
        }
        config_path = folder / 'generation_config.json'
        config_path.write_text(json.dumps(config, indent=2), encoding='utf-8')
        run_rc = _run_logged_bounded(
            [PYTHON, SCRATCH_ROOT / 'trellis_native_runner.py', config_path],
            runtime,
            log,
            timeout=1200,
            label='TRELLIS runner',
        )
    status_path = folder / 'run_status.json'
    status = json.loads(status_path.read_text(encoding='utf-8')) if status_path.is_file() else {}
    safe_keys = (
        'status', 'error_type', 'generation_seconds', 'export_seconds', 'elapsed_seconds',
        'peak_gpu0_gib', 'peak_gpu1_gib', 'total_peak_vram_gib', 'encoder_gpu', 'dual_gpu_sharded', 'dtype_used',
        'pipeline_type', 'actual_pipeline_type', 'fallback_reason',
        'remesh_used', 'texture_size_used', 'export_fallback_reason', 'mobile_status',
        'vertices', 'faces', 'mobile_vertices', 'mobile_faces',
    )
    status = {k: v for k, v in status.items() if k in safe_keys}
    status_path.write_text(json.dumps(status, indent=2), encoding='utf-8')
    if run_rc or status.get('status') != 'success':
        raise DemoError(f"TRELLIS exited with code {run_rc} ({status.get('error_type', 'unknown error')}).")
    validate_glb(folder / 'generated_original.glb')
    return status


checkpoint_revisions = {
    DINOV3_REPO: DINOV3_REVISION,
    TRELLIS_MODEL_REPO: TRELLIS_MODEL_REVISION,
}
if 'trellis2-4b' in ENABLED_MODELS:
    print('[trellis2-4b] Running on active slice...')
for label in (ACTIVE_LABELS if 'trellis2-4b' in ENABLED_MODELS else []):
    in_entry = image_results[label]
    key = f'{label}/trellis2-4b'
    rec = {
        'input_label': label,
        'image_source': in_entry['image_source'],
        'image_model': in_entry['model'],
        'backend': 'TRELLIS.2-4B',
        'appearance': 'pbr_texture',
        'artifact_paths': [],
    }
    if not (DINOV3_OK and TRELLIS_BUILD_OK):
        skip_reason = TRELLIS_SKIP_REASON if (DINOV3_OK and not TRELLIS_BUILD_OK and globals().get('TRELLIS_SKIP_REASON')) else DINOV3_NOTE
        rec.update(status='skipped', mobile_status='not_run', error=f'Skipped ({skip_reason})')
        RUNS_3D[key] = rec
        continue
    cm = in_entry.get('cutout_metrics') or {}
    is_plate_preserved = bool(cm.get('plate_preserved', False))
    plate_method = cm.get('plate_method', 'food_only_fallback' if not is_plate_preserved else 'unknown')
    rec['plate_preserved'] = is_plate_preserved
    rec['plate_method'] = plate_method
    if not is_plate_preserved:
        print(f'[trellis2-4b] {label}: plate not preserved ({plate_method}); falling back to food-only cutout for generation.', flush=True)
    runtime = RUNTIME_ENV.copy()
    if HF_TOKEN:
        runtime['HF_TOKEN'] = HF_TOKEN
    try:
        cutout_input = in_entry.get('cutout_path')
        if not cutout_input or not Path(cutout_input).is_file():
            raise DemoError(f'Missing cutout_path for {label}.')
        attempts_dir = SCRATCH_ROOT / 'trellis_runs' / label
        attempts_dir.mkdir(parents=True, exist_ok=True)
        work_folder = Path(mkdtemp(prefix='attempt-', dir=attempts_dir))
        stats = trellis_generation(work_folder, cutout_input, runtime, checkpoint_revisions)
        run_revs = json.loads((work_folder / 'model_revision.json').read_text(encoding='utf-8')).get('revisions', {})
        checkpoint_revisions.update(run_revs)
        REVISIONS['trellis_checkpoints'] = dict(checkpoint_revisions)
        out_glb = RESULTS_DIR / label / 'trellis2_4b.glb'
        shutil.copyfile(work_folder / 'generated_original.glb', out_glb)
        glb_meta = validate_glb(out_glb)
        artifacts = [out_glb]
        collect_mobile_glb(
            work_folder / 'generated_mobile.glb',
            RESULTS_DIR / label / 'trellis2_4b_mobile.glb',
            stats,
            rec,
        )
        if rec['mobile_status'] == 'available':
            artifacts.append(rec['mobile_glb_path'])
        rec.update(
            status='success',
            appearance='pbr_texture',
            plate_preserved=is_plate_preserved,
            plate_method=plate_method,
            device='cuda:0+cuda:1' if stats.get('dual_gpu_sharded') else 'cuda:0',
            checkpoint_revisions=dict(checkpoint_revisions),
            metrics={**stats, **glb_meta},
            artifact_paths=artifacts,
        )
        TRELLIS_WORKERS[label] = work_folder
        display(FileLink(str(out_glb)))
    except Exception as exc:
        rec.update(status='failed', mobile_status='not_run', error=str(exc))
        print(key, 'failed:', exc)
    RUNS_3D[key] = rec


In [ ]:
if DINOV3_OK and TRELLIS_BUILD_OK and NVDIFFREC_AVAILABLE and TRELLIS_WORKERS:
    for label, work_folder in TRELLIS_WORKERS.items():
        render_dir = RESULTS_DIR / label / 'trellis_hf_modes'
        space_cfg = work_folder / 'space_render_config.json'
        space_cfg.write_text(json.dumps({
            'source_dir': str(TRELLIS_SOURCE),
            'output_dir': str(work_folder),
            'result_dir': str(render_dir),
            'preview_resolution': 512,
            'preview_ssaa': 1,
        }, indent=2), encoding='utf-8')
        try:
            render_env = RUNTIME_ENV.copy()
            render_env['CUDA_VISIBLE_DEVICES'] = '1' if GPU_COUNT > 1 else '0'
            run([PYTHON, SCRATCH_ROOT / 'trellis_native_space_tools.py', 'render', space_cfg], env=render_env)
            montage = render_dir / 'all_modes_comparison.jpg'
            if montage.is_file():
                display(DisplayImage(filename=str(montage)))
        except Exception as err:
            print('Optional TRELLIS Space render skipped:', type(err).__name__)
else:
    print('TRELLIS HF 6-mode render skipped (requires DINOV3_OK, NVDIFFREC_AVAILABLE, and a TRELLIS GLB).')


## 7. Multi-Model Studio HTML & Archive Export


In [ ]:
import html as html_lib
import math
from PIL import ImageDraw
import pymeshlab
import trimesh


def make_checkerboard(size, tile=16, c1=(28, 34, 44), c2=(42, 49, 62)):
    w, h = size
    ys, xs = np.indices((h, w))
    pat = ((xs // tile) + (ys // tile)) % 2 == 0
    board = np.empty((h, w, 3), dtype=np.uint8)
    board[pat] = c1
    board[~pat] = c2
    return Image.fromarray(board, mode='RGB')


def _load_decimated_mesh_data(glb_path, target_faces=10000):
    scene = trimesh.load(str(glb_path), force='scene', process=False)
    meshes = [g for g in scene.geometry.values() if isinstance(g, trimesh.Trimesh) and len(g.faces) > 0]
    if not meshes:
        raise ValueError(f'No triangle mesh found in {glb_path}')
    combined = trimesh.util.concatenate(meshes) if len(meshes) > 1 else meshes[0].copy()
    verts = np.asarray(combined.vertices, dtype=np.float32)
    faces = np.asarray(combined.faces, dtype=np.int32)
    has_vc = combined.visual.kind == 'vertex' and hasattr(combined.visual, 'vertex_colors')
    face_rgb = None
    if has_vc:
        v_col = np.asarray(combined.visual.vertex_colors, dtype=np.float32)[:, :3]
        v_srgb = np.power(np.clip(v_col / 255.0, 0.0, 1.0), 1.0 / 2.2) * 255.0
        face_rgb = v_srgb[faces].mean(axis=1)

    if len(faces) > target_faces:
        try:
            ms = pymeshlab.MeshSet()
            if has_vc:
                vc = np.asarray(combined.visual.vertex_colors, dtype=np.float64) / 255.0
                if vc.shape[1] == 3:
                    vc = np.concatenate([vc, np.ones((len(vc), 1))], axis=1)
                pm = pymeshlab.Mesh(
                    vertex_matrix=verts.astype(np.float64),
                    face_matrix=faces.astype(np.int32),
                    v_color_matrix=vc,
                )
            else:
                pm = pymeshlab.Mesh(
                    vertex_matrix=verts.astype(np.float64),
                    face_matrix=faces.astype(np.int32),
                )
            ms.add_mesh(pm)
            ms.apply_filter('meshing_decimation_quadric_edge_collapse', targetfacenum=target_faces, preserveboundary=True)
            cm = ms.current_mesh()
            verts = cm.vertex_matrix().astype(np.float32)
            faces = cm.face_matrix().astype(np.int32)
            if has_vc and cm.has_vertex_color():
                v_lin = np.clip(cm.vertex_color_matrix()[:, :3], 0.0, 1.0).astype(np.float32)
                v_srgb = np.power(v_lin, 1.0 / 2.2) * 255.0
                face_rgb = v_srgb[faces].mean(axis=1)
            else:
                face_rgb = None
        except Exception:
            if len(faces) > 40000:
                step = max(1, len(faces) // 40000)
                faces = faces[::step]
                if face_rgb is not None:
                    face_rgb = face_rgb[::step]

    if face_rgb is None:
        face_rgb = np.full((len(faces), 3), [214.0, 200.0, 184.0], dtype=np.float32)

    vmin, vmax = verts.min(axis=0), verts.max(axis=0)
    center = 0.5 * (vmin + vmax)
    scale = max(float(np.linalg.norm(vmax - vmin)) * 0.5, 1e-5)
    verts = (verts - center) / scale
    return verts, faces, face_rgb


def _project_triangles(verts, faces, face_rgb, yaw_deg, pitch_deg):
    yaw, pitch = math.radians(yaw_deg), math.radians(pitch_deg)
    ry = np.array([[math.cos(yaw), 0, math.sin(yaw)], [0, 1, 0], [-math.sin(yaw), 0, math.cos(yaw)]], dtype=np.float32)
    rx = np.array([[1, 0, 0], [0, math.cos(pitch), -math.sin(pitch)], [0, math.sin(pitch), math.cos(pitch)]], dtype=np.float32)
    rv = (verts @ ry.T) @ rx.T
    tri = rv[faces]
    e1 = tri[:, 1] - tri[:, 0]
    e2 = tri[:, 2] - tri[:, 0]
    norms = np.cross(e1, e2)
    nlen = np.linalg.norm(norms, axis=1, keepdims=True)
    valid = nlen[:, 0] > 1e-10
    tri, frgb, norms = tri[valid], face_rgb[valid], norms[valid] / nlen[valid]
    z_order = np.argsort(tri[:, :, 2].mean(axis=1))
    return tri[z_order], frgb[z_order], norms[z_order]


def render_glb_three_views(glb_path, out_png, tile_size=260):
    verts, faces, face_rgb = _load_decimated_mesh_data(glb_path)
    angles = [('Front 3/4', 35, 28), ('Side +90', 125, 22), ('Top 3/4', 35, 68)]
    strip = Image.new('RGB', (tile_size * 3, tile_size + 26), (26, 30, 38))
    s_draw = ImageDraw.Draw(strip)
    key_light = np.array([0.35, 0.78, 0.52], dtype=np.float32)
    key_light /= np.linalg.norm(key_light)
    fill_light = np.array([-0.55, 0.35, 0.75], dtype=np.float32)
    fill_light /= np.linalg.norm(fill_light)

    for idx, (title, yaw_deg, pitch_deg) in enumerate(angles):
        tri, frgb, norms = _project_triangles(verts, faces, face_rgb, yaw_deg, pitch_deg)
        d_key = np.clip((norms * key_light).sum(axis=1), 0.0, 1.0)
        d_fill = np.clip((norms * fill_light).sum(axis=1), 0.0, 1.0)
        shade = np.clip(0.52 + 0.38 * d_key + 0.16 * d_fill, 0.52, 1.06)
        shaded_rgb = np.clip(frgb * shade[:, None], 0, 255).astype(np.uint8)

        scale_px = tile_size * 0.41
        px = tri[:, :, 0] * scale_px + tile_size * 0.5
        py = -tri[:, :, 1] * scale_px + tile_size * 0.52
        tile = Image.new('RGB', (tile_size, tile_size), (22, 26, 34))
        draw = ImageDraw.Draw(tile)
        for pxs, pys, col in zip(px.tolist(), py.tolist(), shaded_rgb.tolist()):
            draw.polygon([(pxs[0], pys[0]), (pxs[1], pys[1]), (pxs[2], pys[2])], fill=tuple(col))
        strip.paste(tile, (idx * tile_size, 26))
        s_draw.text((idx * tile_size + 10, 6), title, fill=(220, 226, 236))
    strip.save(out_png)
    return out_png


def render_hf_trellis_modes(glb_path, out_png, tile_size=220):
    """Render HF TRELLIS Space-inspired 4-mode studio strip: Normal, Clay, Base Color, Studio HDRI."""
    verts, faces, face_rgb = _load_decimated_mesh_data(glb_path)
    tri, frgb, norms = _project_triangles(verts, faces, face_rgb, yaw_deg=30, pitch_deg=32)

    scale_px = tile_size * 0.41
    px = tri[:, :, 0] * scale_px + tile_size * 0.5
    py = -tri[:, :, 1] * scale_px + tile_size * 0.52

    key_l = np.array([0.40, 0.76, 0.51], dtype=np.float32)
    key_l /= np.linalg.norm(key_l)
    fill_l = np.array([-0.60, 0.30, 0.74], dtype=np.float32)
    fill_l /= np.linalg.norm(fill_l)
    rim_l = np.array([0.0, 0.45, -0.89], dtype=np.float32)
    d_key = np.clip((norms * key_l).sum(axis=1), 0.0, 1.0)
    d_fill = np.clip((norms * fill_l).sum(axis=1), 0.0, 1.0)
    d_rim = np.clip((norms * rim_l).sum(axis=1), 0.0, 1.0)

    normal_rgb = np.clip((norms * 0.5 + 0.5) * 255.0, 0, 255).astype(np.uint8)
    clay_shade = np.clip(0.42 + 0.44 * d_key + 0.16 * d_fill + 0.14 * (d_rim ** 2), 0.42, 1.08)
    clay_rgb = np.clip(np.array([218.0, 206.0, 192.0], dtype=np.float32) * clay_shade[:, None], 0, 255).astype(np.uint8)
    base_rgb = np.clip(frgb, 0, 255).astype(np.uint8)
    hdri_tint = (
        0.54
        + 0.40 * d_key[:, None] * np.array([1.08, 0.99, 0.90], dtype=np.float32)
        + 0.18 * d_fill[:, None] * np.array([0.88, 0.96, 1.12], dtype=np.float32)
        + 0.12 * (d_rim[:, None] ** 2)
    )
    hdri_rgb = np.clip(frgb * np.clip(hdri_tint, 0.50, 1.15), 0, 255).astype(np.uint8)

    modes = [
        ('Normal', normal_rgb),
        ('Clay', clay_rgb),
        ('Base Color', base_rgb),
        ('Studio HDRI', hdri_rgb),
    ]
    strip = Image.new('RGB', (tile_size * len(modes), tile_size + 26), (20, 24, 32))
    s_draw = ImageDraw.Draw(strip)
    coords = [[(p0, q0), (p1, q1), (p2, q2)] for (p0, p1, p2), (q0, q1, q2) in zip(px.tolist(), py.tolist())]
    for m_idx, (title, colors) in enumerate(modes):
        tile = Image.new('RGB', (tile_size, tile_size), (18, 22, 30))
        t_draw = ImageDraw.Draw(tile)
        for poly, col in zip(coords, colors.tolist()):
            t_draw.polygon(poly, fill=tuple(col))
        strip.paste(tile, (m_idx * tile_size, 26))
        s_draw.text((m_idx * tile_size + 10, 6), title, fill=(224, 232, 242))
    strip.save(out_png)
    return out_png


extra_artifacts = []
for key, rec in RUNS_3D.items():
    if rec.get('status') == 'success' and rec.get('artifact_paths'):
        glb_path = Path(rec['artifact_paths'][0])
        preview_png = glb_path.with_name(glb_path.stem + '_preview.png')
        hf_modes_png = glb_path.with_name(glb_path.stem + '_hf_modes.png')
        try:
            render_glb_three_views(glb_path, preview_png)
            rec['artifact_paths'].append(preview_png)
            rec['preview_path'] = preview_png
            render_hf_trellis_modes(glb_path, hf_modes_png)
            rec['artifact_paths'].append(hf_modes_png)
            rec['hf_modes_path'] = hf_modes_png
        except Exception as err:
            print(f'Preview render failed for {key}:', type(err).__name__, err)

# Assemble comparison_sheet.png with Dish Input + Cutout (Plate + Food) + 3D backends
backends_order = [
    ('moge2-relief', 'MoGe-2 Relief + Plate'),
    ('trellis2-4b', 'TRELLIS.2-4B (PBR)'),
    ('triposg', 'TripoSG + Photo PBR'),
    ('hunyuan3d2', 'Hunyuan3D-2 + PBR'),
    ('hunyuan3d21', 'Hunyuan3D-2.1 + PBR'),
    ('hi3dgen', 'Hi3DGen Normal-Bridge'),
]
in_w = 240
cutout_w = 240
col_w = 540
row_h = 260
left_w = in_w + cutout_w
sheet_w = left_w + len(backends_order) * col_w
sheet_h = 56 + len(ACTIVE_LABELS) * row_h
sheet = Image.new('RGB', (sheet_w, sheet_h), (16, 20, 27))
draw = ImageDraw.Draw(sheet)
draw.text((16, 18), 'Dish Input (Source)', fill=(240, 244, 250))
draw.text((in_w + 16, 18), 'Cutout (Plate + Food)', fill=(240, 244, 250))
for c_idx, (_, title) in enumerate(backends_order):
    draw.text((left_w + c_idx * col_w + 14, 18), title, fill=(240, 244, 250))

for r_idx, label in enumerate(ACTIVE_LABELS):
    y0 = 56 + r_idx * row_h
    in_entry = image_results[label]
    draw.rectangle([8, y0 + 4, sheet_w - 8, y0 + row_h - 6], outline=(55, 65, 82), width=1)
    with Image.open(in_entry['image_path']) as im:
        contained = ImageOps.contain(im.convert('RGB'), (204, 200), Image.Resampling.LANCZOS)
        in_tile = Image.new('RGB', (204, 200), (22, 26, 34))
        in_tile.paste(contained, ((204 - contained.width) // 2, (200 - contained.height) // 2))
        sheet.paste(in_tile, (18, y0 + 38))
    src_tag = 'user' if in_entry['image_source'] == 'user-uploaded-dataset' else in_entry['image_source']
    draw.text((18, y0 + 12), f"{label} ({src_tag})", fill=(170, 210, 255))

    cut_path = in_entry.get('cutout_path')
    cm = in_entry.get('cutout_metrics', {})
    if cut_path and Path(cut_path).is_file():
        with Image.open(cut_path) as cim:
            contained_rgba = ImageOps.contain(cim.convert('RGBA'), (204, 200), Image.Resampling.LANCZOS)
            board = make_checkerboard((204, 200)).convert('RGBA')
            px_off = ((204 - contained_rgba.width) // 2, (200 - contained_rgba.height) // 2)
            board.alpha_composite(contained_rgba, px_off)
            sheet.paste(board.convert('RGB'), (in_w + 18, y0 + 38))
        pl_flag = 'plate=ON' if cm.get('plate_preserved') else 'plate=foodOnly'
        cut_cap = f"{pl_flag} (+{cm.get('plate_added_pct', 0)}%) {cm.get('fg_coverage_pct', '?')}%"
        draw.text((in_w + 18, y0 + 12), cut_cap[:34], fill=(170, 235, 205))

    for c_idx, (b_key, _) in enumerate(backends_order):
        x0 = left_w + c_idx * col_w
        rec = RUNS_3D.get(f'{label}/{b_key}', {})
        st = rec.get('status', 'not_run')
        if st == 'success' and rec.get('preview_path') and Path(rec['preview_path']).is_file():
            with Image.open(rec['preview_path']) as pv:
                pv_res = pv.resize((col_w - 20, 184), Image.Resampling.LANCZOS)
                sheet.paste(pv_res, (x0 + 10, y0 + 38))
            m = rec['metrics']
            mob_m = rec.get('mobile_metrics', {})
            v_num = m.get('vertices', m.get('vertex_count', 0))
            f_num = m.get('faces')
            f_txt = f'{f_num:,}' if isinstance(f_num, int) else '?'
            mob_status = mobile_delivery_status(rec)
            mob_txt = f" | mob={mob_m['faces']:,}" if isinstance(mob_m.get('faces'), int) else f" | mob={mob_status}"
            cap = f"v={v_num:,} f={f_txt}{mob_txt} | {m.get('elapsed_seconds', '?')}s | {m.get('bytes', 0)//1024}KB"
            draw.text((x0 + 12, y0 + 12), cap[:72], fill=(175, 240, 185))
        else:
            msg = rec.get('error', st)[:62]
            draw.text((x0 + 14, y0 + 105), f"{st.upper()}: {msg}", fill=(245, 185, 130))

sheet_path = RESULTS_DIR / 'comparison_sheet.png'
sheet.save(sheet_path)
extra_artifacts.append(sheet_path)
display(DisplayImage(filename=str(sheet_path)))

# Assemble hf_modes_sheet.png (Normal / Clay / Base Color / Studio HDRI per dish & backend)
hf_backends = [('moge2-relief', 'MoGe-2 Relief'), ('trellis2-4b', 'TRELLIS.2-4B'), ('triposg', 'TripoSG'), ('hunyuan3d2', 'Hunyuan3D-2'), ('hunyuan3d21', 'Hunyuan3D-2.1'), ('hi3dgen', 'Hi3DGen')]
hf_label_w = 220
hf_col_w = 660
hf_row_h = 216
hf_w = hf_label_w + len(hf_backends) * hf_col_w
hf_h = 56 + len(ACTIVE_LABELS) * hf_row_h
hf_sheet = Image.new('RGB', (hf_w, hf_h), (14, 18, 25))
hf_draw = ImageDraw.Draw(hf_sheet)
hf_draw.text((16, 18), 'Dish + Preserved Cutout', fill=(240, 244, 250))
for c_idx, (_, title) in enumerate(hf_backends):
    hf_draw.text((hf_label_w + c_idx * hf_col_w + 14, 18), title, fill=(240, 244, 250))

for r_idx, label in enumerate(ACTIVE_LABELS):
    y0 = 56 + r_idx * hf_row_h
    hf_draw.rectangle([8, y0 + 4, hf_w - 8, y0 + hf_row_h - 6], outline=(55, 65, 82), width=1)
    cut_path = image_results[label].get('cutout_path')
    hf_draw.text((16, y0 + 10), label, fill=(170, 210, 255))
    if cut_path and Path(cut_path).is_file():
        with Image.open(cut_path) as cim:
            contained_rgba = ImageOps.contain(cim.convert('RGBA'), (184, 160), Image.Resampling.LANCZOS)
            board = make_checkerboard((184, 160)).convert('RGBA')
            board.alpha_composite(contained_rgba, ((184 - contained_rgba.width) // 2, (160 - contained_rgba.height) // 2))
            hf_sheet.paste(board.convert('RGB'), (16, y0 + 36))
    for c_idx, (b_key, _) in enumerate(hf_backends):
        x0 = hf_label_w + c_idx * hf_col_w
        rec = RUNS_3D.get(f'{label}/{b_key}', {})
        hm = rec.get('hf_modes_path')
        if rec.get('status') == 'success' and hm and Path(hm).is_file():
            with Image.open(hm) as hpv:
                hf_sheet.paste(hpv.resize((hf_col_w - 20, 176), Image.Resampling.LANCZOS), (x0 + 10, y0 + 28))

hf_sheet_path = RESULTS_DIR / 'hf_modes_sheet.png'
hf_sheet.save(hf_sheet_path)
extra_artifacts.append(hf_sheet_path)
display(DisplayImage(filename=str(hf_sheet_path)))

# Write per-dish worker_output.json receipts + Docker blueprint artifacts
for label in ACTIVE_LABELS:
    jm_path = Path(image_results[label]['job_manifest_path'])
    jm = json.loads(jm_path.read_text(encoding='utf-8'))
    dish_runs = {k: v for k, v in RUNS_3D.items() if v.get('input_label') == label}
    receipt = build_worker_receipt(jm, image_results[label], dish_runs, RESULTS_DIR)
    receipt_path = RESULTS_DIR / label / 'worker_output.json'
    receipt_path.write_text(json.dumps(receipt, indent=2) + '\n', encoding='utf-8')
    extra_artifacts.append(receipt_path)

blueprint_dir = RESULTS_DIR / 'docker_blueprint'
blueprint_dir.mkdir(parents=True, exist_ok=True)
pinned_path = blueprint_dir / 'pinned_revisions.json'
pinned_path.write_text(json.dumps({
    'schema_version': 1,
    'demo_label': DEMO_TRUTHFUL_LABEL,
    'publication_eligible': False,
    'plate_preservation_mode': PLATE_PRESERVATION_MODE,
    'models': {k: v for k, v in REVISIONS.items() if not k.startswith(('input_', 'cutout_'))},
}, indent=2) + '\n', encoding='utf-8')
extra_artifacts.append(pinned_path)

lock_lines = [
    '# Pinned Python dependencies captured from the verified Kaggle CUDA runtime',
    *[f'{pkg}=={ver}' for pkg, ver in sorted(RUNTIME_PACKAGE_VERSIONS.items())],
]
lock_path = blueprint_dir / 'requirements.lock.txt'
lock_path.write_text('\n'.join(lock_lines) + '\n', encoding='utf-8')
extra_artifacts.append(lock_path)

worker_entrypoint_template = r"""# Docker worker entrypoint implementing the immutable job_manifest -> worker_output contract.

import argparse
import hashlib
import json
import os
from pathlib import Path


def sha256_file(path):
    with Path(path).open('rb') as stream:
        return hashlib.file_digest(stream, 'sha256').hexdigest()


def _valid_sha256(value):
    return isinstance(value, str) and len(value) == 64 and all(c in '0123456789abcdef' for c in value)


def verify_job_manifest(manifest_path, input_dir):
    manifest = json.loads(Path(manifest_path).read_text(encoding='utf-8'))
    if manifest.get('publication_eligible') is not False:
        raise ValueError('Single-photo worker jobs must retain publication_eligible=false.')
    for key in ('source_sha256', 'input_image_sha256'):
        if key == 'input_image_sha256' and key not in manifest:
            continue
        if not _valid_sha256(manifest.get(key)):
            raise ValueError(f'job_manifest.json must contain a valid 64-char lowercase hex {key}.')
    src_file = Path(input_dir) / Path(manifest['input_filename']).name
    if not src_file.is_file():
        raise FileNotFoundError(f'Missing input file: {src_file}')
    actual_sha = sha256_file(src_file)
    expected_sha = manifest.get('input_image_sha256') or manifest['source_sha256']
    if actual_sha != expected_sha:
        raise ValueError(f'Staged input SHA-256 mismatch: expected {expected_sha}, got {actual_sha}')
    if manifest.get('input_byte_identical') and actual_sha != manifest['source_sha256']:
        raise ValueError(
            'job_manifest.json declares byte-identical staging, but input_image_sha256 '
            f"{actual_sha} differs from source_sha256 {manifest['source_sha256']}"
        )
    return manifest, src_file, actual_sha


def run_worker(manifest_path, input_dir, out_dir, dry_run=False):
    manifest, src_file, actual_sha = verify_job_manifest(manifest_path, input_dir)
    out = Path(out_dir)
    out.mkdir(parents=True, exist_ok=True)
    receipt = {
        'schema_version': 1,
        'job_id': manifest['job_id'],
        'dish_label': manifest['dish_label'],
        'demo_label': manifest.get('demo_label'),
        'publication_eligible': False,
        'source_sha256': manifest.get('source_sha256'),
        'input_image_sha256': actual_sha,
        'input_transformation': manifest.get('input_transformation', 'none'),
        'source_file': src_file.name,
        'plate_preservation_mode': manifest.get('plate_preservation_mode', 'preserve_plate'),
        'status': 'validated_dry_run' if dry_run else 'ready_for_gpu_backends',
        'export_profiles': manifest.get('export_profiles', {}),
        'pinned_revisions': manifest.get('pinned_revisions', {}),
    }
    receipt_path = out / 'worker_output.json'
    receipt_path.write_text(json.dumps(receipt, indent=2) + '\n', encoding='utf-8')
    return receipt_path


def main():
    parser = argparse.ArgumentParser(description='PeekaPlate Docker 3D dish demo worker')
    parser.add_argument('--manifest', required=True, help='Path to immutable job_manifest.json')
    parser.add_argument('--input-dir', required=True, help='Directory holding the verified dish photo')
    parser.add_argument('--out', required=True, help='Output artifact directory')
    parser.add_argument('--dry-run', action='store_true', help='Verify contract & hashes without running GPU models')
    args = parser.parse_args()
    os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True')
    out_receipt = run_worker(args.manifest, args.input_dir, args.out, dry_run=args.dry_run)
    print(f'Wrote verified receipt: {out_receipt}')


if __name__ == '__main__':
    main()
"""
entrypoint_path = blueprint_dir / 'worker_entrypoint.py'
entrypoint_path.write_text(worker_entrypoint_template, encoding='utf-8')
extra_artifacts.append(entrypoint_path)

_cuda_raw = (getattr(torch.version, 'cuda', None) or '12.8').strip()
_cuda_base = '.'.join(_cuda_raw.split('.')[:2]) if '.' in _cuda_raw else _cuda_raw
_cuda_tag = 'cu' + _cuda_base.replace('.', '')[:3]
dockerfile_template = f"""# PeekaPlate Dish-to-3D Demo Worker Template (CUDA {_cuda_base}, private ticket 86bd5m6q0 demo)
# Build context must provide requirements.lock.txt, pinned_revisions.json, and worker_entrypoint.py.
# Never bake HF_TOKEN, OPENROUTER_API_KEY, or customer photos into image layers.

FROM nvidia/cuda:{_cuda_base}.0-runtime-ubuntu24.04

ENV DEBIAN_FRONTEND=noninteractive \\
    PYTHONUNBUFFERED=1 \\
    PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True \\
    HF_HOME=/var/cache/peekaplate/hf \\
    TORCH_HOME=/var/cache/peekaplate/torch \\
    U2NET_HOME=/var/cache/peekaplate/u2net

RUN apt-get update && apt-get install -y --no-install-recommends \\
    python3 python3-pip git libgl1 libglib2.0-0 libgomp1 \\
    && rm -rf /var/lib/apt/lists/*

RUN useradd -m -u 10001 worker \\
    && mkdir -p /app /var/cache/peekaplate /work \\
    && chown -R worker:worker /app /var/cache/peekaplate /work

WORKDIR /app
COPY requirements.lock.txt /app/requirements.lock.txt
RUN pip3 install --no-cache-dir --break-system-packages \\
    --extra-index-url https://download.pytorch.org/whl/{_cuda_tag} \\
    -r /app/requirements.lock.txt

COPY pinned_revisions.json /app/pinned_revisions.json
COPY worker_entrypoint.py /app/worker_entrypoint.py

USER worker
VOLUME ["/var/cache/peekaplate", "/work"]
ENTRYPOINT ["python3", "/app/worker_entrypoint.py"]
CMD ["--help"]
"""
dockerfile_path = blueprint_dir / 'Dockerfile.worker.template'
dockerfile_path.write_text(dockerfile_template, encoding='utf-8')
extra_artifacts.append(dockerfile_path)

# Interactive Three.js 0.182.0 WebGL 3D Dish Studios (HTTP multi-file + self-contained standalone HTML)
STUDIO_THREE_VERSION = '0.182.0'
STUDIO_THREE_CDN_BASE = 'https://cdn.jsdelivr.net/npm/three@' + STUDIO_THREE_VERSION
STUDIO_HTML_TEMPLATE = """<!doctype html>
<html lang="en">
<head>
  <meta charset="utf-8" />
  <meta name="viewport" content="width=device-width, initial-scale=1" />
  <title>__TITLE__</title>
  <script type="importmap">__IMPORTMAP_JSON__</script>
  <style>
    :root {
      color-scheme: dark;
      --bg: #0b0f17;
      --panel: #131a26;
      --panel-2: #192232;
      --border: #2a374d;
      --text: #eef3fa;
      --muted: #93a6c2;
      --accent: #3b82f6;
      --accent-soft: rgba(59, 130, 246, 0.16);
      --ok: #34d399;
      --warn: #f59e0b;
    }
    * { box-sizing: border-box; }
    body {
      margin: 0;
      font: 14px/1.45 ui-sans-serif, system-ui, -apple-system, Segoe UI, Roboto, sans-serif;
      background: var(--bg);
      color: var(--text);
      min-height: 760px;
      display: flex;
      flex-direction: column;
    }
    header {
      padding: 12px 18px;
      border-bottom: 1px solid var(--border);
      background: #0f1520;
      display: flex;
      flex-wrap: wrap;
      align-items: center;
      justify-content: space-between;
      gap: 10px;
    }
    .brand { display: flex; flex-direction: column; gap: 2px; }
    .brand h1 { margin: 0; font-size: 16px; font-weight: 700; letter-spacing: 0.01em; }
    .sub { font-size: 12px; color: var(--muted); }
    .badges { display: flex; flex-wrap: wrap; gap: 8px; align-items: center; }
    .badge {
      padding: 4px 10px;
      border-radius: 999px;
      font-size: 12px;
      font-weight: 600;
      border: 1px solid #3c5478;
      background: #1a273b;
      color: #bfdbfe;
    }
    .layout {
      flex: 1;
      display: grid;
      grid-template-columns: 270px 1fr;
      min-height: 0;
    }
    @media (max-width: 960px) {
      .layout { grid-template-columns: 1fr; }
    }
    aside {
      border-right: 1px solid var(--border);
      background: #0f151f;
      padding: 14px;
      display: flex;
      flex-direction: column;
      gap: 12px;
      overflow-y: auto;
    }
    .sec-title {
      font-size: 11px;
      font-weight: 700;
      text-transform: uppercase;
      letter-spacing: 0.08em;
      color: var(--muted);
      margin-bottom: 4px;
    }
    .dish-list { display: flex; flex-direction: column; gap: 8px; }
    .dish-btn {
      display: flex;
      align-items: center;
      gap: 10px;
      width: 100%;
      text-align: left;
      padding: 8px;
      border-radius: 10px;
      border: 1px solid var(--border);
      background: var(--panel);
      color: var(--text);
      cursor: pointer;
    }
    .dish-btn.active {
      border-color: var(--accent);
      background: var(--accent-soft);
    }
    .dish-btn img {
      width: 52px;
      height: 52px;
      border-radius: 6px;
      object-fit: cover;
      background: repeating-conic-gradient(#1f2733 0% 25%, #2b3545 0% 50%) 50% / 12px 12px;
      flex-shrink: 0;
    }
    .dish-meta { display: flex; flex-direction: column; min-width: 0; }
    .dish-name { font-weight: 600; white-space: nowrap; overflow: hidden; text-overflow: ellipsis; }
    .dish-sub { font-size: 11px; color: var(--muted); }
    .dish-pill { font-size: 11px; color: var(--ok); }
    .ref-card {
      background: var(--panel);
      border: 1px solid var(--border);
      border-radius: 10px;
      padding: 10px;
      display: flex;
      flex-direction: column;
      gap: 8px;
    }
    .ref-pair { display: grid; grid-template-columns: 1fr 1fr; gap: 8px; }
    .ref-box { display: flex; flex-direction: column; gap: 4px; font-size: 11px; color: var(--muted); }
    .ref-box img {
      width: 100%;
      height: 100px;
      object-fit: contain;
      border-radius: 6px;
      background: #0a0e14;
    }
    .ref-box img.checker {
      background: repeating-conic-gradient(#1e2633 0% 25%, #2b3648 0% 50%) 50% / 14px 14px;
    }
    .kv { font-size: 12px; color: var(--muted); display: flex; flex-direction: column; gap: 3px; }
    .kv b { color: var(--text); font-weight: 600; }

    main {
      display: flex;
      flex-direction: column;
      min-width: 0;
      padding: 14px;
      gap: 12px;
    }
    .toolbar {
      display: flex;
      flex-wrap: wrap;
      gap: 10px;
      align-items: center;
      padding: 10px 12px;
      background: var(--panel);
      border: 1px solid var(--border);
      border-radius: 10px;
    }
    .grp { display: flex; flex-wrap: wrap; align-items: center; gap: 4px; padding-right: 8px; border-right: 1px solid var(--border); }
    .grp:last-child { border-right: none; padding-right: 0; }
    .grp-lbl { font-size: 11px; color: var(--muted); margin-right: 4px; }
    button.ctrl, a.ctrl {
      background: var(--panel-2);
      color: var(--text);
      border: 1px solid var(--border);
      border-radius: 6px;
      padding: 5px 10px;
      font-size: 12px;
      font-weight: 500;
      cursor: pointer;
      text-decoration: none;
    }
    button.ctrl.active { background: var(--accent); border-color: #60a5fa; color: #fff; }
    button.ctrl:disabled { opacity: 0.45; cursor: not-allowed; }
    label.slider { display: flex; align-items: center; gap: 6px; font-size: 12px; color: var(--muted); }

    .stage-wrap {
      flex: 1;
      display: grid;
      grid-template-columns: 1fr;
      gap: 12px;
      min-height: 480px;
    }
    .stage-wrap.compare { grid-template-columns: 1fr 1fr; }
    @media (max-width: 960px) {
      .stage-wrap.compare { grid-template-columns: 1fr; }
    }
    .vp-card {
      position: relative;
      display: flex;
      flex-direction: column;
      border-radius: 12px;
      border: 1px solid var(--border);
      background: #080b10;
      overflow: hidden;
      min-height: 460px;
    }
    .vp-card.hidden { display: none; }
    .vp-head {
      display: flex;
      justify-content: space-between;
      align-items: center;
      padding: 8px 12px;
      background: rgba(19, 26, 38, 0.92);
      border-bottom: 1px solid var(--border);
      font-size: 12px;
      gap: 8px;
      flex-wrap: wrap;
    }
    .vp-title { font-weight: 700; }
    .vp-stats { color: var(--muted); }
    .vp-canvas-host {
      position: relative;
      flex: 1;
      width: 100%;
      min-height: 400px;
    }
    .vp-canvas-host.no-webgl { display: none; }
    .vp-canvas-host canvas { display: block; width: 100%; height: 100%; outline: none; }
    .vp-overlay {
      position: absolute;
      inset: 12px;
      pointer-events: none;
      display: flex;
      justify-content: space-between;
      align-items: flex-end;
      font-size: 11px;
      color: var(--muted);
    }
    .vp-pill {
      background: rgba(11, 15, 23, 0.78);
      border: 1px solid var(--border);
      padding: 4px 8px;
      border-radius: 6px;
    }
    .fallback-box {
      display: none;
      padding: 16px;
      gap: 10px;
      flex-direction: column;
      align-items: flex-start;
    }
    .fallback-box.active { display: flex; }
    .fallback-box img { max-width: 100%; border-radius: 8px; border: 1px solid var(--border); }
    .fallback-msg { font-size: 12px; color: var(--muted); }
    .notice {
      padding: 10px 12px;
      border-radius: 8px;
      border: 1px solid #6b4e1b;
      background: rgba(245, 158, 11, 0.1);
      color: #fde68a;
      font-size: 12px;
    }
  </style>
</head>
<body>
  <header>
    <div class="brand">
      <h1>PeekaPlate Interactive 3D Dish Studio</h1>
      <div class="sub">__MODE_TAG__ · Drag to orbit · Scroll to zoom · Right-drag to pan</div>
    </div>
    <div class="badges">
      <span class="badge" id="demo-badge">__DEMO_BADGE__</span>
    </div>
  </header>

  <div class="layout">
    <aside>
      <div>
        <div class="sec-title">Dish Samples</div>
        <input id="dish-search" type="search" placeholder="Search dishes" aria-label="Search dishes" />
        <div id="tier-filters"><button data-tier="">All</button><button data-tier="benchmark_4">Benchmark</button><button data-tier="hero">Hero</button><button data-tier="page12">Menu P12</button></div>
        <div class="dish-list" id="dish-list"></div>
      </div>
      <div class="ref-card">
        <div class="sec-title">Plate + Food Reference</div>
        <div class="ref-pair">
          <div class="ref-box"><span>Source Photo</span><img id="ref-source" alt="Source photo" /></div>
          <div class="ref-box"><span>Cutout (Plate + Food)</span><img id="ref-cutout" class="checker" alt="Plate cutout" /></div>
        </div>
        <div class="kv" id="dish-telemetry"></div>
      </div>
    </aside>

    <main>
      <div class="toolbar">
        <div class="grp" id="grp-backend">
          <span class="grp-lbl">View</span>
          <button class="ctrl active" data-view="trellis2-4b">TRELLIS.2-4B (PBR)</button>
        </div>
        <div class="grp" id="grp-profile">
          <span class="grp-lbl">Mesh</span>
          <button class="ctrl active" data-profile="mobile">TRELLIS Fast (&lt;=35k)</button>
          <button class="ctrl" data-profile="raw">TRELLIS Full (Raw)</button>
        </div>
        <div class="grp" id="grp-shader">
          <span class="grp-lbl">Shading</span>
          <button class="ctrl active" data-shader="pbr">Studio PBR</button>
          <button class="ctrl" data-shader="base_color">Base Color</button>
          <button class="ctrl" data-shader="clay">Clay</button>
          <button class="ctrl" data-shader="normal">Normal</button>
          <button class="ctrl" id="btn-wireframe">Wireframe</button>
        </div>
        <div class="grp" id="grp-camera">
          <span class="grp-lbl">Camera</span>
          <button class="ctrl" data-preset="front">Front 3/4</button>
          <button class="ctrl" data-preset="top">Top</button>
          <button class="ctrl" data-preset="side">Side</button>
          <button class="ctrl" data-preset="low">Low Rim</button>
          <button class="ctrl" id="btn-rotate">Auto-Rotate: OFF</button>
          <button class="ctrl" id="btn-grid">Grid: ON</button>
        </div>
        <div class="grp" id="grp-env">
          <span class="grp-lbl">Bg</span>
          <button class="ctrl active" data-bg="dark">Dark</button>
          <button class="ctrl" data-bg="neutral">Slate</button>
          <button class="ctrl" data-bg="light">Light</button>
          <label class="slider">Exposure <input id="rng-exposure" type="range" min="0.5" max="2.0" step="0.05" value="1.10" /></label>
          <a class="ctrl" id="btn-download" download="dish.glb" href="#">Download GLB</a>
        </div>
      </div>

      <div id="webgl-notice" class="notice" style="display:none"></div>

      <div id="compare-controls" hidden><label>Left <select id="cmp-left"></select></label> <label>Right <select id="cmp-right"></select></label></div>
      <div class="stage-wrap" id="stage-wrap">
        <div class="vp-card" id="vp-primary">
          <div class="vp-head">
            <span class="vp-title" id="vp-primary-title">Loading…</span>
            <span class="vp-stats" id="vp-primary-stats"></span>
          </div>
          <div class="vp-canvas-host" id="vp-primary-host">
            <div class="vp-overlay">
              <span class="vp-pill" id="vp-primary-status">Initializing WebGL…</span>
              <span class="vp-pill" id="vp-primary-hint">3D Orbit Active</span>
            </div>
          </div>
          <div class="fallback-box" id="vp-primary-fallback"></div>
        </div>

        <div class="vp-card hidden" id="vp-secondary">
          <div class="vp-head">
            <span class="vp-title" id="vp-secondary-title">Secondary View</span>
            <span class="vp-stats" id="vp-secondary-stats"></span>
          </div>
          <div class="vp-canvas-host" id="vp-secondary-host">
            <div class="vp-overlay">
              <span class="vp-pill" id="vp-secondary-status">Synced Camera</span>
              <span class="vp-pill">Geometry Compare</span>
            </div>
          </div>
          <div class="fallback-box" id="vp-secondary-fallback"></div>
        </div>
      </div>
      <section id="model-matrix" aria-label="Per-Dish Model Benchmark Matrix"></section>
    </main>
  </div>

  <script id="studio-data" type="application/json">__STUDIO_DATA_JSON__</script>
  <script type="module">
    const DATA = JSON.parse(document.getElementById('studio-data').textContent);
    const BG_HEX = { dark: 0x080b10, neutral: 0x263142, light: 0xe6ebf2 };
    const TEX_SLOTS = ['map', 'normalMap', 'roughnessMap', 'metalnessMap', 'aoMap', 'emissiveMap', 'alphaMap', 'bumpMap', 'displacementMap', 'specularMap', 'envMap'];

    const state = {
      dishIdx: 0,
      view: 'trellis2-4b',
      profile: 'mobile',
      shader: 'pbr',
      wireframe: false,
      autoRotate: false,
      grid: true,
      bg: 'dark',
      exposure: 1.1,
      webglReady: false,
    };

    const dishListEl = document.getElementById('dish-list');
    const refSourceEl = document.getElementById('ref-source');
    const refCutoutEl = document.getElementById('ref-cutout');
    const telemetryEl = document.getElementById('dish-telemetry');
    const noticeEl = document.getElementById('webgl-notice');
    const stageWrap = document.getElementById('stage-wrap');
    const btnDownload = document.getElementById('btn-download');

    let THREE, OrbitControls, GLTFLoader, RoomEnvironment;
    let vpA = { hostEl: document.getElementById('vp-primary-host'), titleEl: document.getElementById('vp-primary-title'), statsEl: document.getElementById('vp-primary-stats'), statusEl: document.getElementById('vp-primary-status'), fallbackEl: document.getElementById('vp-primary-fallback'), originalMeshes: [], loadToken: 0 };
    let vpB = { hostEl: document.getElementById('vp-secondary-host'), titleEl: document.getElementById('vp-secondary-title'), statsEl: document.getElementById('vp-secondary-stats'), statusEl: document.getElementById('vp-secondary-status'), fallbackEl: document.getElementById('vp-secondary-fallback'), originalMeshes: [], loadToken: 0 };

    const initThreeAsync = async () => {
      try {
        [THREE, { OrbitControls }, { GLTFLoader }, { RoomEnvironment }] = await Promise.all([
          import('three'),
          import('three/addons/controls/OrbitControls.js'),
          import('three/addons/loaders/GLTFLoader.js'),
          import('three/addons/environments/RoomEnvironment.js'),
        ]);
        const testCanvas = document.createElement('canvas');
        if (!testCanvas.getContext('webgl2') && !testCanvas.getContext('webgl')) {
          throw new Error('WebGL context is unavailable in this browser.');
        }
        state.webglReady = true;
      } catch (err) {
        noticeEl.style.display = 'block';
        noticeEl.textContent = 'WebGL 3D renderer unavailable (' + (err?.message || err) + '). Showing static plate-preserving renders and GLB downloads.';
        state.webglReady = false;
      }
      vpA = createViewport(
        document.getElementById('vp-primary-host'),
        document.getElementById('vp-primary-title'),
        document.getElementById('vp-primary-stats'),
        document.getElementById('vp-primary-status'),
        document.getElementById('vp-primary-fallback'),
      );
      vpB = createViewport(
        document.getElementById('vp-secondary-host'),
        document.getElementById('vp-secondary-title'),
        document.getElementById('vp-secondary-stats'),
        document.getElementById('vp-secondary-status'),
        document.getElementById('vp-secondary-fallback'),
      );
      linkCameras(vpA, vpB);
      linkCameras(vpB, vpA);
      if (state.webglReady) animate();
      refreshStage();
    };

    function base64ToArrayBuffer(b64) {
      const bin = atob(b64);
      const bytes = new Uint8Array(bin.length);
      for (let i = 0; i < bin.length; i++) bytes[i] = bin.charCodeAt(i);
      return bytes.buffer;
    }

    function disposeMaterial(mat, disposeTextures = true) {
      if (!mat) return;
      const mats = Array.isArray(mat) ? mat : [mat];
      for (const m of mats) {
        if (!m) continue;
        if (disposeTextures) {
          for (const slot of TEX_SLOTS) {
            const tex = m[slot];
            if (tex && typeof tex.dispose === 'function') {
              tex.source?.data?.close?.();
              tex.dispose();
            }
          }
        }
        if (typeof m.dispose === 'function') m.dispose();
      }
    }

    function disposeObject3D(root, disposeTextures = true) {
      if (!root) return;
      root.traverse((obj) => {
        if (obj.geometry) obj.geometry.dispose();
        if (obj.material) disposeMaterial(obj.material, disposeTextures);
      });
    }

    function clearViewportModel(vp) {
      if (vp.scene && vp.root) vp.scene.remove(vp.root);
      for (const entry of vp.originalMeshes) {
        if (entry.origMat && entry.origMat !== entry.mesh?.material) {
          disposeMaterial(entry.origMat, true);
        }
      }
      if (vp.root) disposeObject3D(vp.root, true);
      vp.root = null;
      vp.originalMeshes = [];
    }

    function showFallback(vp, run, dish, message) {
      vp.hostEl.classList.add('no-webgl');
      vp.fallbackEl.innerHTML = '';
      vp.fallbackEl.classList.add('active');
      if (message) {
        const p = document.createElement('div');
        p.className = 'fallback-msg';
        p.textContent = message;
        vp.fallbackEl.appendChild(p);
      }
      const imgSrc = (run && run.fallback_image) || dish?.cutout_image || dish?.source_image || '';
      if (imgSrc) {
        const img = document.createElement('img');
        img.src = imgSrc;
        img.alt = (run ? run.backend : dish?.label || 'Dish') + ' static preview';
        vp.fallbackEl.appendChild(img);
      }
    }

    function createViewport(hostEl, titleEl, statsEl, statusEl, fallbackEl) {
      const vp = {
        hostEl, titleEl, statsEl, statusEl, fallbackEl,
        renderer: null, scene: null, camera: null, controls: null,
        gridHelper: null, root: null, originalMeshes: [],
        lastRun: null, loadToken: 0,
      };
      if (!state.webglReady) {
        hostEl.classList.add('no-webgl');
        return vp;
      }

      const renderer = new THREE.WebGLRenderer({ antialias: true, powerPreference: 'high-performance' });
      renderer.setPixelRatio(Math.min(window.devicePixelRatio || 1, 2));
      renderer.outputColorSpace = THREE.SRGBColorSpace;
      renderer.toneMapping = THREE.ACESFilmicToneMapping;
      renderer.toneMappingExposure = state.exposure;
      renderer.shadowMap.enabled = true;
      renderer.shadowMap.type = THREE.PCFSoftShadowMap;
      hostEl.insertBefore(renderer.domElement, hostEl.firstChild);

      renderer.domElement.addEventListener('webglcontextlost', (ev) => {
        ev.preventDefault();
        state.webglReady = false;
        clearViewportModel(vp);
        noticeEl.style.display = 'block';
        noticeEl.textContent = 'WebGL context lost. Showing static plate-preserving renders and GLB downloads.';
        showFallback(vp, vp.lastRun, DATA.dishes[state.dishIdx], 'WebGL context lost');
      });

      const scene = new THREE.Scene();
      scene.background = new THREE.Color(BG_HEX[state.bg]);
      const pmrem = new THREE.PMREMGenerator(renderer);
      const envScene = new RoomEnvironment();
      const envTex = pmrem.fromScene(envScene, 0.04).texture;
      scene.environment = envTex;
      envScene.dispose();
      pmrem.dispose();

      const hemi = new THREE.HemisphereLight(0xf8fafc, 0x1e293b, 0.45);
      const key = new THREE.DirectionalLight(0xfff5e8, 1.85);
      key.position.set(2.5, 4.0, 2.2);
      key.castShadow = true;
      const rim = new THREE.DirectionalLight(0x9ec5fe, 0.75);
      rim.position.set(-2.6, 2.2, -2.5);
      scene.add(hemi, key, rim);

      const grid = new THREE.GridHelper(3.2, 16, 0x3b4d6b, 0x1e293b);
      grid.visible = state.grid;
      scene.add(grid);

      const camera = new THREE.PerspectiveCamera(38, 1, 0.05, 50);
      camera.position.set(1.85, 1.35, 2.05);

      const controls = new OrbitControls(camera, renderer.domElement);
      controls.enableDamping = true;
      controls.dampingFactor = 0.07;
      controls.target.set(0, 0.28, 0);
      controls.minDistance = 0.7;
      controls.maxDistance = 7.0;
      controls.update();

      vp.renderer = renderer;
      vp.scene = scene;
      vp.camera = camera;
      vp.controls = controls;
      vp.gridHelper = grid;

      const resize = () => {
        const w = Math.max(hostEl.clientWidth, 100);
        const h = Math.max(hostEl.clientHeight, 360);
        renderer.setSize(w, h, false);
        camera.aspect = w / h;
        camera.updateProjectionMatrix();
      };
      new ResizeObserver(resize).observe(hostEl);
      resize();
      return vp;
    }

    let syncingCam = false;
    function linkCameras(src, dst) {
      if (!src.controls || !dst.controls) return;
      src.controls.addEventListener('change', () => {
        if (syncingCam || state.view !== 'compare') return;
        syncingCam = true;
        dst.camera.position.copy(src.camera.position);
        dst.controls.target.copy(src.controls.target);
        dst.controls.update();
        syncingCam = false;
      });
    }

    function applyShaderToViewport(vp) {
      if (!vp.root) return;
      for (const entry of vp.originalMeshes) {
        const { mesh, origMat, hasVC } = entry;
        const baseOrig = Array.isArray(origMat) ? origMat[0] : origMat;
        const baseMap = baseOrig?.map || null;
        let nextMat;
        if (state.shader === 'base_color') {
          nextMat = new THREE.MeshBasicMaterial({
            map: hasVC ? null : baseMap,
            vertexColors: hasVC,
            color: (hasVC || baseMap) ? 0xffffff : 0xd8cdbf,
            side: THREE.DoubleSide,
            wireframe: state.wireframe,
          });
        } else if (state.shader === 'clay') {
          nextMat = new THREE.MeshStandardMaterial({
            color: 0xdfd3c3,
            roughness: 0.62,
            metalness: 0.04,
            vertexColors: false,
            side: THREE.DoubleSide,
            wireframe: state.wireframe,
          });
        } else if (state.shader === 'normal') {
          nextMat = new THREE.MeshNormalMaterial({
            side: THREE.DoubleSide,
            wireframe: state.wireframe,
          });
        } else {
          if (Array.isArray(origMat)) {
            nextMat = origMat.map((m) => {
              const c = m.clone();
              c.side = THREE.DoubleSide;
              c.wireframe = state.wireframe;
              return c;
            });
          } else if (origMat) {
            nextMat = origMat.clone();
            nextMat.side = THREE.DoubleSide;
            nextMat.wireframe = state.wireframe;
            if (!hasVC && !nextMat.map && nextMat.color && origMat.color.getHex() === 0xffffff) {
              nextMat.color.setHex(0xd9cec1);
            }
          } else {
            nextMat = new THREE.MeshStandardMaterial({
              color: 0xd9cec1,
              roughness: 0.55,
              metalness: 0.05,
              vertexColors: hasVC,
              side: THREE.DoubleSide,
              wireframe: state.wireframe,
            });
          }
        }
        const prev = mesh.material;
        mesh.material = nextMat;
        if (prev && prev !== origMat) {
          disposeMaterial(prev, false);
        }
      }
    }

    function pickProfile(run) {
      if (!run) return { prof: null, label: 'None', isRaw: false };
      const canRaw = Boolean(run.raw && (run.raw.path || run.raw.base64));
      const canMob = Boolean(run.mobile && (run.mobile.path || run.mobile.base64));
      if ((state.profile === 'raw' && canRaw) || !canMob) {
        return { prof: run.raw, label: 'TRELLIS Full (Raw)', isRaw: true };
      }
      return { prof: run.mobile, label: 'TRELLIS Fast (<=35k)', isRaw: false };
    }

    async function loadRunIntoViewport(vp, run, dish) {
      const token = ++vp.loadToken;
      vp.lastRun = run;
      vp.fallbackEl.classList.remove('active');
      vp.fallbackEl.innerHTML = '';
      if (!run) {
        clearViewportModel(vp);
        vp.titleEl.textContent = 'Unavailable backend';
        vp.statsEl.textContent = '';
        vp.statusEl.textContent = 'Skipped / Not generated';
        showFallback(vp, null, dish, 'Backend did not produce a GLB for this dish; showing plate+food reference.');
        return;
      }

      const { prof, label: profLabel } = pickProfile(run);
      vp.titleEl.textContent = run.backend + ' (' + profLabel + ')';
      vp.statsEl.textContent = (prof?.vertices ?? '?') + 'v / ' + (prof?.faces ?? '?') + 'f · ' + Math.round((prof?.bytes || 0) / 1024) + ' KB · ' + run.appearance;

      if (!state.webglReady || !prof || (!prof.path && !prof.base64)) {
        clearViewportModel(vp);
        const reason = !state.webglReady ? 'Static fallback (WebGL unavailable)' : 'No GLB payload available';
        vp.statusEl.textContent = reason;
        showFallback(vp, run, dish, reason);
        return;
      }

      vp.hostEl.classList.remove('no-webgl');
      vp.statusEl.textContent = 'Loading ' + (prof.filename || 'mesh') + '…';
      try {
        const loader = new GLTFLoader();
        const gltf = prof.base64
          ? await new Promise((resolve, reject) => loader.parse(base64ToArrayBuffer(prof.base64), '', resolve, reject))
          : await loader.loadAsync(prof.path);
        if (token !== vp.loadToken) {
          disposeObject3D(gltf.scene, true);
          return;
        }
        clearViewportModel(vp);

        const root = gltf.scene;
        const box = new THREE.Box3().setFromObject(root);
        const sz = box.getSize(new THREE.Vector3());
        const scale = 1.8 / Math.max(sz.x, sz.y, sz.z, 1e-4);
        root.scale.setScalar(scale);
        const sBox = new THREE.Box3().setFromObject(root);
        const center = sBox.getCenter(new THREE.Vector3());
        root.position.set(-center.x, -sBox.min.y, -center.z);

        const meshes = [];
        root.traverse((obj) => {
          if (obj.isMesh && obj.geometry) {
            if (!obj.geometry.attributes.normal) obj.geometry.computeVertexNormals();
            obj.castShadow = true;
            obj.receiveShadow = true;
            meshes.push({
              mesh: obj,
              origMat: obj.material,
              hasVC: Boolean(obj.geometry.attributes.color),
            });
          }
        });
        vp.root = root;
        vp.originalMeshes = meshes;
        vp.scene.add(root);
        applyShaderToViewport(vp);

        const midY = (sBox.max.y - sBox.min.y) * 0.42;
        vp.controls.target.set(0, midY, 0);
        vp.controls.update();
        vp.statusEl.textContent = 'Loaded (' + (prof.faces || '?') + ' faces)';
      } catch (err) {
        clearViewportModel(vp);
        const errMsg = 'Load failed: ' + (err?.message || err);
        vp.statusEl.textContent = errMsg;
        showFallback(vp, run, dish, errMsg);
      }
    }

    function setPreset(preset) {
      const presets = {
        front: [1.85, 1.25, 2.05],
        top: [0.01, 2.85, 0.35],
        side: [2.75, 0.85, 0.01],
        low: [1.95, 0.42, 1.95],
      };
      const pos = presets[preset] || presets.front;
      for (const vp of [vpA, vpB]) {
        if (!vp.camera || !vp.controls) continue;
        vp.camera.position.set(pos[0], pos[1], pos[2]);
        vp.controls.update();
      }
    }

    function updateDownloadLink(dish, primaryRun) {
      const { prof } = pickProfile(primaryRun);
      if (!dish || !primaryRun || !prof || (!prof.path && !prof.base64)) {
        btnDownload.style.display = 'none';
        return;
      }
      btnDownload.style.display = 'inline-block';
      btnDownload.download = dish.label + '_' + (prof.filename || 'dish.glb');
      if (prof.base64) {
        btnDownload.onclick = (ev) => {
          ev.preventDefault();
          const blob = new Blob([base64ToArrayBuffer(prof.base64)], { type: 'model/gltf-binary' });
          const url = URL.createObjectURL(blob);
          const a = document.createElement('a');
          a.href = url;
          a.download = btnDownload.download;
          document.body.appendChild(a);
          a.click();
          a.remove();
          setTimeout(() => URL.revokeObjectURL(url), 1000);
        };
        btnDownload.href = '#';
      } else {
        btnDownload.onclick = null;
        btnDownload.href = prof.path || '#';
      }
    }

    let tierFilter = '';
    document.getElementById('dish-search').oninput = renderDishSidebar;
    document.getElementById('tier-filters').onclick = (event) => {
      if (event.target.dataset.tier !== undefined) { tierFilter = event.target.dataset.tier; renderDishSidebar(); }
    };
    function renderDishSidebar() {
      dishListEl.innerHTML = '';
      DATA.dishes.forEach((dish, idx) => {
        const query = document.getElementById('dish-search').value.toLowerCase();
        if (!(dish.title || dish.label).toLowerCase().includes(query) || (tierFilter && dish.tier !== tierFilter && String(dish.page) !== tierFilter)) return;
        const cm = dish.cutout_metrics || {};
        const btn = document.createElement('button');
        btn.className = 'dish-btn' + (idx === state.dishIdx ? ' active' : '');
        btn.dataset.dish = dish.label;
        const img = document.createElement('img');
        img.src = dish.cutout_image || dish.source_image || '';
        img.alt = dish.label;
        const meta = document.createElement('div');
        meta.className = 'dish-meta';
        const nm = document.createElement('span');
        nm.className = 'dish-name';
        nm.textContent = dish.title || dish.label;
        const sb = document.createElement('span');
        sb.className = 'dish-sub';
        sb.textContent = (cm.plate_method || 'cutout') + ' · fg ' + (cm.fg_coverage_pct ?? '?') + '%';
        const pl = document.createElement('span');
        pl.className = 'dish-pill';
        pl.textContent = cm.plate_preserved ? ('Plate +' + cm.plate_added_pct + '%') : 'Food only';
        meta.append(nm, sb, pl);
        btn.append(img, meta);
        btn.onclick = () => { state.dishIdx = idx; refreshStage(); };
        dishListEl.appendChild(btn);
      });
    }

    async function refreshStage() {
      renderDishSidebar();
      const dish = DATA.dishes[state.dishIdx];
      if (!dish) return;
      const cm = dish.cutout_metrics || {};
      refSourceEl.src = dish.source_image || '';
      refCutoutEl.src = dish.cutout_image || '';
      telemetryEl.innerHTML = '';
      const lines = [
        ['Dish', dish.label + ' (' + (dish.size ? dish.size.join('×') : '?') + ')'],
        ['Plate Mode', (cm.plate_preserved ? 'PRESERVED' : 'food-only') + ' (+' + (cm.plate_added_pct ?? 0) + '%)'],
        ['Method', cm.plate_method || '?'],
        ['Source SHA', dish.source_sha256 ? dish.source_sha256.slice(0, 12) + '…' : 'n/a'],
      ];
      for (const [k, v] of lines) {
        const div = document.createElement('div');
        const b = document.createElement('b');
        b.textContent = k + ': ';
        div.appendChild(b);
        div.appendChild(document.createTextNode(String(v)));
        telemetryEl.appendChild(div);
      }

      const rawBtn = document.querySelector('#grp-profile [data-profile="raw"]');
      const mobBtn = document.querySelector('#grp-profile [data-profile="mobile"]');
      const hasRaw = dish.runs.some((r) => r.raw && (r.raw.path || r.raw.base64));
      const hasMob = dish.runs.some((r) => r.mobile && (r.mobile.path || r.mobile.base64));
      rawBtn.disabled = !hasRaw;
      mobBtn.disabled = !hasMob;
      if (!hasRaw && state.profile === 'raw') state.profile = 'mobile';
      if (!hasMob && hasRaw && state.profile === 'mobile') state.profile = 'raw';
      rawBtn.classList.toggle('active', state.profile === 'raw');
      mobBtn.classList.toggle('active', state.profile === 'mobile');

      const group = document.getElementById('grp-backend');
      group.replaceChildren();
      for (const model of dish.model_statuses) {
        const btn = document.createElement('button');
        btn.className = 'ctrl'; btn.dataset.view = model.slug; btn.textContent = model.name;
        btn.disabled = !dish.runs.some(r => r.slug === model.slug);
        btn.classList.toggle('active', state.view === model.slug);
        btn.onclick = () => { state.view = model.slug; refreshStage(); };
        group.appendChild(btn);
      }
      const compare = document.createElement('button');
      compare.className = 'ctrl'; compare.dataset.view = 'compare'; compare.textContent = 'Compare Side-by-Side';
      compare.disabled = dish.runs.length < 2;
      compare.onclick = () => { state.view = 'compare'; refreshStage(); };
      group.appendChild(compare);
      const comparing = state.view === 'compare' && dish.runs.length >= 2;
      stageWrap.classList.toggle('compare', comparing);
      document.getElementById('vp-secondary').classList.toggle('hidden', !comparing);
      document.getElementById('compare-controls').hidden = !comparing;
      const left = document.getElementById('cmp-left'), right = document.getElementById('cmp-right');
      for (const [select, fallback] of [[left, 0], [right, 1]]) {
        const previous = select.value;
        select.replaceChildren(...dish.runs.map(r => { const o = document.createElement('option'); o.value = r.slug; o.textContent = dish.model_statuses.find(m => m.slug === r.slug)?.name || r.slug; return o; }));
        select.value = dish.runs.some(r => r.slug === previous) ? previous : dish.runs[Math.min(fallback, dish.runs.length - 1)]?.slug || '';
        select.onchange = () => refreshStage();
      }
      const chosen = dish.runs.find(r => r.slug === (comparing ? left.value : state.view)) || dish.runs[0];
      updateDownloadLink(dish, chosen);
      renderMatrix(dish);
      await loadRunIntoViewport(vpA, chosen, dish);
      if (comparing) await loadRunIntoViewport(vpB, dish.runs.find(r => r.slug === right.value), dish);
    }

    function renderMatrix(dish) {
      const matrix = document.getElementById('model-matrix'); matrix.replaceChildren();
      const summary = document.createElement('p');
      summary.textContent = DATA.dishes.length + ' dishes · ' + dish.model_statuses.map(m => m.name + ': ' + DATA.dishes.filter(d => d.runs.some(r => r.slug === m.slug)).length).join(' · ');
      matrix.appendChild(summary);
      const table = document.createElement('table');
      for (const row of [['Model','Status','License','Vertices','Faces','GLB KB','Latency (s)','Watertight','Appearance'], ...dish.model_statuses.map(m => [m.name,m.status,m.license,m.vertices,m.faces,m.kb,m.elapsed_seconds,m.watertight,m.appearance])]) {
        const tr = document.createElement('tr'); for (const value of row) { const cell = document.createElement('td'); cell.textContent = value ?? '—'; tr.appendChild(cell); } table.appendChild(tr);
      }
      matrix.appendChild(table);
    }
    document.querySelectorAll('#grp-profile [data-profile]').forEach((btn) => {
      btn.onclick = () => {
        if (btn.disabled) return;
        state.profile = btn.dataset.profile;
        document.querySelectorAll('#grp-profile [data-profile]').forEach((b) => b.classList.toggle('active', b === btn));
        refreshStage();
      };
    });
    document.querySelectorAll('#grp-shader [data-shader]').forEach((btn) => {
      btn.onclick = () => {
        state.shader = btn.dataset.shader;
        document.querySelectorAll('#grp-shader [data-shader]').forEach((b) => b.classList.toggle('active', b === btn));
        applyShaderToViewport(vpA);
        applyShaderToViewport(vpB);
      };
    });
    document.getElementById('btn-wireframe').onclick = (ev) => {
      state.wireframe = !state.wireframe;
      ev.currentTarget.classList.toggle('active', state.wireframe);
      applyShaderToViewport(vpA);
      applyShaderToViewport(vpB);
    };
    document.querySelectorAll('#grp-camera [data-preset]').forEach((btn) => {
      btn.onclick = () => setPreset(btn.dataset.preset);
    });
    document.getElementById('btn-rotate').onclick = (ev) => {
      state.autoRotate = !state.autoRotate;
      ev.currentTarget.classList.toggle('active', state.autoRotate);
      ev.currentTarget.textContent = 'Auto-Rotate: ' + (state.autoRotate ? 'ON' : 'OFF');
      for (const vp of [vpA, vpB]) if (vp.controls) vp.controls.autoRotate = state.autoRotate;
    };
    document.getElementById('btn-grid').onclick = (ev) => {
      state.grid = !state.grid;
      ev.currentTarget.classList.toggle('active', state.grid);
      ev.currentTarget.textContent = 'Grid: ' + (state.grid ? 'ON' : 'OFF');
      for (const vp of [vpA, vpB]) if (vp.gridHelper) vp.gridHelper.visible = state.grid;
    };
    document.querySelectorAll('#grp-env [data-bg]').forEach((btn) => {
      btn.onclick = () => {
        state.bg = btn.dataset.bg;
        document.querySelectorAll('#grp-env [data-bg]').forEach((b) => b.classList.toggle('active', b === btn));
        for (const vp of [vpA, vpB]) {
          if (vp.scene) vp.scene.background = new THREE.Color(BG_HEX[state.bg]);
        }
      };
    });
    document.getElementById('rng-exposure').oninput = (ev) => {
      state.exposure = parseFloat(ev.target.value);
      for (const vp of [vpA, vpB]) if (vp.renderer) vp.renderer.toneMappingExposure = state.exposure;
    };

    window.__PEEK_STUDIO__ = {
      state,
      setDish: (label) => {
        const idx = DATA.dishes.findIndex((d) => d.label === label);
        if (idx >= 0) { state.dishIdx = idx; return refreshStage(); }
      },
      setView: (v) => { state.view = v; return refreshStage(); },
      setShader: (s) => { state.shader = s; applyShaderToViewport(vpA); applyShaderToViewport(vpB); },
    };

    function animate() {
      requestAnimationFrame(animate);
      for (const vp of [vpA, vpB]) {
        if (!vp.renderer || vp.hostEl.closest('.vp-card')?.classList.contains('hidden')) continue;
        vp.controls.update();
        vp.renderer.render(vp.scene, vp.camera);
      }
    }
    refreshStage();
    initThreeAsync();
  </script>
</body>
</html>
"""


def _studio_encode_png_data_url(path, max_dim=512):
    from io import BytesIO
    with Image.open(path) as im:
        norm = ImageOps.exif_transpose(im)
        mode = 'RGBA' if 'A' in norm.getbands() else 'RGB'
        conv = norm.convert(mode)
        if max(conv.size) > max_dim:
            conv.thumbnail((max_dim, max_dim), Image.Resampling.LANCZOS)
        buf = BytesIO()
        conv.save(buf, format='PNG', optimize=True)
    return 'data:image/png;base64,' + base64.b64encode(buf.getvalue()).decode('ascii')


def _build_notebook_studio_payload(embed_mobile_assets=False, max_thumb_px=512):
    dishes = []
    for label in ACTIVE_LABELS:
        in_entry = image_results[label]
        if in_entry.get('image_status') != 'success':
            continue
        cm = in_entry.get('cutout_metrics', {})
        src_path = Path(in_entry['image_path'])
        cut_path = Path(in_entry['cutout_path'])
        src_rel = src_path.relative_to(RESULTS_DIR).as_posix()
        cut_rel = cut_path.relative_to(RESULTS_DIR).as_posix()
        jm_path = in_entry.get('job_manifest_path')
        jm_data = json.loads(Path(jm_path).read_text(encoding='utf-8')) if jm_path and Path(jm_path).is_file() else {}
        if jm_data and jm_data.get('publication_eligible') is not False:
            raise ValueError(f'Dish {label} job_manifest must retain publication_eligible=false.')

        models_cfg = {
            'moge2-relief': ('MoGe-2 Relief + Ceramic Plate', 'MIT'),
            'trellis2-4b': ('TRELLIS.2-4B', 'MIT'),
            'triposg': ('TripoSG + Photo PBR', 'MIT'),
            'hunyuan3d2': ('Hunyuan3D-2 / 2.1', 'Tencent Community R&D'),
            'hunyuan3d21': ('Hunyuan3D-2.1', 'Tencent Community R&D'),
            'hi3dgen': ('Hi3DGen Normal-Bridge', 'MIT'),
        }
        order_list = list(models_cfg.keys())
        model_statuses = []
        runs_by_slug = {}
        runs_out = []
        for run_key, rec in RUNS_3D.items():
            if rec.get('input_label') != label:
                continue
            _slug_key = run_key.split('/')[-1]
            runs_by_slug[_slug_key] = rec
            if rec.get('status') != 'success' or not rec.get('artifact_paths'):
                continue
            raw_path = Path(rec['artifact_paths'][0])
            mob_path = Path(rec['mobile_glb_path']) if rec.get('mobile_glb_path') else None
            raw_rel = raw_path.relative_to(RESULTS_DIR).as_posix()
            mob_exists = bool(mob_path and mob_path.is_file())
            mob_rel = mob_path.relative_to(RESULTS_DIR).as_posix() if mob_exists else None

            embed_mob_b64 = base64.b64encode(mob_path.read_bytes()).decode('ascii') if (embed_mobile_assets and mob_exists) else None
            embed_raw_b64 = (
                base64.b64encode(raw_path.read_bytes()).decode('ascii')
                if (embed_mobile_assets and not mob_exists and raw_path.is_file())
                else None
            )

            fb_path = Path(rec['hf_modes_path']) if rec.get('hf_modes_path') else (
                Path(rec['preview_path']) if rec.get('preview_path') else None
            )
            fb_ref = None
            if fb_path and fb_path.is_file():
                fb_ref = _studio_encode_png_data_url(fb_path, max_thumb_px) if embed_mobile_assets else fb_path.relative_to(RESULTS_DIR).as_posix()

            m = rec.get('metrics') or {}
            mob_m = rec.get('mobile_metrics') or {}
            raw_verts = m.get('vertices', m.get('vertex_count'))
            runs_out.append({
                'key': run_key,
                'backend': rec['backend'],
                'slug': run_key.split('/')[-1],
                'appearance': rec.get('appearance', 'unknown'),
                'device': rec.get('device', 'unknown'),
                'elapsed_seconds': m.get('elapsed_seconds'),
                'has_vertex_color': bool(m.get('has_vertex_color', False)),
                'has_texture': bool(m.get('has_texture', False)),
                'raw': {
                    'path': None if embed_mobile_assets else raw_rel,
                    'base64': embed_raw_b64,
                    'filename': raw_path.name,
                    'vertices': raw_verts,
                    'faces': m.get('faces'),
                    'bytes': m.get('bytes', raw_path.stat().st_size if raw_path.is_file() else None),
                },
                'mobile_status': rec.get('mobile_status', 'available' if (mob_exists or embed_mob_b64) else 'missing'),
                'mobile': {
                    'path': None if embed_mobile_assets else mob_rel,
                    'base64': embed_mob_b64,
                    'filename': mob_path.name if mob_path else None,
                    'vertices': mob_m.get('vertices', mob_m.get('vertex_count', raw_verts)),
                    'faces': mob_m.get('faces', m.get('faces')),
                    'bytes': mob_m.get('bytes', mob_path.stat().st_size if mob_exists else m.get('bytes')),
                } if mob_exists or embed_mob_b64 else None,
                'fallback_image': fb_ref,
            })

        runs_out.sort(key=lambda _r: order_list.index(_r['slug']) if _r['slug'] in order_list else len(order_list))
        for _sk, (_sn, _sl) in models_cfg.items():
            _sr = runs_by_slug.get(_sk, {})
            _sm = _sr.get('metrics') or {}
            _status = _sr.get('status', 'not_run')
            if _status == 'success' and not any(_r['slug'] == _sk for _r in runs_out):
                _status = 'failed'
            model_statuses.append({
                'slug': _sk, 'name': _sn, 'status': _status,
                'license': _sr.get('license') or _sl,
                'watertight': _sr.get('watertight', _sm.get('watertight')),
                'elapsed_seconds': _sr.get('elapsed_seconds', _sm.get('elapsed_seconds')),
                'vertices': _sm.get('vertices', _sm.get('vertex_count')),
                'faces': _sm.get('faces'),
                'kb': (_sm['bytes'] / 1024.0) if _sm.get('bytes') is not None else None,
                'error': _sr.get('error'), 'appearance': _sr.get('appearance'),
            })
        dishes.append({
            'label': label,
            'title': in_entry.get('title'),
            'tier': in_entry.get('tier'),
            'page': in_entry.get('page'),
            'model_statuses': model_statuses,
            'job_id': jm_data.get('job_id'),
            'image_source': in_entry.get('image_source'),
            'size': in_entry.get('size'),
            'source_sha256': in_entry.get('source_sha256') or jm_data.get('source_sha256'),
            'cutout_metrics': cm,
            'source_image': _studio_encode_png_data_url(src_path, max_thumb_px) if embed_mobile_assets else src_rel,
            'cutout_image': _studio_encode_png_data_url(cut_path, max_thumb_px) if embed_mobile_assets else cut_rel,
            'runs': runs_out,
        })

    return {
        'schema_version': 1,
        'demo_label': DEMO_TRUTHFUL_LABEL,
        'publication_eligible': False,
        'plate_preservation_mode': PLATE_PRESERVATION_MODE,
        'standalone': bool(embed_mobile_assets),
        'three_version': STUDIO_THREE_VERSION,
        'dishes': dishes,
    }


def _render_notebook_studio(payload, three_base_url=STUDIO_THREE_CDN_BASE):
    if payload.get('publication_eligible') is not False:
        raise ValueError('Studio payload must retain publication_eligible=false.')
    base_url = three_base_url.rstrip('/')
    import_map = {
        'imports': {
            'three': f'{base_url}/build/three.module.js',
            'three/addons/': f'{base_url}/examples/jsm/',
        }
    }
    demo_badge = f"{payload['demo_label']} | publication_eligible=false"
    mode_tag = (
        'Standalone Offline HTML (Embedded Mobile <=35k GLBs)'
        if payload.get('standalone')
        else 'Interactive Multi-File Studio (Mobile <=35k + Raw GLBs)'
    )
    replacements = {
        '__TITLE__': html_lib.escape('PeekaPlate — Interactive Plate-Preserving 3D Dish Studio'),
        '__IMPORTMAP_JSON__': json.dumps(import_map, ensure_ascii=False).replace('<', '\\u003c'),
        '__STUDIO_DATA_JSON__': json.dumps(payload, ensure_ascii=False).replace('<', '\\u003c'),
        '__DEMO_BADGE__': html_lib.escape(demo_badge),
        '__MODE_TAG__': html_lib.escape(mode_tag),
    }
    return re.sub(
        r'__(?:TITLE|IMPORTMAP_JSON|STUDIO_DATA_JSON|DEMO_BADGE|MODE_TAG)__',
        lambda match: replacements[match.group(0)],
        STUDIO_HTML_TEMPLATE,
    )


viewer_path = RESULTS_DIR / 'comparison_viewer.html'
viewer_path.write_text(_render_notebook_studio(_build_notebook_studio_payload(embed_mobile_assets=False)), encoding='utf-8')
extra_artifacts.append(viewer_path)

standalone_studio_path = RESULTS_DIR / 'interactive_3d_studio.html'
standalone_studio_path.write_text(_render_notebook_studio(_build_notebook_studio_payload(embed_mobile_assets=True)), encoding='utf-8')
extra_artifacts.append(standalone_studio_path)

manifest_path, zip_path = write_results(RESULTS_DIR, image_results, RUNS_3D, SETTINGS, REVISIONS, extra_artifacts)
display(FileLink(str(manifest_path)))
display(FileLink(str(zip_path)))

print('\n=== FINAL PLATE-PRESERVING DEMO SUMMARY ===')
for key, rec in RUNS_3D.items():
    m = rec.get('metrics', {})
    mob = rec.get('mobile_metrics', {})
    print(f"{key:36s} | {rec['status']:7s} | {rec['appearance']:14s} | "
          f"mobile={mobile_delivery_status(rec)} | raw_f={str(m.get('faces', '-')):>7s} | mob_f={str(mob.get('faces', '-')):>6s} | "
          f"raw_B={str(m.get('bytes', '-')):>9s} | mob_B={str(mob.get('bytes', '-')):>8s}")
